In [ ]:
# ============================================================
# CELL 1 — PROJECT SETUP, IMPORTS, AND DATA PATHS
# ============================================================

# Reusable Kalman implementation:
#
#   src/py4dgeo/m3c2_kalman.py
#
# Kalman-seed / 4D-OBC integration:
#
#   src/py4dgeo/segmentation_kalman.py
#
# Analysis, plotting, diagnostics, and evaluation remain
# in this notebook.
# ============================================================


# ------------------------------------------------------------
# 1. STANDARD PYTHON LIBRARIES
# ------------------------------------------------------------

import os
import sys
import glob
import time
import warnings

from pathlib import Path


# ------------------------------------------------------------
# 2. SCIENTIFIC COMPUTING
# ------------------------------------------------------------

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ------------------------------------------------------------
# 3. POINT CLOUD AND 4D CHANGE ANALYSIS
# ------------------------------------------------------------

import py4dgeo


# ------------------------------------------------------------
# 4. FILTERING, SPATIAL ANALYSIS, AND EVALUATION
# ------------------------------------------------------------

from scipy.spatial import cKDTree
from scipy.ndimage import median_filter
from scipy.spatial.distance import directed_hausdorff


# ------------------------------------------------------------
# 5. CLUSTERING AND EVALUATION
# ------------------------------------------------------------

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import adjusted_rand_score


# ------------------------------------------------------------
# 6. SUPPRESS WARNINGS
# ------------------------------------------------------------

warnings.filterwarnings(
    "ignore"
)


# ============================================================
# VERIFY PY4DGEO INSTALLATION
# ============================================================

print(
    "py4dgeo loaded from:"
)

print(
    py4dgeo.__file__
)


py4dgeo_package_dir = Path(
    py4dgeo.__file__
).parent


print(
    "\npy4dgeo package directory:"
)

print(
    py4dgeo_package_dir
)


# ------------------------------------------------------------
# Verify integrated thesis modules
# ------------------------------------------------------------

INTEGRATED_MODULES = [
    "m3c2_kalman.py",
    "segmentation_kalman.py",
]


print(
    "\nChecking integrated Kalman modules:\n"
)


for module_name in INTEGRATED_MODULES:

    module_path = (
        py4dgeo_package_dir
        / module_name
    )

    if module_path.is_file():

        print(
            f"[FOUND]   {module_name}"
        )

    else:

        print(
            f"[MISSING] {module_name}"
        )


# ============================================================
# DATA DIRECTORY SETUP
# ============================================================
# Dataset folder expected beside the notebook:
#
#   kijkduin/
#       pointclouds/
# ============================================================

data_path = os.path.join(
    os.getcwd(),
    "kijkduin"
)


if not os.path.isdir(
    data_path
):

    raise FileNotFoundError(
        f'Dataset directory not found:\n{data_path}\n\n'
        'Please make sure the "kijkduin" folder is in the '
        'same directory as the notebook.'
    )


print(
    "\nDataset directory:"
)

print(
    data_path
)


# ------------------------------------------------------------
# Point-cloud directory
# ------------------------------------------------------------

pc_dir = os.path.join(
    data_path,
    "pointclouds"
)


if not os.path.isdir(
    pc_dir
):

    raise FileNotFoundError(
        f'Point-cloud directory not found:\n{pc_dir}\n\n'
        'Please make sure the "pointclouds" folder exists '
        'inside "kijkduin".'
    )


print(
    "\nPoint-cloud directory:"
)

print(
    pc_dir
)


# ------------------------------------------------------------
# List available point-cloud files
# ------------------------------------------------------------

pc_list = sorted(
    os.listdir(
        pc_dir
    )
)


print(
    f"\nFound {len(pc_list)} files "
    "in the point-cloud directory.\n"
)


print(
    "First five files:"
)


for file_name in pc_list[:5]:

    print(
        file_name
    )


# ============================================================
# OUTPUT DIRECTORIES
# ============================================================

RESULTS_DIR = Path(
    "results"
)

FIGURES_DIR = Path(
    "figures"
)

TABLES_DIR = Path(
    "tables"
)


for folder in [
    RESULTS_DIR,
    FIGURES_DIR,
    TABLES_DIR
]:

    folder.mkdir(
        exist_ok=True,
        parents=True
    )


print(
    "\nOutput directories:"
)

print(
    "Results:",
    RESULTS_DIR.resolve()
)

print(
    "Figures:",
    FIGURES_DIR.resolve()
)

print(
    "Tables:",
    TABLES_DIR.resolve()
)


# ============================================================
# FINAL STATUS
# ============================================================

print(
    "\n=========================================="
)

print(
    "CELL 1 COMPLETE"
)

print(
    "=========================================="
)

print(
    "Project setup finished."
)

print(
    "The notebook is using the integrated "
    "py4dgeo Kalman implementation."
)

In [ ]:
# ============================================================
# CELL 2 — LOAD POINT CLOUD TIME SERIES
# ============================================================



# ------------------------------------------------------------
# 1. BUILD FULL FILE PATHS
# ------------------------------------------------------------

SUPPORTED_POINTCLOUD_EXTENSIONS = (
    ".ply",
    ".las",
    ".laz",
    ".xyz",
    ".txt",
    ".csv",
    ".pcd",
)


pc_files = sorted([
    os.path.join(
        pc_dir,
        file_name
    )
    for file_name in pc_list
    if file_name.lower().endswith(
        SUPPORTED_POINTCLOUD_EXTENSIONS
    )
])


if len(pc_files) == 0:

    raise FileNotFoundError(
        f"No supported point-cloud files found in:\n{pc_dir}"
    )


print(
    "Number of usable point-cloud files:",
    len(pc_files)
)

print(
    "First file:",
    os.path.basename(
        pc_files[0]
    )
)

print(
    "Last file:",
    os.path.basename(
        pc_files[-1]
    )
)


# ------------------------------------------------------------
# 2. POINT-CLOUD LOADER
# ------------------------------------------------------------

def load_pointcloud_py4dgeo(
    file_path
):
    """
    Load one point cloud as a py4dgeo Epoch.

    Parameters
    ----------
    file_path : str or pathlib.Path
        Path to the point-cloud file.

    Returns
    -------
    py4dgeo.Epoch
        Loaded point-cloud epoch.
    """

    file_path = str(
        file_path
    )

    extension = Path(
        file_path
    ).suffix.lower()


    if extension in (
        ".las",
        ".laz"
    ):

        return py4dgeo.read_from_las(
            file_path
        )


    elif extension in (
        ".xyz",
        ".txt",
        ".csv",
        ".ply",
        ".pcd"
    ):

        return py4dgeo.read_from_xyz(
            file_path
        )


    else:

        raise ValueError(
            f"Unsupported point-cloud format: "
            f"{extension}"
        )


# ------------------------------------------------------------
# 3. LOAD ALL EPOCHS
# ------------------------------------------------------------

epochs = []


print(
    "\nLoading point-cloud time series..."
)


for epoch_index, file_path in enumerate(
    pc_files
):

    epoch = load_pointcloud_py4dgeo(
        file_path
    )

    epochs.append(
        epoch
    )


    print(
        f"Loaded epoch {epoch_index:03d}: "
        f"{os.path.basename(file_path)}"
    )


print(
    "\nAll point clouds loaded."
)

print(
    "Total epochs:",
    len(epochs)
)


# ------------------------------------------------------------
# 4. DEFINE REFERENCE AND COMPARISON EPOCHS
# ------------------------------------------------------------
# The first epoch is used as the reference/null epoch.

reference_epoch = (
    epochs[0]
)

comparison_epochs = (
    epochs[1:]
)


reference_file = (
    pc_files[0]
)

comparison_files = (
    pc_files[1:]
)


print(
    "\nReference epoch:"
)

print(
    os.path.basename(
        reference_file
    )
)


print(
    "\nNumber of comparison epochs:"
)

print(
    len(
        comparison_epochs
    )
)


# ------------------------------------------------------------
# 5. INSPECT REFERENCE EPOCH
# ------------------------------------------------------------

reference_points = np.asarray(
    reference_epoch.cloud
)


print(
    "\nReference point-cloud shape:"
)

print(
    reference_points.shape
)


print(
    "\nFirst five reference points:"
)

print(
    reference_points[:5]
)


# ------------------------------------------------------------
# 6. QUICK REFERENCE-ELEVATION PLOT
# ------------------------------------------------------------

plt.figure(
    figsize=(8, 6)
)


scatter = plt.scatter(

    reference_points[:, 0],

    reference_points[:, 1],

    c=reference_points[:, 2],

    s=2,

    cmap="terrain"
)


plt.colorbar(
    scatter,
    label="Elevation / Z"
)

plt.xlabel(
    "X"
)

plt.ylabel(
    "Y"
)

plt.title(
    "Reference epoch elevation"
)

plt.axis(
    "equal"
)

plt.tight_layout()

plt.show()


# ------------------------------------------------------------
# 7. FINAL STATUS
# ------------------------------------------------------------

print(
    "\n=========================================="
)

print(
    "CELL 2 COMPLETE"
)

print(
    "=========================================="
)

print(
    "Point-cloud time series loaded successfully."
)

In [ ]:
# ============================================================
# CELL 3 — PREPARE EPOCH FILES, TIMESTAMPS, AND METADATA
# ============================================================



# ------------------------------------------------------------
# 1. IMPORT TIMESTAMP PARSER
# ------------------------------------------------------------

from datetime import datetime


# ------------------------------------------------------------
# 2. KEEP ONLY LAZ FILES AND SORT THEM
# ------------------------------------------------------------

pc_list = sorted([
    file_name
    for file_name in os.listdir(pc_dir)
    if file_name.lower().endswith(".laz")
])


if len(pc_list) == 0:

    raise FileNotFoundError(
        f"No .laz files found in:\n{pc_dir}\n"
        "Please check the pointclouds folder."
    )


print(
    "Number of LAZ point clouds found:",
    len(pc_list)
)


print(
    "\nFirst five files:"
)


for file_name in pc_list[:5]:

    print(
        file_name
    )


# ------------------------------------------------------------
# 3. PARSE TIMESTAMPS FROM FILENAMES
# ------------------------------------------------------------
# Expected filename structure:
#
#   prefix_YYMMDD_HHMMSS.laz
#
# Example:
#
#   Kijkduin_190101_120000.laz
#
# The first underscore-separated component is treated as the
# file prefix and everything after it as YYMMDD_HHMMSS.
# ------------------------------------------------------------

timestamps = []


for file_name in pc_list:

    filename_without_ext = (
        Path(file_name).stem
    )


    filename_parts = (
        filename_without_ext.split("_")
    )


    if len(filename_parts) < 3:

        raise ValueError(
            f"Filename does not match the expected "
            f"'prefix_YYMMDD_HHMMSS.laz' format:\n"
            f"{file_name}"
        )


    timestamp_str = "_".join(
        filename_parts[1:]
    )


    try:

        timestamp = datetime.strptime(
            timestamp_str,
            "%y%m%d_%H%M%S"
        )

    except ValueError as exc:

        raise ValueError(
            f"Could not parse timestamp from:\n"
            f"{file_name}\n"
            f"Extracted timestamp string: {timestamp_str}"
        ) from exc


    timestamps.append(
        timestamp
    )


print(
    "\nFirst five timestamps:"
)


for timestamp in timestamps[:5]:

    print(
        timestamp
    )


# ------------------------------------------------------------
# 4. DEFINE REFERENCE FILE
# ------------------------------------------------------------
# The first acquisition is the global reference/null epoch.

reference_epoch_file = os.path.join(
    pc_dir,
    pc_list[0]
)


reference_timestamp = (
    timestamps[0]
)


print(
    "\nReference epoch file:"
)

print(
    pc_list[0]
)


print(
    "Reference timestamp:",
    reference_timestamp
)


# ------------------------------------------------------------
# 5. DEFINE COMPARISON FILES AND TIMESTAMPS
# ------------------------------------------------------------

comparison_epoch_files = [
    os.path.join(
        pc_dir,
        file_name
    )
    for file_name in pc_list[1:]
]


comparison_timestamps = (
    timestamps[1:]
)


print(
    "\nNumber of comparison epochs:",
    len(comparison_epoch_files)
)


# ------------------------------------------------------------
# 6. BASIC CONSISTENCY CHECKS
# ------------------------------------------------------------

if (
    len(comparison_epoch_files)
    != len(comparison_timestamps)
):

    raise RuntimeError(
        "Comparison file count does not match "
        "comparison timestamp count."
    )


if timestamps != sorted(timestamps):

    print(
        "\nWARNING:"
    )

    print(
        "Parsed timestamps are not chronologically ordered."
    )

    print(
        "Check whether lexical filename sorting corresponds "
        "to acquisition order."
    )


# ------------------------------------------------------------
# 7. CREATE AND SAVE EPOCH METADATA
# ------------------------------------------------------------

metadata_table = pd.DataFrame({

    "epoch_index":
        np.arange(
            len(pc_list),
            dtype=int
        ),

    "epoch_type":
        [
            "reference"
        ]
        + [
            "comparison"
        ] * (
            len(pc_list) - 1
        ),

    "filename":
        pc_list,

    "timestamp":
        timestamps,
})


display(
    metadata_table.head()
)


metadata_table.to_csv(
    TABLES_DIR
    / "kijkduin_epoch_metadata.csv",
    index=False
)


# ------------------------------------------------------------
# 8. FINAL STATUS
# ------------------------------------------------------------

print(
    "\n=========================================="
)

print(
    "CELL 3 COMPLETE"
)

print(
    "=========================================="
)

print(
    "Epoch files and timestamps prepared."
)

print(
    "Reference/comparison paths are ready for Cell 4."
)

In [ ]:
# ============================================================
# CELL 4 — CREATE ANALYSIS, SELECT COREPOINTS,
# CONFIGURE M3C2, AND ADD EPOCHS
# ============================================================



# ------------------------------------------------------------
# 1. USER SETTINGS — COREPOINT SELECTION
# ------------------------------------------------------------

COREPOINT_SELECTION = "full"

# options:
#   "full"
#   "step"
#   "grid"


# ------------------------------------------------------------
# Option A — deterministic step sampling
# ------------------------------------------------------------

COREPOINT_STEP = 5


# ------------------------------------------------------------
# Option B — approximately uniform XY-grid sampling
# ------------------------------------------------------------
# Units are the same as the point-cloud coordinates.
# For projected coordinates this will normally be metres.

COREPOINT_GRID_SIZE = 1.0


# ------------------------------------------------------------
# 2. VALIDATE COREPOINT SETTINGS
# ------------------------------------------------------------

if COREPOINT_SELECTION not in [
    "full",
    "step",
    "grid"
]:

    raise ValueError(
        "COREPOINT_SELECTION must be "
        "'full', 'step', or 'grid'."
    )


if (
    not isinstance(
        COREPOINT_STEP,
        int
    )
    or
    COREPOINT_STEP < 1
):

    raise ValueError(
        "COREPOINT_STEP must be an integer >= 1."
    )


if COREPOINT_GRID_SIZE <= 0:

    raise ValueError(
        "COREPOINT_GRID_SIZE must be greater than zero."
    )


# ============================================================
# 3. RECREATE ANALYSIS ARCHIVE
# ============================================================

analysis_file = os.path.join(
    data_path,
    "kijkduin.zip"
)


if os.path.exists(
    analysis_file
):

    os.remove(
        analysis_file
    )

    print(
        "Old analysis archive removed:"
    )

    print(
        analysis_file
    )


analysis = py4dgeo.SpatiotemporalAnalysis(
    analysis_file,
    force=True
)


print(
    "\nNew SpatiotemporalAnalysis archive created:"
)

print(
    analysis_file
)


# ============================================================
# 4. LOAD AND SET REFERENCE EPOCH
# ============================================================
# reference_epoch_file and reference_timestamp were prepared
# in Cell 3.

reference_epoch = py4dgeo.read_from_las(
    reference_epoch_file
)


reference_epoch.timestamp = (
    reference_timestamp
)


analysis.reference_epoch = (
    reference_epoch
)


reference_cloud = np.asarray(
    reference_epoch.cloud
)


n_reference_points = (
    reference_cloud.shape[0]
)


print(
    "\nReference epoch set."
)

print(
    "Reference file:",
    os.path.basename(
        reference_epoch_file
    )
)

print(
    "Reference timestamp:",
    reference_epoch.timestamp
)

print(
    "Reference points:",
    n_reference_points
)


# ============================================================
# 5. COREPOINT SELECTION FUNCTIONS
# ============================================================

def select_step_corepoints(
    cloud,
    step
):
    """
    Deterministically retain every `step`-th reference point.

    Returns
    -------
    selected_points : np.ndarray

    source_indices : np.ndarray
        Indices in the original reference cloud.
    """

    source_indices = np.arange(
        0,
        cloud.shape[0],
        step,
        dtype=np.int64
    )


    selected_points = (
        cloud[
            source_indices
        ]
    )


    return (
        selected_points,
        source_indices
    )


def select_xy_grid_corepoints(
    cloud,
    grid_size
):
    """
    Select approximately one point per XY grid cell.

    This produces a more spatially uniform corepoint distribution
    than simple index-based step sampling.

    The first reference point encountered in each occupied XY cell
    is retained.

    Returns
    -------
    selected_points : np.ndarray

    source_indices : np.ndarray
        Indices in the original reference cloud.
    """

    xy = np.asarray(
        cloud[:, :2],
        dtype=float
    )


    xy_min = np.nanmin(
        xy,
        axis=0
    )


    grid_coordinates = np.floor(
        (
            xy
            - xy_min
        )
        / float(
            grid_size
        )
    ).astype(
        np.int64
    )


    # Unique XY cells.
    # return_index gives the first point occurring in each cell.

    _, first_indices = np.unique(
        grid_coordinates,
        axis=0,
        return_index=True
    )


    # Sort indices to preserve original point ordering.

    source_indices = np.sort(
        first_indices.astype(
            np.int64
        )
    )


    selected_points = (
        cloud[
            source_indices
        ]
    )


    return (
        selected_points,
        source_indices
    )


# ============================================================
# 6. SELECT COMMON ANALYSIS COREPOINTS
# ============================================================

if COREPOINT_SELECTION == "full":

    corepoint_source_indices = np.arange(
        n_reference_points,
        dtype=np.int64
    )


    selected_corepoints = (
        reference_cloud.copy()
    )


elif COREPOINT_SELECTION == "step":

    (
        selected_corepoints,
        corepoint_source_indices
    ) = select_step_corepoints(

        cloud=(
            reference_cloud
        ),

        step=(
            COREPOINT_STEP
        )
    )


elif COREPOINT_SELECTION == "grid":

    (
        selected_corepoints,
        corepoint_source_indices
    ) = select_xy_grid_corepoints(

        cloud=(
            reference_cloud
        ),

        grid_size=(
            COREPOINT_GRID_SIZE
        )
    )


# ------------------------------------------------------------
# Store the selected corepoints in py4dgeo
# ------------------------------------------------------------

analysis.corepoints = (
    selected_corepoints
)


n_corepoints = (
    analysis.corepoints.cloud.shape[0]
)


retained_percentage = (
    100.0
    * n_corepoints
    / n_reference_points
)


reduction_percentage = (
    100.0
    - retained_percentage
)


print(
    "\nCommon analysis corepoints set."
)

print(
    "Selection method:",
    COREPOINT_SELECTION
)

print(
    "Original reference points:",
    n_reference_points
)

print(
    "Selected corepoints:",
    n_corepoints
)

print(
    f"Retained percentage: "
    f"{retained_percentage:.2f}%"
)

print(
    f"Reduction percentage: "
    f"{reduction_percentage:.2f}%"
)


if COREPOINT_SELECTION == "step":

    print(
        "COREPOINT_STEP:",
        COREPOINT_STEP
    )


elif COREPOINT_SELECTION == "grid":

    print(
        "COREPOINT_GRID_SIZE:",
        COREPOINT_GRID_SIZE
    )


# ============================================================
# 7. SAVE COREPOINT-SELECTION INFORMATION
# ============================================================

np.save(
    TABLES_DIR
    / "corepoint_source_indices.npy",
    corepoint_source_indices
)


corepoint_sampling_summary = pd.DataFrame([
    {
        "selection_method":
            COREPOINT_SELECTION,

        "corepoint_step":
            (
                COREPOINT_STEP
                if
                COREPOINT_SELECTION
                == "step"
                else
                np.nan
            ),

        "grid_size":
            (
                COREPOINT_GRID_SIZE
                if
                COREPOINT_SELECTION
                == "grid"
                else
                np.nan
            ),

        "original_reference_points":
            n_reference_points,

        "selected_corepoints":
            n_corepoints,

        "retained_percentage":
            retained_percentage,

        "reduction_percentage":
            reduction_percentage,
    }
])


display(
    corepoint_sampling_summary
)


corepoint_sampling_summary.to_csv(
    TABLES_DIR
    / "corepoint_sampling_summary.csv",
    index=False
)


# ============================================================
# 8. CONFIGURE M3C2
# ============================================================
# These are the existing M3C2 parameters and are intentionally
# kept unchanged.

analysis.m3c2 = py4dgeo.M3C2(

    normal_radii=(
        5.0,
    ),

    cyl_radius=1.0,

    max_distance=10.0,

    registration_error=0.019
)


print(
    "\nM3C2 parameters set."
)

print(
    "normal_radii = (5.0,)"
)

print(
    "cyl_radius = 1.0"
)

print(
    "max_distance = 10.0"
)

print(
    "registration_error = 0.019"
)


# ============================================================
# 9. LOAD AND ADD COMPARISON EPOCHS
# ============================================================
# analysis.add_epochs() computes M3C2 distances at the common
# corepoints and stores them in the analysis archive.

comparison_epochs = []


print(
    "\nLoading and adding comparison epochs..."
)


start_time = time.time()


for epoch_index, (
    epoch_file,
    timestamp
) in enumerate(

    zip(
        comparison_epoch_files,
        comparison_timestamps
    ),

    start=1
):

    epoch = py4dgeo.read_from_las(
        epoch_file
    )


    epoch.timestamp = (
        timestamp
    )


    comparison_epochs.append(
        epoch
    )


    print(
        f"Adding epoch "
        f"{epoch_index:03d}/"
        f"{len(comparison_epoch_files)}: "
        f"{os.path.basename(epoch_file)} | "
        f"{timestamp}"
    )


    analysis.add_epochs(
        epoch
    )


elapsed = (
    time.time()
    - start_time
)


print(
    "\nAll comparison epochs added."
)

print(
    f"Elapsed time: "
    f"{elapsed:.2f} seconds"
)

print(
    "M3C2 analysis corepoints per epoch:",
    n_corepoints
)


# ============================================================
# 10. VERIFY RAW M3C2 OUTPUT
# ============================================================

if analysis.distances is None:

    raise RuntimeError(
        "M3C2 distances were not created."
    )


print(
    "\nRaw M3C2 distance array shape:"
)

print(
    analysis.distances.shape
)


# ============================================================
# 11. BUILD EPOCH SUMMARY TABLE
# ============================================================

epoch_summary_records = []


epoch_summary_records.append({

    "epoch_index":
        0,

    "type":
        "reference",

    "filename":
        pc_list[0],

    "timestamp":
        timestamps[0],

    "n_points":
        reference_epoch.cloud.shape[0],

    "n_analysis_corepoints":
        n_corepoints,

    "corepoint_selection":
        COREPOINT_SELECTION,
})


for epoch_index, epoch in enumerate(
    comparison_epochs,
    start=1
):

    epoch_summary_records.append({

        "epoch_index":
            epoch_index,

        "type":
            "comparison",

        "filename":
            pc_list[
                epoch_index
            ],

        "timestamp":
            epoch.timestamp,

        "n_points":
            epoch.cloud.shape[0],

        "n_analysis_corepoints":
            n_corepoints,

        "corepoint_selection":
            COREPOINT_SELECTION,
    })


epoch_summary_table = pd.DataFrame(
    epoch_summary_records
)


display(
    epoch_summary_table.head()
)


display(
    epoch_summary_table.tail()
)


epoch_summary_table.to_csv(
    TABLES_DIR
    / "epoch_summary_table.csv",
    index=False
)


# ============================================================
# 12. PLOT RAW POINT COUNT AND ANALYSIS COREPOINT COUNT
# ============================================================

plt.figure(
    figsize=(
        10,
        4
    )
)


plt.plot(

    epoch_summary_table[
        "epoch_index"
    ],

    epoch_summary_table[
        "n_points"
    ],

    marker="o",

    label="Raw points per epoch"
)


plt.axhline(

    y=(
        n_corepoints
    ),

    linestyle="--",

    label=(
        f"Analysis corepoints "
        f"({COREPOINT_SELECTION})"
    )
)


plt.xlabel(
    "Epoch index"
)

plt.ylabel(
    "Number of points"
)

plt.title(
    "Raw point count and common analysis corepoint count"
)

plt.grid(
    True
)

plt.legend()

plt.tight_layout()

plt.show()


# ============================================================
# 13. FINAL VERIFICATION
# ============================================================

if (
    analysis.corepoints.cloud.shape[0]
    != n_corepoints
):

    raise RuntimeError(
        "Stored py4dgeo corepoint count does not match "
        "the selected corepoint count."
    )


if n_corepoints == 0:

    raise RuntimeError(
        "No analysis corepoints were selected."
    )


print(
    "\n=========================================="
)

print(
    "CELL 4 COMPLETE"
)

print(
    "=========================================="
)


print(
    "Corepoint selection:",
    COREPOINT_SELECTION
)


print(
    "Common analysis corepoints:",
    n_corepoints
)


print(
    "Raw M3C2 shape:",
    analysis.distances.shape
)


print(
    "\nAll later methods must use this same "
    "analysis/corepoint configuration."
)

In [ ]:
# ============================================================
# CELL 5 — INSPECT M3C2 DISTANCES AND VISUALIZE CHANGE SERIES
# ============================================================


# ------------------------------------------------------------
# 1. VERIFY M3C2 OUTPUTS
# ------------------------------------------------------------

if analysis.distances is None:

    raise RuntimeError(
        "analysis.distances is None. "
        "Please run Cell 4 first."
    )


if analysis.uncertainties is None:

    raise RuntimeError(
        "analysis.uncertainties is None. "
        "Please run Cell 4 first."
    )


if "lodetection" not in analysis.uncertainties.dtype.names:

    raise RuntimeError(
        "'lodetection' is not available in "
        "analysis.uncertainties."
    )


# ------------------------------------------------------------
# 2. STORE COMMON ANALYSIS ARRAYS
# ------------------------------------------------------------

corepoints = np.asarray(
    analysis.corepoints.cloud
)


distances = np.asarray(
    analysis.distances
)


lodetection = np.asarray(
    analysis.uncertainties[
        "lodetection"
    ]
)


uncertainty_fields = (
    analysis.uncertainties.dtype.names
)


print(
    "Corepoints shape:",
    corepoints.shape
)

print(
    "Distances shape:",
    distances.shape
)

print(
    "LoDetection shape:",
    lodetection.shape
)


print(
    "\nAvailable uncertainty fields:"
)

print(
    uncertainty_fields
)


# ------------------------------------------------------------
# 3. PRINT SMALL ARRAY SAMPLES
# ------------------------------------------------------------

print(
    "\nFirst 3 corepoints x first 5 epochs "
    "of M3C2 distances:"
)

print(
    distances[
        :3,
        :5
    ]
)


print(
    "\nFirst 3 corepoints x first 5 epochs "
    "of M3C2 LoD:"
)

print(
    lodetection[
        :3,
        :5
    ]
)


print(
    "\nFirst five analysis time deltas:"
)

print(
    analysis.timedeltas[
        :5
    ]
)


# ------------------------------------------------------------
# 4. BUILD ANALYSIS TIMESTAMPS
# ------------------------------------------------------------
# These timestamps are used throughout the later 4DOBC,
# KF-Mag, KF-Rate, NMS, visualization, and evaluation cells.

timestamps_analysis = [

    analysis.reference_epoch.timestamp
    + timedelta

    for timedelta
    in analysis.timedeltas
]


if (
    len(timestamps_analysis)
    != distances.shape[1]
):

    raise RuntimeError(
        "Number of analysis timestamps does not match "
        "the number of M3C2 epochs."
    )


print(
    "\nFirst five analysis timestamps:"
)


for timestamp in timestamps_analysis[:5]:

    print(
        timestamp
    )


# ------------------------------------------------------------
# 5. CREATE M3C2 SUMMARY TABLE
# ------------------------------------------------------------

m3c2_summary = pd.DataFrame({

    "epoch_index":
        np.arange(
            distances.shape[1],
            dtype=int
        ),

    "timestamp":
        timestamps_analysis,

    "mean_distance":
        np.nanmean(
            distances,
            axis=0
        ),

    "std_distance":
        np.nanstd(
            distances,
            axis=0
        ),

    "min_distance":
        np.nanmin(
            distances,
            axis=0
        ),

    "max_distance":
        np.nanmax(
            distances,
            axis=0
        ),

    "mean_lodetection":
        np.nanmean(
            lodetection,
            axis=0
        ),
})


display(
    m3c2_summary.head()
)


m3c2_summary.to_csv(
    TABLES_DIR
    / "m3c2_distance_summary.csv",
    index=False
)


# ------------------------------------------------------------
# 6. USER SETTINGS FOR VISUALIZATION
# ------------------------------------------------------------

VISUALIZATION_COREPOINT_INDEX = 15162

VISUALIZATION_EPOCH_INDEX = 28


# ------------------------------------------------------------
# Ensure requested indices exist
# ------------------------------------------------------------

cp_idx_sel = min(
    VISUALIZATION_COREPOINT_INDEX,
    corepoints.shape[0] - 1
)


epoch_idx_sel = min(
    VISUALIZATION_EPOCH_INDEX,
    distances.shape[1] - 1
)


if (
    cp_idx_sel
    != VISUALIZATION_COREPOINT_INDEX
):

    print(
        "\nWARNING:"
    )

    print(
        f"Requested visualization corepoint "
        f"{VISUALIZATION_COREPOINT_INDEX} does not exist."
    )

    print(
        f"Using corepoint {cp_idx_sel} instead."
    )


if (
    epoch_idx_sel
    != VISUALIZATION_EPOCH_INDEX
):

    print(
        "\nWARNING:"
    )

    print(
        f"Requested visualization epoch "
        f"{VISUALIZATION_EPOCH_INDEX} does not exist."
    )

    print(
        f"Using epoch {epoch_idx_sel} instead."
    )


print(
    "\nSelected corepoint index:",
    cp_idx_sel
)

print(
    "Selected epoch index:",
    epoch_idx_sel
)


# ------------------------------------------------------------
# 7. EXTRACT VALUES FOR PLOTTING
# ------------------------------------------------------------

distances_epoch = (
    distances[
        :,
        epoch_idx_sel
    ]
)


coord_sel = (
    corepoints[
        cp_idx_sel
    ]
)


timeseries_sel = (
    distances[
        cp_idx_sel
    ]
)


# ------------------------------------------------------------
# 8. VISUALIZE SPATIAL CHANGE + TIME SERIES
# ------------------------------------------------------------

import matplotlib.dates as mdates


dtFmt = mdates.DateFormatter(
    "%b-%d"
)


fig = plt.figure(
    figsize=(
        15,
        5
    )
)


ax1 = fig.add_subplot(
    1,
    2,
    1
)


ax2 = fig.add_subplot(
    1,
    2,
    2
)


# ------------------------------------------------------------
# Left panel — spatial M3C2 change at selected epoch
# ------------------------------------------------------------

scatter = ax1.scatter(

    corepoints[
        :,
        0
    ],

    corepoints[
        :,
        1
    ],

    c=(
        distances_epoch
    ),

    cmap="seismic_r",

    vmin=-1.5,

    vmax=1.5,

    s=1,

    zorder=1
)


plt.colorbar(

    scatter,

    format="%.2f",

    label="Distance [m]",

    ax=ax1,

    pad=0.15
)


ax1.scatter(

    coord_sel[
        0
    ],

    coord_sel[
        1
    ],

    facecolor="yellow",

    edgecolor="black",

    s=100,

    zorder=2,

    label=(
        f"Corepoint {cp_idx_sel}"
    ),

    marker="*"
)


ax1.legend()


ax1.set_xlabel(
    "X [m]"
)


ax1.set_ylabel(
    "Y [m]"
)


ax1.set_aspect(
    "equal"
)


ax1.set_title(
    "M3C2 change at "
    f"{timestamps_analysis[epoch_idx_sel]}"
)


# ------------------------------------------------------------
# Right panel — raw M3C2 time series
# ------------------------------------------------------------

ax2.scatter(

    timestamps_analysis,

    timeseries_sel,

    s=7,

    color="black",

    alpha=0.45,

    label="Raw M3C2 observations"
)


ax2.plot(

    timestamps_analysis,

    timeseries_sel,

    color="blue",

    linewidth=1.2,

    label="M3C2 time series"
)


ax2.scatter(

    timestamps_analysis[
        epoch_idx_sel
    ],

    timeseries_sel[
        epoch_idx_sel
    ],

    facecolor="yellow",

    edgecolor="black",

    s=100,

    marker="*",

    label=(
        f"Selected epoch {epoch_idx_sel}"
    )
)


ax2.xaxis.set_major_formatter(
    dtFmt
)


ax2.set_xlabel(
    "Date"
)


ax2.set_ylabel(
    "Distance [m]"
)


ax2.tick_params(
    axis="x",
    rotation=15
)


ax2.grid(
    True
)


ax2.set_title(
    f"Raw M3C2 time series at "
    f"corepoint {cp_idx_sel}"
)


ax2.legend()


plt.tight_layout()

plt.show()


# ------------------------------------------------------------
# 9. SAVE COMMON ARRAYS
# ------------------------------------------------------------

np.save(
    RESULTS_DIR
    / "corepoints.npy",
    corepoints
)


np.save(
    RESULTS_DIR
    / "m3c2_distances.npy",
    distances
)


np.save(
    RESULTS_DIR
    / "m3c2_lodetection.npy",
    lodetection
)


# ------------------------------------------------------------
# 10. FINAL STATUS
# ------------------------------------------------------------

print(
    "\n=========================================="
)

print(
    "CELL 5 COMPLETE"
)

print(
    "=========================================="
)


print(
    "M3C2 distances inspected and visualized."
)


print(
    "Common arrays prepared for later methods:"
)


print(
    " - corepoints"
)


print(
    " - distances"
)


print(
    " - lodetection"
)


print(
    " - timestamps_analysis"
)

In [ ]:
# ============================================================
# CELL 6 — TEMPORAL SMOOTHING OF M3C2 TIME SERIES
# ============================================================



# ------------------------------------------------------------
# 1. USER SETTING
# ------------------------------------------------------------

SMOOTHING_WINDOW = 14


if (
    not isinstance(
        SMOOTHING_WINDOW,
        int
    )
    or
    SMOOTHING_WINDOW < 1
):
    raise ValueError(
        "SMOOTHING_WINDOW must be an integer >= 1."
    )


print(
    "Applying temporal smoothing..."
)

print(
    "Smoothing window:",
    SMOOTHING_WINDOW
)


# ------------------------------------------------------------
# 2. VERIFY RAW M3C2 DISTANCES
# ------------------------------------------------------------

if analysis.distances is None:

    raise RuntimeError(
        "analysis.distances is None. "
        "Please run Cell 4 first."
    )


# ------------------------------------------------------------
# 3. APPLY PY4DGEO TEMPORAL AVERAGING
# ------------------------------------------------------------

analysis.smoothed_distances = (
    py4dgeo.temporal_averaging(
        analysis.distances,
        smoothing_window=SMOOTHING_WINDOW
    )
)


# ------------------------------------------------------------
# 4. VERIFY SMOOTHED OUTPUT
# ------------------------------------------------------------

if analysis.smoothed_distances is None:

    raise RuntimeError(
        "Temporal smoothing failed: "
        "analysis.smoothed_distances is None."
    )


if (
    analysis.smoothed_distances.shape
    != analysis.distances.shape
):

    raise RuntimeError(
        "Smoothed distance array shape does not match "
        "the raw distance array shape."
    )


print(
    "\nTemporal smoothing complete."
)

print(
    "Raw distances shape:",
    analysis.distances.shape
)

print(
    "Smoothed distances shape:",
    analysis.smoothed_distances.shape
)


# ------------------------------------------------------------
# 5. SELECT COREPOINT FOR VISUAL CHECK
# ------------------------------------------------------------

if "cp_idx_sel" not in globals():

    cp_idx_sel = min(
        15162,
        analysis.distances.shape[0] - 1
    )


if (
    cp_idx_sel < 0
    or
    cp_idx_sel >= analysis.distances.shape[0]
):

    raise ValueError(
        f"cp_idx_sel={cp_idx_sel} is outside the "
        "available corepoint range."
    )


print(
    "\nSelected corepoint:",
    cp_idx_sel
)


# ------------------------------------------------------------
# 6. EXTRACT RAW AND SMOOTHED TIME SERIES
# ------------------------------------------------------------

raw_timeseries_sel = (
    analysis.distances[
        cp_idx_sel
    ]
)


smooth_timeseries_sel = (
    analysis.smoothed_distances[
        cp_idx_sel
    ]
)


# ------------------------------------------------------------
# 7. PLOT RAW VS SMOOTHED TIME SERIES
# ------------------------------------------------------------

import matplotlib.dates as mdates


dtFmt = mdates.DateFormatter(
    "%b-%d"
)


fig, ax = plt.subplots(
    figsize=(8, 5)
)


ax.scatter(
    timestamps_analysis,
    raw_timeseries_sel,
    s=8,
    alpha=0.7,
    label="Raw M3C2"
)


ax.plot(
    timestamps_analysis,
    smooth_timeseries_sel,
    linewidth=2,
    label="Smoothed M3C2"
)


ax.xaxis.set_major_formatter(
    dtFmt
)


ax.set_xlabel(
    "Date"
)

ax.set_ylabel(
    "Distance [m]"
)

ax.set_title(
    f"Raw vs smoothed M3C2 time series "
    f"at corepoint {cp_idx_sel}"
)

ax.tick_params(
    axis="x",
    rotation=15
)

ax.grid(
    True
)

ax.legend()


plt.tight_layout()

plt.show()


# ------------------------------------------------------------
# 8. SAVE SMOOTHED DISTANCES
# ------------------------------------------------------------

np.save(
    RESULTS_DIR
    / "m3c2_smoothed_distances.npy",
    analysis.smoothed_distances
)


# ------------------------------------------------------------
# 9. FINAL STATUS
# ------------------------------------------------------------

print(
    "\n=========================================="
)

print(
    "CELL 6 COMPLETE"
)

print(
    "=========================================="
)

print(
    "Temporal smoothing applied successfully."
)

print(
    "analysis.smoothed_distances is ready "
    "for later clustering and 4DOBC extraction."
)

In [ ]:
# ============================================================
# CELL 7 — OPTIONAL K-MEANS CLUSTERING OF M3C2 CHANGE PATTERNS
# ============================================================



# ------------------------------------------------------------
# 1. USER SETTINGS
# ------------------------------------------------------------

RUN_KMEANS_CLUSTERING = True


KMEANS_CLUSTER_NUMBERS = [
    5,
    10,
    20,
    50
]


KMEANS_RANDOM_STATE = 0

KMEANS_N_INIT = 10


# ------------------------------------------------------------
# 2. VERIFY SMOOTHED INPUT
# ------------------------------------------------------------

if analysis.smoothed_distances is None:

    raise RuntimeError(
        "analysis.smoothed_distances is None. "
        "Please run Cell 6 first."
    )


distances_for_clustering = np.asarray(
    analysis.smoothed_distances,
    dtype=float
)


print(
    "Input for clustering:"
)

print(
    "Smoothed-distance shape:",
    distances_for_clustering.shape
)


# ------------------------------------------------------------
# 3. IDENTIFY COMPLETE / VALID TIME SERIES
# ------------------------------------------------------------
# KMeans cannot handle NaN or infinite values.
#
# A corepoint is therefore considered valid only when every
# epoch in its smoothed time series is finite.

valid_timeseries = np.all(
    np.isfinite(
        distances_for_clustering
    ),
    axis=1
)


n_valid_timeseries = int(
    np.sum(
        valid_timeseries
    )
)


n_invalid_timeseries = int(
    np.sum(
        ~valid_timeseries
    )
)


print(
    "\nValid time series:",
    n_valid_timeseries
)

print(
    "Invalid time series:",
    n_invalid_timeseries
)


if n_valid_timeseries == 0:

    raise RuntimeError(
        "No complete finite M3C2 time series are available "
        "for K-means clustering."
    )


# ------------------------------------------------------------
# 4. PREPARE CLUSTERING OUTPUT
# ------------------------------------------------------------

ks = list(
    KMEANS_CLUSTER_NUMBERS
)


labels = np.full(

    (
        distances_for_clustering.shape[0],
        len(ks)
    ),

    np.nan,

    dtype=float
)


# ------------------------------------------------------------
# 5. RUN OPTIONAL K-MEANS
# ------------------------------------------------------------

if RUN_KMEANS_CLUSTERING:

    print(
        "\nRunning exploratory K-means clustering."
    )

    print(
        "Cluster numbers:",
        ks
    )


    valid_data = (
        distances_for_clustering[
            valid_timeseries,
            :
        ]
    )


    for k_index, k in enumerate(
        ks
    ):

        # ----------------------------------------------------
        # K cannot exceed the number of available observations
        # ----------------------------------------------------

        if k > n_valid_timeseries:

            print(
                f"\nSkipping k={k}: "
                f"only {n_valid_timeseries} valid "
                "time series are available."
            )

            continue


        print(
            f"\nPerforming clustering with k={k}..."
        )


        kmeans = KMeans(

            n_clusters=k,

            random_state=(
                KMEANS_RANDOM_STATE
            ),

            n_init=(
                KMEANS_N_INIT
            )
        )


        cluster_labels = (
            kmeans.fit_predict(
                valid_data
            )
        )


        labels[
            valid_timeseries,
            k_index
        ] = (
            cluster_labels
        )


        print(
            f"Finished clustering with k={k}"
        )


else:

    print(
        "\nK-means clustering skipped."
    )

    print(
        "This does not affect the five extraction methods."
    )


# ------------------------------------------------------------
# 6. SAVE CLUSTER LABELS
# ------------------------------------------------------------

np.save(

    RESULTS_DIR
    / "kmeans_labels_smoothed_m3c2.npy",

    labels
)


print(
    "\nCluster-label array saved to:"
)

print(
    RESULTS_DIR
    / "kmeans_labels_smoothed_m3c2.npy"
)


# ------------------------------------------------------------
# 7. SAVE CLUSTERING METADATA
# ------------------------------------------------------------

clustering_metadata = pd.DataFrame({

    "k_index":
        np.arange(
            len(ks),
            dtype=int
        ),

    "n_clusters":
        ks,

    "n_valid_timeseries":
        n_valid_timeseries,

    "n_invalid_timeseries":
        n_invalid_timeseries,

    "random_state":
        KMEANS_RANDOM_STATE,

    "n_init":
        KMEANS_N_INIT,

    "clustering_enabled":
        RUN_KMEANS_CLUSTERING,
})


display(
    clustering_metadata
)


clustering_metadata.to_csv(

    TABLES_DIR
    / "kmeans_clustering_metadata.csv",

    index=False
)


# ------------------------------------------------------------
# 8. FINAL STATUS
# ------------------------------------------------------------

print(
    "\n=========================================="
)

print(
    "CELL 7 COMPLETE"
)

print(
    "=========================================="
)


if RUN_KMEANS_CLUSTERING:

    print(
        "Exploratory K-means clustering finished."
    )

else:

    print(
        "Exploratory K-means clustering was skipped."
    )


print(
    "This cell does not affect 4DOBC or "
    "the four Kalman-based extraction methods."
)

In [ ]:
# ============================================================
# CELL 8 — OPTIONAL VISUALIZATION OF K-MEANS RESULTS
# ============================================================



# ------------------------------------------------------------
# 1. CHECK REQUIRED VARIABLES
# ------------------------------------------------------------

required_objects = [
    "RUN_KMEANS_CLUSTERING",
    "labels",
    "ks",
    "corepoints",
    "distances_for_clustering",
    "timestamps_analysis",
]


for obj_name in required_objects:

    if obj_name not in globals():

        raise ValueError(
            f"{obj_name} is not defined. "
            "Please run Cell 7 first."
        )


# ------------------------------------------------------------
# 2. SKIP SAFELY IF CLUSTERING WAS DISABLED
# ------------------------------------------------------------

if not RUN_KMEANS_CLUSTERING:

    print(
        "K-means visualization skipped because "
        "RUN_KMEANS_CLUSTERING = False."
    )

    print(
        "This does not affect any extraction method."
    )

else:

    # --------------------------------------------------------
    # 3. USER SETTING
    # --------------------------------------------------------

    K_SELECTED = 5


    if K_SELECTED not in ks:

        raise ValueError(
            f"K_SELECTED={K_SELECTED} is not available. "
            f"Available values: {ks}"
        )


    k_idx = (
        ks.index(
            K_SELECTED
        )
    )


    cluster_labels = (
        labels[
            :,
            k_idx
        ]
    )


    valid_label_mask = np.isfinite(
        cluster_labels
    )


    n_labelled_points = int(
        np.sum(
            valid_label_mask
        )
    )


    print(
        f"Visualizing K-means result for k={K_SELECTED}"
    )


    print(
        "Number of labelled corepoints:",
        n_labelled_points
    )


    if n_labelled_points == 0:

        raise RuntimeError(
            f"No valid cluster labels exist for k={K_SELECTED}."
        )


    # --------------------------------------------------------
    # 4. SPATIAL MAP OF CLUSTER LABELS
    # --------------------------------------------------------

    fig, ax = plt.subplots(
        figsize=(
            8,
            6
        )
    )


    scatter = ax.scatter(

        corepoints[
            :,
            0
        ],

        corepoints[
            :,
            1
        ],

        c=(
            cluster_labels
        ),

        cmap="tab20",

        s=2
    )


    plt.colorbar(
        scatter,
        ax=ax,
        label="Cluster label"
    )


    ax.set_xlabel(
        "X [m]"
    )


    ax.set_ylabel(
        "Y [m]"
    )


    ax.set_aspect(
        "equal"
    )


    ax.set_title(
        f"K-means clusters of smoothed M3C2 time series "
        f"(k={K_SELECTED})"
    )


    plt.tight_layout()

    plt.show()


    # --------------------------------------------------------
    # 5. MEAN TIME SERIES PER CLUSTER
    # --------------------------------------------------------

    import matplotlib.dates as mdates


    dtFmt = mdates.DateFormatter(
        "%b-%d"
    )


    fig, ax = plt.subplots(
        figsize=(
            10,
            5
        )
    )


    cluster_ids = sorted(
        np.unique(
            cluster_labels[
                valid_label_mask
            ]
        )
    )


    for cluster_id in cluster_ids:

        cluster_id = int(
            cluster_id
        )


        cluster_mask = (
            cluster_labels
            == cluster_id
        )


        n_cluster_points = int(
            np.sum(
                cluster_mask
            )
        )


        if n_cluster_points == 0:

            continue


        mean_timeseries = np.nanmean(

            distances_for_clustering[
                cluster_mask,
                :
            ],

            axis=0
        )


        ax.plot(

            timestamps_analysis,

            mean_timeseries,

            linewidth=1.5,

            label=(
                f"Cluster {cluster_id} "
                f"(n={n_cluster_points})"
            )
        )


    ax.xaxis.set_major_formatter(
        dtFmt
    )


    ax.set_xlabel(
        "Date"
    )


    ax.set_ylabel(
        "Mean M3C2 distance [m]"
    )


    ax.set_title(
        f"Mean smoothed M3C2 time series per cluster "
        f"(k={K_SELECTED})"
    )


    ax.tick_params(
        axis="x",
        rotation=15
    )


    ax.grid(
        True
    )


    ax.legend(
        fontsize=7,
        ncol=2
    )


    plt.tight_layout()

    plt.show()


    # --------------------------------------------------------
    # 6. SAVE SELECTED CLUSTER MAP
    # --------------------------------------------------------

    cluster_map_table = pd.DataFrame({

        "x":
            corepoints[
                :,
                0
            ],

        "y":
            corepoints[
                :,
                1
            ],

        "z":
            corepoints[
                :,
                2
            ],

        f"cluster_k{K_SELECTED}":
            cluster_labels
    })


    cluster_map_path = (
        TABLES_DIR
        / f"kmeans_cluster_map_k{K_SELECTED}.csv"
    )


    cluster_map_table.to_csv(
        cluster_map_path,
        index=False
    )


    print(
        "\nCluster map saved to:"
    )


    print(
        cluster_map_path
    )


    # --------------------------------------------------------
    # 7. FINAL STATUS
    # --------------------------------------------------------

    print(
        "\n=========================================="
    )

    print(
        "CELL 8 COMPLETE"
    )

    print(
        "=========================================="
    )

    print(
        "Exploratory K-means visualization finished."
    )

    print(
        "This cell does not affect 4DOBC or "
        "the four Kalman-based methods."
    )

In [ ]:
# ============================================================
# CELL 9 — ORIGINAL PY4DGEO 4D-OBC BASELINE EXTRACTION
# ============================================================



# ------------------------------------------------------------
# 1. USER SETTINGS
# ------------------------------------------------------------

SEED_MODE = "range"

# Options:
#   "single"
#   "range"
#   "full"


# ------------------------------------------------------------
# Single-corepoint mode
# User numbering starts from 1.
# ------------------------------------------------------------

SEED_COREPOINT_NUMBER = 15163


# ------------------------------------------------------------
# Range mode
# User numbering starts from 1 and the end is inclusive.
# ------------------------------------------------------------

SEED_RANGE_START = 15000
SEED_RANGE_END = 15999


# ------------------------------------------------------------
# Visualization corepoint
# Python indexing starts from 0.
# ------------------------------------------------------------

PLOT_COREPOINT_INDEX = 15162


# ------------------------------------------------------------
# Existing py4dgeo 4D-OBC parameters
# ------------------------------------------------------------

WINDOW_WIDTH = 14

MINPERIOD = 2

HEIGHT_THRESHOLD = 0.05

NEIGHBORHOOD_RADIUS = 1.0

MIN_SEGMENTS = 10

THRESHOLDS = [
    0.3,
    0.4,
    0.5,
    0.6,
    0.7,
    0.8,
    0.9,
]


# ============================================================
# 2. CHECK REQUIRED INPUTS
# ============================================================

required_objects = [
    "analysis",
    "corepoints",
    "timestamps_analysis",
]


for obj_name in required_objects:

    if obj_name not in globals():

        raise RuntimeError(
            f"{obj_name} is not defined. "
            "Please run the previous cells first."
        )


if analysis.smoothed_distances is None:

    raise RuntimeError(
        "analysis.smoothed_distances is None. "
        "Please run Cell 6 before Cell 9."
    )


n_corepoints = (
    analysis.corepoints.cloud.shape[0]
)


n_epochs = (
    analysis.smoothed_distances.shape[1]
)


if n_corepoints == 0:

    raise RuntimeError(
        "No analysis corepoints are available."
    )


if (
    len(timestamps_analysis)
    != n_epochs
):

    raise RuntimeError(
        "Timestamp count does not match "
        "the smoothed-distance epoch count."
    )


print(
    "Number of available corepoints:",
    n_corepoints
)

print(
    "Number of analysis epochs:",
    n_epochs
)


# ============================================================
# 3. VALIDATE SEED MODE
# ============================================================

VALID_SEED_MODES = [
    "single",
    "range",
    "full",
]


if SEED_MODE not in VALID_SEED_MODES:

    raise ValueError(
        f"Invalid SEED_MODE='{SEED_MODE}'. "
        f"Use one of {VALID_SEED_MODES}."
    )


# ============================================================
# 4. DEFINE SEED-CANDIDATE COREPOINTS
# ============================================================

if SEED_MODE == "single":

    if (
        not isinstance(
            SEED_COREPOINT_NUMBER,
            int
        )
        or
        SEED_COREPOINT_NUMBER <= 0
    ):

        raise ValueError(
            "SEED_COREPOINT_NUMBER must be "
            "a positive integer using user numbering "
            "starting from 1."
        )


    if SEED_COREPOINT_NUMBER > n_corepoints:

        raise ValueError(
            f"SEED_COREPOINT_NUMBER="
            f"{SEED_COREPOINT_NUMBER} exceeds "
            f"the available {n_corepoints} corepoints."
        )


    selected_python_index = (
        SEED_COREPOINT_NUMBER
        - 1
    )


    seed_candidates = [
        selected_python_index
    ]


    cp_idx_sel = (
        selected_python_index
    )


    print(
        "\nSeed mode: SINGLE"
    )

    print(
        "User corepoint number:",
        SEED_COREPOINT_NUMBER
    )

    print(
        "Python corepoint index:",
        cp_idx_sel
    )


elif SEED_MODE == "range":

    if (
        not isinstance(
            SEED_RANGE_START,
            int
        )
        or
        not isinstance(
            SEED_RANGE_END,
            int
        )
    ):

        raise ValueError(
            "SEED_RANGE_START and SEED_RANGE_END "
            "must be integers."
        )


    if (
        SEED_RANGE_START <= 0
        or
        SEED_RANGE_END <= 0
    ):

        raise ValueError(
            "Seed-range numbering starts from 1."
        )


    if (
        SEED_RANGE_END
        < SEED_RANGE_START
    ):

        raise ValueError(
            "SEED_RANGE_END must be greater than "
            "or equal to SEED_RANGE_START."
        )


    if SEED_RANGE_START > n_corepoints:

        raise ValueError(
            f"SEED_RANGE_START={SEED_RANGE_START} "
            f"exceeds the available "
            f"{n_corepoints} corepoints."
        )


    # --------------------------------------------------------
    # Restrict range safely to available corepoints
    # --------------------------------------------------------

    effective_seed_range_end = min(
        SEED_RANGE_END,
        n_corepoints
    )


    if (
        effective_seed_range_end
        != SEED_RANGE_END
    ):

        print(
            f"\nWARNING: SEED_RANGE_END="
            f"{SEED_RANGE_END} exceeds the available "
            f"{n_corepoints} corepoints."
        )

        print(
            "Effective range end:",
            effective_seed_range_end
        )


    start_idx = (
        SEED_RANGE_START
        - 1
    )


    end_idx_exclusive = (
        effective_seed_range_end
    )


    seed_candidates = list(
        range(
            start_idx,
            end_idx_exclusive
        )
    )


    if (
        PLOT_COREPOINT_INDEX
        not in seed_candidates
    ):

        raise ValueError(
            f"PLOT_COREPOINT_INDEX="
            f"{PLOT_COREPOINT_INDEX} is not inside "
            "the selected seed-candidate range."
        )


    cp_idx_sel = (
        PLOT_COREPOINT_INDEX
    )


    print(
        "\nSeed mode: RANGE"
    )

    print(
        "User corepoint range:",
        f"{SEED_RANGE_START} → "
        f"{effective_seed_range_end}"
    )

    print(
        "Python index range:",
        f"{start_idx} → "
        f"{end_idx_exclusive - 1}"
    )

    print(
        "Number of candidate corepoints:",
        len(seed_candidates)
    )

    print(
        "Visualization corepoint Python index:",
        cp_idx_sel
    )


else:

    # --------------------------------------------------------
    # None tells the existing py4dgeo algorithm to use
    # its normal full candidate set.
    # --------------------------------------------------------

    seed_candidates = None


    if (
        PLOT_COREPOINT_INDEX < 0
        or
        PLOT_COREPOINT_INDEX >= n_corepoints
    ):

        raise ValueError(
            f"PLOT_COREPOINT_INDEX="
            f"{PLOT_COREPOINT_INDEX} is outside "
            f"the available range 0–{n_corepoints - 1}."
        )


    cp_idx_sel = (
        PLOT_COREPOINT_INDEX
    )


    print(
        "\nSeed mode: FULL"
    )

    print(
        "All analysis corepoints will be "
        "available as seed candidates."
    )

    print(
        "Visualization corepoint Python index:",
        cp_idx_sel
    )


# ============================================================
# 5. CONFIGURE EXISTING PY4DGEO 4D-OBC ALGORITHM
# ============================================================

region_growing_parameters = {

    "window_width":
        WINDOW_WIDTH,

    "minperiod":
        MINPERIOD,

    "height_threshold":
        HEIGHT_THRESHOLD,

    "neighborhood_radius":
        NEIGHBORHOOD_RADIUS,

    "min_segments":
        MIN_SEGMENTS,

    "thresholds":
        THRESHOLDS,
}


if seed_candidates is not None:

    region_growing_parameters[
        "seed_candidates"
    ] = (
        seed_candidates
    )


algo_original_4dobc = (
    py4dgeo.RegionGrowingAlgorithm(
        **region_growing_parameters
    )
)


print(
    "\nOriginal py4dgeo 4D-OBC algorithm configured."
)

print(
    "Region growing implementation: "
    "py4dgeo.RegionGrowingAlgorithm"
)


# ============================================================
# 6. CLEAR PREVIOUS BASELINE RESULTS
# ============================================================
# Keep smoothed_distances because they were prepared in Cell 6.

analysis.invalidate_results(

    seeds=True,

    objects=True,

    smoothed_distances=False
)


print(
    "Previous seeds and objects cleared."
)


# ============================================================
# 7. RUN ORIGINAL 4D-OBC BASELINE
# ============================================================

print(
    "\nRunning original py4dgeo 4D-OBC extraction..."
)


start_time = (
    time.time()
)


objects_original_4dobc = (
    algo_original_4dobc.run(
        analysis
    )
)


elapsed = (
    time.time()
    - start_time
)


print(
    "\nOriginal 4D-OBC extraction complete."
)

print(
    f"Elapsed time: {elapsed:.2f} seconds"
)


# ============================================================
# 8. STORE BASELINE RESULTS
# ============================================================

original_4dobc_objects = (
    objects_original_4dobc
)


original_4dobc_seeds = list(
    analysis.seeds
)


print(
    "\nNumber of detected original 4D-OBC seeds:",
    len(original_4dobc_seeds)
)


print(
    "Number of extracted original 4D-OBC objects:",
    len(original_4dobc_objects)
)


# ============================================================
# 9. VISUALIZE ORIGINAL SEEDS AT SELECTED COREPOINT
# ============================================================

import matplotlib.dates as mdates


seed_timeseries = (
    analysis.smoothed_distances[
        cp_idx_sel
    ]
)


fig, ax = plt.subplots(
    figsize=(
        10,
        5
    )
)


ax.plot(

    timestamps_analysis,

    seed_timeseries,

    linestyle="--",

    linewidth=0.8,

    label="Smoothed M3C2 time series"
)


n_plotted_seeds = 0


for seed_id, seed in enumerate(
    original_4dobc_seeds,
    start=1
):

    if seed.index != cp_idx_sel:

        continue


    seed_start = int(
        seed.start_epoch
    )


    seed_end = int(
        seed.end_epoch
    )


    ax.plot(

        timestamps_analysis[
            seed_start:
            seed_end + 1
        ],

        seed_timeseries[
            seed_start:
            seed_end + 1
        ],

        linewidth=2,

        label=(
            f"Seed {seed_id}"
        )
    )


    n_plotted_seeds += 1


dtFmt = mdates.DateFormatter(
    "%b-%d"
)


ax.xaxis.set_major_formatter(
    dtFmt
)


ax.set_xlabel(
    "Date"
)


ax.set_ylabel(
    "Distance [m]"
)


ax.tick_params(
    axis="x",
    rotation=15
)


ax.set_title(
    "Original 4D-OBC seed intervals\n"
    f"Corepoint Python index {cp_idx_sel}"
)


ax.grid(
    True
)


if n_plotted_seeds > 0:

    ax.legend()


plt.tight_layout()

plt.show()


print(
    "\nNumber of seed intervals plotted "
    "for selected corepoint:",
    n_plotted_seeds
)


# ============================================================
# 10. SUMMARIZE ORIGINAL 4D-OBC SEEDS
# ============================================================

seed_records = []


for seed_id, seed in enumerate(
    original_4dobc_seeds,
    start=1
):

    seed_records.append({

        "seed_number_user":
            seed_id,

        "corepoint_index_python":
            int(
                seed.index
            ),

        "corepoint_number_user":
            int(
                seed.index
            )
            + 1,

        "start_epoch":
            int(
                seed.start_epoch
            ),

        "end_epoch":
            int(
                seed.end_epoch
            ),

        "duration_epochs":
            int(
                seed.end_epoch
                - seed.start_epoch
                + 1
            ),

        "start_time":
            timestamps_analysis[
                seed.start_epoch
            ],

        "end_time":
            timestamps_analysis[
                seed.end_epoch
            ],
    })


original_seed_summary = pd.DataFrame(
    seed_records
)


display(
    original_seed_summary.head()
)


original_seed_summary.to_csv(

    TABLES_DIR
    / "original_4dobc_seed_summary.csv",

    index=False
)


# ============================================================
# 11. SUMMARIZE ORIGINAL 4D-OBC OBJECTS
# ============================================================

object_records = []


for object_id, obj in enumerate(
    original_4dobc_objects,
    start=1
):

    record = {

        "object_number_user":
            object_id
    }


    if hasattr(
        obj,
        "indices"
    ):

        record[
            "n_corepoints"
        ] = len(
            obj.indices
        )

    else:

        record[
            "n_corepoints"
        ] = np.nan


    if hasattr(
        obj,
        "start_epoch"
    ):

        record[
            "start_epoch"
        ] = int(
            obj.start_epoch
        )

        record[
            "start_time"
        ] = (
            timestamps_analysis[
                obj.start_epoch
            ]
        )


    if hasattr(
        obj,
        "end_epoch"
    ):

        record[
            "end_epoch"
        ] = int(
            obj.end_epoch
        )

        record[
            "end_time"
        ] = (
            timestamps_analysis[
                obj.end_epoch
            ]
        )


    if (
        hasattr(
            obj,
            "start_epoch"
        )
        and
        hasattr(
            obj,
            "end_epoch"
        )
    ):

        record[
            "duration_epochs"
        ] = int(
            obj.end_epoch
            - obj.start_epoch
            + 1
        )


    object_records.append(
        record
    )


original_object_summary = pd.DataFrame(
    object_records
)


display(
    original_object_summary.head()
)


original_object_summary.to_csv(

    TABLES_DIR
    / "original_4dobc_object_summary.csv",

    index=False
)


# ============================================================
# 12. SAVE BASELINE RUN CONFIGURATION
# ============================================================

run_config_original_4dobc = pd.DataFrame([
    {

        "method":
            "4DOBC",

        "implementation":
            "py4dgeo.RegionGrowingAlgorithm",

        "seed_mode":
            SEED_MODE,

        "seed_corepoint_number_single":
            (
                SEED_COREPOINT_NUMBER
                if
                SEED_MODE == "single"
                else
                np.nan
            ),

        "seed_range_start_user":
            (
                SEED_RANGE_START
                if
                SEED_MODE == "range"
                else
                np.nan
            ),

        "seed_range_end_user":
            (
                effective_seed_range_end
                if
                SEED_MODE == "range"
                else
                np.nan
            ),

        "n_candidate_corepoints":
            (
                n_corepoints
                if
                seed_candidates is None
                else
                len(seed_candidates)
            ),

        "window_width":
            WINDOW_WIDTH,

        "minperiod":
            MINPERIOD,

        "height_threshold":
            HEIGHT_THRESHOLD,

        "neighborhood_radius":
            NEIGHBORHOOD_RADIUS,

        "min_segments":
            MIN_SEGMENTS,

        "thresholds":
            str(
                THRESHOLDS
            ),

        "n_detected_seeds":
            len(
                original_4dobc_seeds
            ),

        "n_extracted_objects":
            len(
                original_4dobc_objects
            ),

        "elapsed_seconds":
            elapsed,
    }
])


display(
    run_config_original_4dobc
)


run_config_original_4dobc.to_csv(

    TABLES_DIR
    / "original_4dobc_run_config.csv",

    index=False
)


# ============================================================
# 13. SIMPLE BASELINE OBJECT-COUNT PLOT
# ============================================================

plt.figure(
    figsize=(
        6,
        4
    )
)


plt.bar(
    [
        "4DOBC"
    ],
    [
        len(
            original_4dobc_objects
        )
    ]
)


plt.ylabel(
    "Number of extracted objects"
)


plt.title(
    "Original py4dgeo 4D-OBC baseline"
)


plt.tight_layout()

plt.show()


# ============================================================
# 14. FINAL SUMMARY
# ============================================================

print(
    "\n=========================================="
)

print(
    "CELL 9 FINAL SUMMARY"
)

print(
    "=========================================="
)


print(
    "Method: 4DOBC"
)


print(
    "Implementation: existing "
    "py4dgeo.RegionGrowingAlgorithm"
)


print(
    "Seed mode:",
    SEED_MODE
)


print(
    "Candidate corepoints:",
    (
        n_corepoints
        if
        seed_candidates is None
        else
        len(seed_candidates)
    )
)


print(
    "Detected seeds:",
    len(
        original_4dobc_seeds
    )
)


print(
    "Extracted objects:",
    len(
        original_4dobc_objects
    )
)


print(
    f"Elapsed time: {elapsed:.2f} seconds"
)


print(
    "\nCell 9 complete."
)

In [ ]:
# ============================================================
# CELL 10 — VISUALIZE ORIGINAL 4D-OBC OBJECT PROPERTIES
# ============================================================



# ------------------------------------------------------------
# 1. CHECK REQUIRED VARIABLES
# ------------------------------------------------------------

required_objects = [
    "analysis",
    "original_4dobc_objects",
    "timestamps_analysis",
    "corepoints",
]


for obj_name in required_objects:

    if obj_name not in globals():

        raise ValueError(
            f"{obj_name} is not defined. "
            "Please run Cell 9 first."
        )


if analysis.smoothed_distances is None:

    raise RuntimeError(
        "analysis.smoothed_distances is None. "
        "Please run Cell 6 first."
    )


if len(original_4dobc_objects) == 0:

    raise ValueError(
        "No original 4DOBC objects were extracted in Cell 9."
    )


# ------------------------------------------------------------
# 2. USER SETTINGS
# ------------------------------------------------------------

OBJECT_NUMBER = 1

VISUALIZATION_COREPOINT_INDEX = 15162

CRANGE = 1.0


# ------------------------------------------------------------
# 3. VALIDATE OBJECT NUMBER
# ------------------------------------------------------------

if (
    not isinstance(
        OBJECT_NUMBER,
        int
    )
    or
    OBJECT_NUMBER <= 0
):

    raise ValueError(
        "OBJECT_NUMBER must be a positive integer "
        "using user numbering starting from 1."
    )


if OBJECT_NUMBER > len(
    original_4dobc_objects
):

    raise ValueError(
        f"OBJECT_NUMBER={OBJECT_NUMBER} exceeds the "
        f"{len(original_4dobc_objects)} available objects."
    )


selected_object_index = (
    OBJECT_NUMBER - 1
)


selected_object = (
    original_4dobc_objects[
        selected_object_index
    ]
)


print(
    "Visualizing original 4DOBC object:",
    OBJECT_NUMBER
)

print(
    "Python object index:",
    selected_object_index
)


# ------------------------------------------------------------
# 4. EXTRACT OBJECT INFORMATION
# ------------------------------------------------------------

object_indices = np.asarray(
    selected_object.indices,
    dtype=int
)


start_epoch = int(
    selected_object.start_epoch
)


end_epoch = int(
    selected_object.end_epoch
)


duration_epochs = (
    end_epoch
    - start_epoch
    + 1
)


epoch_of_interest = int(
    (
        start_epoch
        + end_epoch
    )
    / 2
)


print(
    "\nObject start epoch:",
    start_epoch
)

print(
    "Object end epoch:",
    end_epoch
)

print(
    "Object duration:",
    duration_epochs
)

print(
    "Epoch of interest:",
    epoch_of_interest
)

print(
    "Number of object corepoints:",
    len(
        object_indices
    )
)


# ------------------------------------------------------------
# 5. ACTUAL OBJECT-GENERATING SEED
# ------------------------------------------------------------

if (
    hasattr(
        selected_object,
        "seed"
    )
    and
    selected_object.seed is not None
):

    object_seed = (
        selected_object.seed
    )


    seed_cp_idx = int(
        object_seed.index
    )


    seed_start_epoch = int(
        object_seed.start_epoch
    )


    seed_end_epoch = int(
        object_seed.end_epoch
    )


    seed_duration_epochs = (
        seed_end_epoch
        - seed_start_epoch
        + 1
    )


    print(
        "\nActual generating seed found."
    )

    print(
        "Seed corepoint Python index:",
        seed_cp_idx
    )

    print(
        "Seed corepoint user number:",
        seed_cp_idx + 1
    )

    print(
        "Seed start/end:",
        seed_start_epoch,
        seed_end_epoch
    )

    print(
        "Seed duration:",
        seed_duration_epochs
    )


else:

    object_seed = None

    seed_cp_idx = np.nan

    seed_start_epoch = np.nan

    seed_end_epoch = np.nan

    seed_duration_epochs = np.nan


    print(
        "\nNo generating seed is attached to "
        "the selected object."
    )


# ------------------------------------------------------------
# 6. FIXED VISUALIZATION COREPOINT
# ------------------------------------------------------------

n_corepoints = (
    analysis.corepoints.cloud.shape[0]
)


if (
    VISUALIZATION_COREPOINT_INDEX < 0
    or
    VISUALIZATION_COREPOINT_INDEX
    >= n_corepoints
):

    raise ValueError(
        f"VISUALIZATION_COREPOINT_INDEX="
        f"{VISUALIZATION_COREPOINT_INDEX} is outside the "
        f"available range 0–{n_corepoints - 1}."
    )


visualization_cp_idx = int(
    VISUALIZATION_COREPOINT_INDEX
)


print(
    "\nVisualization corepoint Python index:",
    visualization_cp_idx
)

print(
    "Visualization corepoint user number:",
    visualization_cp_idx + 1
)


object_index_set = set(
    object_indices.tolist()
)


print(
    "Object contains visualization corepoint:",
    visualization_cp_idx
    in object_index_set
)


# ------------------------------------------------------------
# 7. COMPUTE CHANGE MAGNITUDE DURING OBJECT TIMESPAN
# ------------------------------------------------------------

smoothed_distances = np.asarray(
    analysis.smoothed_distances
)


magnitudes_of_interest = (

    smoothed_distances[
        :,
        epoch_of_interest
    ]

    -

    smoothed_distances[
        :,
        start_epoch
    ]
)


# ------------------------------------------------------------
# 8. PLOTTING IMPORTS AND COLOR SCALE
# ------------------------------------------------------------

import matplotlib.colors as mcolors
import matplotlib.dates as mdates

from scipy.spatial import ConvexHull
from matplotlib.patches import Polygon


cmap = plt.get_cmap(
    "seismic_r"
).copy()


norm = mcolors.CenteredNorm(
    halfrange=CRANGE
)


cmap_values = norm(
    magnitudes_of_interest
)


dtFmt = mdates.DateFormatter(
    "%b-%d"
)


# ------------------------------------------------------------
# 9. CREATE FIGURE
# ------------------------------------------------------------

fig, axs = plt.subplots(
    1,
    2,
    figsize=(
        15,
        7
    )
)


ax1, ax2 = axs


# ============================================================
# 10. LEFT PANEL — OBJECT TIME SERIES
# ============================================================

for idx in object_indices[
    ::10
]:

    ax1.plot(

        timestamps_analysis,

        smoothed_distances[
            idx
        ],

        color=cmap(
            cmap_values[
                idx
            ]
        ),

        linewidth=0.5,

        alpha=0.6
    )


# ------------------------------------------------------------
# Visualization-corepoint time series
# ------------------------------------------------------------

ax1.plot(

    timestamps_analysis,

    smoothed_distances[
        visualization_cp_idx
    ],

    color="black",

    linewidth=1.5,

    label=(
        f"Visualization corepoint "
        f"{visualization_cp_idx}"
    ),

    zorder=6
)


# ------------------------------------------------------------
# Actual generating-seed time series
# ------------------------------------------------------------

if (
    object_seed is not None
    and
    seed_cp_idx
    != visualization_cp_idx
):

    ax1.plot(

        timestamps_analysis,

        smoothed_distances[
            seed_cp_idx
        ],

        linestyle="--",

        linewidth=1.5,

        label=(
            f"Generating seed "
            f"{seed_cp_idx}"
        ),

        zorder=5
    )


# ------------------------------------------------------------
# Object timespan
# ------------------------------------------------------------

ax1.axvspan(

    timestamps_analysis[
        start_epoch
    ],

    timestamps_analysis[
        end_epoch
    ],

    alpha=0.25,

    color="grey",

    label="4DOBC object timespan"
)


# ------------------------------------------------------------
# Seed timespan
# ------------------------------------------------------------

if object_seed is not None:

    ax1.axvspan(

        timestamps_analysis[
            seed_start_epoch
        ],

        timestamps_analysis[
            seed_end_epoch
        ],

        alpha=0.10,

        hatch="//",

        edgecolor="black",

        facecolor="none",

        label="Generating seed timespan"
    )


ax1.xaxis.set_major_formatter(
    dtFmt
)


ax1.set_title(
    "Original 4DOBC temporal object behaviour"
)


ax1.set_xlabel(
    "Date"
)


ax1.set_ylabel(
    "Distance [m]"
)


ax1.tick_params(
    axis="x",
    rotation=15
)


ax1.grid(
    True
)


ax1.legend(
    fontsize=8
)


# ============================================================
# 11. RIGHT PANEL — SPATIAL OBJECT EXTENT
# ============================================================

cloud = np.asarray(
    analysis.corepoints.cloud
)


object_xy = (
    cloud[
        object_indices,
        :2
    ]
)


scatter = ax2.scatter(

    cloud[
        :,
        0
    ],

    cloud[
        :,
        1
    ],

    c=(
        magnitudes_of_interest
    ),

    cmap="seismic_r",

    vmin=-CRANGE,

    vmax=CRANGE,

    s=1
)


plt.colorbar(

    scatter,

    format="%.2f",

    label="Change magnitude [m]",

    ax=ax2
)


ax2.set_aspect(
    "equal"
)


# ------------------------------------------------------------
# 12. ROBUST CONVEX HULL
# ------------------------------------------------------------

unique_object_xy = np.unique(
    object_xy,
    axis=0
)


hull_created = False


if unique_object_xy.shape[0] >= 3:

    try:

        hull = ConvexHull(
            unique_object_xy
        )


        hull_vertices = (
            unique_object_xy[
                hull.vertices
            ]
        )


        ax2.add_patch(
            Polygon(

                hull_vertices,

                label="4DOBC hull",

                fill=False,

                edgecolor="black",

                linewidth=1.8,

                zorder=5
            )
        )


        hull_created = True


    except Exception as exc:

        print(
            "\nConvex hull could not be created:"
        )

        print(
            exc
        )


else:

    print(
        "\nConvex hull skipped:"
    )

    print(
        "Object has fewer than 3 unique XY points."
    )


# ------------------------------------------------------------
# 13. VISUALIZATION COREPOINT LOCATION
# ------------------------------------------------------------

ax2.scatter(

    cloud[
        visualization_cp_idx,
        0
    ],

    cloud[
        visualization_cp_idx,
        1
    ],

    marker="*",

    color="black",

    s=140,

    label=(
        f"Visualization corepoint "
        f"{visualization_cp_idx}"
    ),

    zorder=8
)


# ------------------------------------------------------------
# 14. ACTUAL GENERATING-SEED LOCATION
# ------------------------------------------------------------

if object_seed is not None:

    ax2.scatter(

        cloud[
            seed_cp_idx,
            0
        ],

        cloud[
            seed_cp_idx,
            1
        ],

        marker="o",

        facecolors="none",

        edgecolors="blue",

        linewidths=2,

        s=120,

        label=(
            f"Generating seed "
            f"{seed_cp_idx}"
        ),

        zorder=9
    )


# ------------------------------------------------------------
# 15. PLOT LABELS
# ------------------------------------------------------------

time_delta = (

    timestamps_analysis[
        epoch_of_interest
    ]

    -

    timestamps_analysis[
        start_epoch
    ]
)


ax2.set_title(
    "Original 4DOBC spatial extent\n"
    f"Time since object start: {time_delta}"
)


ax2.set_xlabel(
    "X [m]"
)


ax2.set_ylabel(
    "Y [m]"
)


ax2.legend(
    loc="upper right",
    fontsize=8
)


plt.tight_layout()

plt.show()


# ============================================================
# 16. SAVE OBJECT-VISUALIZATION SUMMARY
# ============================================================

object_visualization_summary = pd.DataFrame([
    {

        "method":
            "4DOBC",

        "object_number_user":
            OBJECT_NUMBER,

        "object_index_python":
            selected_object_index,

        "generating_seed_available":
            object_seed is not None,

        "generating_seed_corepoint_index_python":
            (
                seed_cp_idx
                if
                object_seed is not None
                else
                np.nan
            ),

        "generating_seed_corepoint_number_user":
            (
                seed_cp_idx + 1
                if
                object_seed is not None
                else
                np.nan
            ),

        "generating_seed_start_epoch":
            seed_start_epoch,

        "generating_seed_end_epoch":
            seed_end_epoch,

        "generating_seed_duration_epochs":
            seed_duration_epochs,

        "visualization_corepoint_index_python":
            visualization_cp_idx,

        "visualization_corepoint_number_user":
            visualization_cp_idx + 1,

        "object_contains_visualization_corepoint":
            (
                visualization_cp_idx
                in object_index_set
            ),

        "start_epoch":
            start_epoch,

        "end_epoch":
            end_epoch,

        "duration_epochs":
            duration_epochs,

        "epoch_of_interest":
            epoch_of_interest,

        "n_corepoints":
            len(
                object_indices
            ),

        "n_unique_xy_points":
            unique_object_xy.shape[0],

        "convex_hull_created":
            hull_created,

        "mean_change_magnitude":
            float(
                np.nanmean(
                    magnitudes_of_interest[
                        object_indices
                    ]
                )
            ),

        "max_abs_change_magnitude":
            float(
                np.nanmax(
                    np.abs(
                        magnitudes_of_interest[
                            object_indices
                        ]
                    )
                )
            ),
    }
])


display(
    object_visualization_summary
)


object_visualization_summary.to_csv(

    TABLES_DIR
    / "original_4dobc_selected_object_visualization_summary.csv",

    index=False
)


# ============================================================
# 17. FINAL STATUS
# ============================================================

print(
    "\n=========================================="
)

print(
    "CELL 10 COMPLETE"
)

print(
    "=========================================="
)


print(
    "Method: 4DOBC"
)


print(
    "Object number:",
    OBJECT_NUMBER
)


print(
    "Generating seed:",
    (
        seed_cp_idx
        if
        object_seed is not None
        else
        "not available"
    )
)


print(
    "Visualization corepoint:",
    visualization_cp_idx
)


print(
    "Convex hull created:",
    hull_created
)

In [ ]:
# ============================================================
# CELL 11 — RUN KALMAN FILTERING OF M3C2 TIME SERIES
# ============================================================

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from py4dgeo.m3c2_kalman import kalman_filter_m3c2


# ------------------------------------------------------------
# 1. USER SETTINGS
# ------------------------------------------------------------

KALMAN_MODE = "range"
# options:
#   "single"
#   "range"
#   "full"

KALMAN_COREPOINT_NUMBER = SEED_COREPOINT_NUMBER

KALMAN_RANGE_START = 15000
KALMAN_RANGE_END = 15999

PROCESS_SIGMA = 0.01
MIN_SIGMA_OBS = 0.005

PLOT_COREPOINT_INDEX = 15162


# ------------------------------------------------------------
# 2. SELECT COREPOINTS
# ------------------------------------------------------------

n_corepoints, n_epochs = (
    analysis.distances.shape
)


if KALMAN_MODE == "single":

    if KALMAN_COREPOINT_NUMBER <= 0:
        raise ValueError(
            "Corepoint numbering starts from 1, not 0."
        )

    cp_idx = (
        KALMAN_COREPOINT_NUMBER - 1
    )

    if cp_idx >= n_corepoints:
        raise ValueError(
            "Selected corepoint exceeds the available "
            "corepoint count."
        )

    kalman_indices = [
        cp_idx
    ]


elif KALMAN_MODE == "range":

    if (
        KALMAN_RANGE_START <= 0
        or KALMAN_RANGE_END <= 0
    ):
        raise ValueError(
            "Corepoint numbering starts from 1, not 0."
        )

    start_idx = (
        KALMAN_RANGE_START - 1
    )

    end_idx = min(
        KALMAN_RANGE_END,
        n_corepoints
    )

    kalman_indices = list(
        range(
            start_idx,
            end_idx
        )
    )


elif KALMAN_MODE == "full":

    kalman_indices = list(
        range(
            n_corepoints
        )
    )


else:

    raise ValueError(
        "KALMAN_MODE must be "
        "'single', 'range', or 'full'."
    )


print(
    "Kalman mode:",
    KALMAN_MODE
)

print(
    "Number of corepoints to process:",
    len(kalman_indices)
)


# ------------------------------------------------------------
# 3. PROGRESS CALLBACK
# ------------------------------------------------------------

def report_kalman_progress(
    processed,
    total
):

    if (
        processed % 100 == 0
        or processed == total
    ):

        print(
            f"Processed "
            f"{processed}/{total} "
            "corepoints"
        )


# ------------------------------------------------------------
# 4. RUN KALMAN FILTERING
# ------------------------------------------------------------

print(
    "\nRunning Kalman filtering..."
)

start_time = time.time()


kalman_result = kalman_filter_m3c2(
    distances=analysis.distances,
    lodetection=analysis.uncertainties[
        "lodetection"
    ],
    timedeltas=analysis.timedeltas,
    process_sigma=PROCESS_SIGMA,
    min_sigma_obs=MIN_SIGMA_OBS,
    corepoint_indices=kalman_indices,
    progress_callback=report_kalman_progress
)


elapsed = (
    time.time()
    - start_time
)


# ------------------------------------------------------------
# 5. CREATE EXISTING NOTEBOOK VARIABLES
# ------------------------------------------------------------


kalman_change = (
    kalman_result["change"]
)

kalman_rate = (
    kalman_result["rate"]
)

kalman_sigma_change = (
    kalman_result["sigma_change"]
)

kalman_sigma_rate = (
    kalman_result["sigma_rate"]
)

sigma_obs_series = (
    kalman_result[
        "observation_sigma"
    ]
)

dt_series = (
    kalman_result[
        "dt_days"
    ]
)


print(
    "\nKalman filtering complete."
)

print(
    f"Elapsed time: "
    f"{elapsed:.2f} seconds"
)


# ------------------------------------------------------------
# 6. VISUALIZATION CHECK
# ------------------------------------------------------------

if (
    PLOT_COREPOINT_INDEX
    not in kalman_indices
):

    raise ValueError(
        f"Python corepoint index "
        f"{PLOT_COREPOINT_INDEX} "
        "was not processed."
    )


plot_idx = (
    PLOT_COREPOINT_INDEX
)


# ------------------------------------------------------------
# 7. RAW M3C2 VS KF-MAG
# ------------------------------------------------------------

plt.figure(
    figsize=(10, 5)
)


plt.scatter(
    timestamps_analysis,
    analysis.distances[
        plot_idx
    ],
    s=8,
    color="black",
    label="Raw M3C2"
)


plt.plot(
    timestamps_analysis,
    kalman_change[
        plot_idx
    ],
    color="blue",
    linewidth=2,
    label="Kalman-smoothed change"
)


plt.fill_between(
    timestamps_analysis,

    kalman_change[
        plot_idx
    ]
    - 1.96
    * kalman_sigma_change[
        plot_idx
    ],

    kalman_change[
        plot_idx
    ]
    + 1.96
    * kalman_sigma_change[
        plot_idx
    ],

    alpha=0.25,
    label="Approx. 95% Kalman interval"
)


dtFmt = mdates.DateFormatter(
    "%b-%d"
)

plt.gca().xaxis.set_major_formatter(
    dtFmt
)

plt.xlabel(
    "Date"
)

plt.tick_params(
    axis="x",
    rotation=15
)

plt.ylabel(
    "Distance [m]"
)

plt.title(
    f"Kalman smoothing at corepoint "
    f"Python index {plot_idx}"
)

plt.grid(
    True
)

plt.legend()

plt.tight_layout()

plt.show()


# ------------------------------------------------------------
# 8. KF-RATE VISUALIZATION
# ------------------------------------------------------------

plt.figure(
    figsize=(10, 4)
)


plt.plot(
    timestamps_analysis,
    kalman_rate[
        plot_idx
    ],
    color="red",
    linewidth=2,
    label="Kalman change rate"
)


plt.fill_between(
    timestamps_analysis,

    kalman_rate[
        plot_idx
    ]
    - 1.96
    * kalman_sigma_rate[
        plot_idx
    ],

    kalman_rate[
        plot_idx
    ]
    + 1.96
    * kalman_sigma_rate[
        plot_idx
    ],

    alpha=0.25,
    label="Approx. 95% rate interval"
)


plt.axhline(
    0,
    color="black",
    linewidth=0.8
)


plt.gca().xaxis.set_major_formatter(
    dtFmt
)

plt.xlabel(
    "Date"
)

plt.tick_params(
    axis="x",
    rotation=15
)

plt.ylabel(
    "Change rate [m/day]"
)

plt.title(
    f"Kalman-estimated change rate "
    f"at corepoint Python index "
    f"{plot_idx}"
)

plt.grid(
    True
)

plt.legend()

plt.tight_layout()

plt.show()


# ------------------------------------------------------------
# 9. SAVE RESULTS
# ------------------------------------------------------------

np.save(
    RESULTS_DIR / "kalman_change.npy",
    kalman_change
)

np.save(
    RESULTS_DIR / "kalman_rate.npy",
    kalman_rate
)

np.save(
    RESULTS_DIR
    / "kalman_sigma_change.npy",
    kalman_sigma_change
)

np.save(
    RESULTS_DIR
    / "kalman_sigma_rate.npy",
    kalman_sigma_rate
)


kalman_config = pd.DataFrame([
    {
        "kalman_mode": (
            KALMAN_MODE
        ),

        "process_sigma": (
            PROCESS_SIGMA
        ),

        "min_sigma_obs": (
            MIN_SIGMA_OBS
        ),

        "n_processed_corepoints": (
            len(
                kalman_indices
            )
        ),

        "elapsed_seconds": (
            elapsed
        )
    }
])


display(
    kalman_config
)


kalman_config.to_csv(
    TABLES_DIR
    / "kalman_filtering_config.csv",
    index=False
)


print(
    "\nCell 11 complete: "
    "Kalman filtering finished."
)

In [ ]:
# ============================================================
# CELL 12 — KF-MAG AND KF-RATE SEED DETECTION
# ============================================================


# ------------------------------------------------------------
# 0. IMPORTS
# ------------------------------------------------------------

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from py4dgeo.m3c2_kalman import (
    detect_magnitude_seeds,
    detect_rate_seeds
)


# ------------------------------------------------------------
# 1. USER SETTINGS
# ------------------------------------------------------------

KALMAN_SEED_MODE = "processed"
# options:
#   "processed"
#   "single"
#   "range"


KALMAN_SEED_COREPOINT_NUMBER = 15163

KALMAN_SEED_RANGE_START = 15000
KALMAN_SEED_RANGE_END = 15999


SEED_DIRECTION = "both"
# options:
#   "positive"
#   "negative"
#   "both"


Z_THRESHOLD = 1.96

MIN_SEED_DURATION = 10

MIN_SEED_MAGNITUDE = 0.05


# ------------------------------------------------------------
# KF-Mag formation-start configuration
# ------------------------------------------------------------

EXTEND_TO_FORMATION_START = True

FORMATION_BASELINE_TOLERANCE = 0.02

FORMATION_BASELINE_SIGMA_FACTOR = 1.0

MAX_FORMATION_EXTENSION_EPOCHS = 45

MIN_EXTENSION_SLOPE = 0.001


# ------------------------------------------------------------
# Visualization/output configuration
# ------------------------------------------------------------

PLOT_COREPOINT_INDEX = 15162

SAVE_CELL12_FIGURES = True


cell12_fig_dir = (
    FIGURES_DIR
    / "cell12_kf_seed_detection"
)

cell12_fig_dir.mkdir(
    exist_ok=True,
    parents=True
)


# ------------------------------------------------------------
# 2. CHECK REQUIRED VARIABLES
# ------------------------------------------------------------

required_objects = [
    "kalman_change",
    "kalman_rate",
    "kalman_sigma_change",
    "kalman_sigma_rate",
    "timestamps_analysis",
    "analysis",
    "kalman_indices"
]


for required_name in required_objects:

    if required_name not in globals():

        raise ValueError(
            f"{required_name} is not defined. "
            "Please run Cell 11 first."
        )


n_corepoints, n_epochs = (
    kalman_change.shape
)


# ------------------------------------------------------------
# 3. SELECT COREPOINTS FOR SEED DETECTION
# ------------------------------------------------------------

if KALMAN_SEED_MODE == "processed":

    seed_detection_indices = list(
        kalman_indices
    )


elif KALMAN_SEED_MODE == "single":

    if (
        KALMAN_SEED_COREPOINT_NUMBER
        <= 0
    ):

        raise ValueError(
            "Corepoint numbering starts from 1, not 0."
        )

    cp_idx = (
        KALMAN_SEED_COREPOINT_NUMBER
        - 1
    )

    if cp_idx >= n_corepoints:

        raise ValueError(
            "Selected seed-detection corepoint exceeds "
            "the available corepoint count."
        )

    seed_detection_indices = [
        cp_idx
    ]


elif KALMAN_SEED_MODE == "range":

    if (
        KALMAN_SEED_RANGE_START <= 0
        or KALMAN_SEED_RANGE_END <= 0
    ):

        raise ValueError(
            "Corepoint numbering starts from 1, not 0."
        )

    start_idx = (
        KALMAN_SEED_RANGE_START
        - 1
    )

    end_idx = min(
        KALMAN_SEED_RANGE_END,
        n_corepoints
    )

    seed_detection_indices = list(
        range(
            start_idx,
            end_idx
        )
    )


else:

    raise ValueError(
        "KALMAN_SEED_MODE must be "
        "'processed', 'single', or 'range'."
    )


print(
    "Seed detection mode:",
    KALMAN_SEED_MODE
)

print(
    "Seed direction:",
    SEED_DIRECTION
)

print(
    "Number of corepoints checked:",
    len(seed_detection_indices)
)

print(
    "Z threshold:",
    Z_THRESHOLD
)

print(
    "Minimum seed duration:",
    MIN_SEED_DURATION,
    "epochs"
)

print(
    "Minimum seed magnitude:",
    MIN_SEED_MAGNITUDE,
    "m"
)


# ------------------------------------------------------------
# 4. RUN KF-MAG SEED DETECTION
# ------------------------------------------------------------

print(
    "\n=========================================="
)

print(
    "Running KF-Mag seed detection"
)

print(
    "=========================================="
)


kf_mag_seed_table = (
    detect_magnitude_seeds(
        change=kalman_change,
        rate=kalman_rate,
        sigma_change=(
            kalman_sigma_change
        ),
        sigma_rate=(
            kalman_sigma_rate
        ),
        timestamps=(
            timestamps_analysis
        ),
        corepoint_indices=(
            seed_detection_indices
        ),
        direction=(
            SEED_DIRECTION
        ),
        z_threshold=(
            Z_THRESHOLD
        ),
        min_duration=(
            MIN_SEED_DURATION
        ),
        min_magnitude=(
            MIN_SEED_MAGNITUDE
        ),
        extend_to_formation_start=(
            EXTEND_TO_FORMATION_START
        ),
        formation_baseline_tolerance=(
            FORMATION_BASELINE_TOLERANCE
        ),
        formation_baseline_sigma_factor=(
            FORMATION_BASELINE_SIGMA_FACTOR
        ),
        max_formation_extension_epochs=(
            MAX_FORMATION_EXTENSION_EPOCHS
        ),
        min_extension_slope=(
            MIN_EXTENSION_SLOPE
        )
    )
)


print(
    "KF-Mag seed intervals:",
    len(
        kf_mag_seed_table
    )
)


if len(
    kf_mag_seed_table
) > 0:

    display(
        kf_mag_seed_table.head()
    )

else:

    display(
        kf_mag_seed_table
    )


# ------------------------------------------------------------
# 5. RUN KF-RATE SEED DETECTION
# ------------------------------------------------------------

print(
    "\n=========================================="
)

print(
    "Running KF-Rate seed detection"
)

print(
    "=========================================="
)


kf_rate_seed_table = (
    detect_rate_seeds(
        change=kalman_change,
        rate=kalman_rate,
        sigma_change=(
            kalman_sigma_change
        ),
        sigma_rate=(
            kalman_sigma_rate
        ),
        timestamps=(
            timestamps_analysis
        ),
        corepoint_indices=(
            seed_detection_indices
        ),
        direction=(
            SEED_DIRECTION
        ),
        z_threshold=(
            Z_THRESHOLD
        ),
        min_duration=(
            MIN_SEED_DURATION
        ),
        min_magnitude=(
            MIN_SEED_MAGNITUDE
        )
    )
)


print(
    "KF-Rate seed intervals:",
    len(
        kf_rate_seed_table
    )
)


if len(
    kf_rate_seed_table
) > 0:

    display(
        kf_rate_seed_table.head()
    )

else:

    display(
        kf_rate_seed_table
    )


# ------------------------------------------------------------
# 6. CREATE COMPATIBILITY VARIABLES
# ------------------------------------------------------------
# These preserve the existing Cell 12B / Cell 13 interface while
# we continue restructuring the notebook.

kalman_seed_table_magnitude = (
    kf_mag_seed_table.copy()
)

kalman_seed_table_rate = (
    kf_rate_seed_table.copy()
)


kalman_seed_tables = {
    "magnitude": (
        kalman_seed_table_magnitude
    ),
    "rate": (
        kalman_seed_table_rate
    )
}


# ------------------------------------------------------------
# 7. SAVE SEED TABLES
# ------------------------------------------------------------

kf_mag_seed_table.to_csv(
    TABLES_DIR
    / "kf_mag_seed_table.csv",
    index=False
)


kf_rate_seed_table.to_csv(
    TABLES_DIR
    / "kf_rate_seed_table.csv",
    index=False
)


# Backward-compatible file names for now.
kalman_seed_table_magnitude.to_csv(
    TABLES_DIR
    / "kalman_seed_table_magnitude.csv",
    index=False
)


kalman_seed_table_rate.to_csv(
    TABLES_DIR
    / "kalman_seed_table_rate.csv",
    index=False
)


# ------------------------------------------------------------
# 8. SEED VISUALIZATION FUNCTION
# ------------------------------------------------------------

def plot_kf_seed_detection(
    seed_table,
    method_name,
    detection_signal
):

    if (
        PLOT_COREPOINT_INDEX
        in seed_detection_indices
    ):

        plot_cp_idx = (
            PLOT_COREPOINT_INDEX
        )

    elif len(seed_table) > 0:

        plot_cp_idx = int(
            seed_table.iloc[0][
                "corepoint_index_python"
            ]
        )

        print(
            f"{method_name}: preferred visualization "
            "corepoint was not processed."
        )

        print(
            "Using first detected seed corepoint:",
            plot_cp_idx
        )

    else:

        plot_cp_idx = int(
            seed_detection_indices[0]
        )

        print(
            f"{method_name}: no seeds detected. "
            "Using first processed corepoint:",
            plot_cp_idx
        )


    plot_seed_table = (
        seed_table[
            seed_table[
                "corepoint_index_python"
            ]
            == plot_cp_idx
        ]
        .copy()
    )


    date_formatter = (
        mdates.DateFormatter(
            "%b-%d"
        )
    )


    fig, (
        ax1,
        ax2
    ) = plt.subplots(
        2,
        1,
        figsize=(12, 8),
        sharex=True
    )


    # --------------------------------------------------------
    # TOP — M3C2 + Kalman change + seed intervals
    # --------------------------------------------------------

    ax1.scatter(
        timestamps_analysis,
        analysis.distances[
            plot_cp_idx
        ],
        s=8,
        color="black",
        alpha=0.5,
        label="Raw M3C2"
    )


    ax1.plot(
        timestamps_analysis,
        kalman_change[
            plot_cp_idx
        ],
        color="blue",
        linewidth=2,
        label="Kalman-smoothed change"
    )


    upper_change = (
        Z_THRESHOLD
        * kalman_sigma_change[
            plot_cp_idx
        ]
    )

    lower_change = (
        -Z_THRESHOLD
        * kalman_sigma_change[
            plot_cp_idx
        ]
    )


    ax1.fill_between(
        timestamps_analysis,
        lower_change,
        upper_change,
        color="blue",
        alpha=0.15,
        label=(
            f"±{Z_THRESHOLD:.2f}σ "
            "change uncertainty"
        )
    )


    if len(
        plot_seed_table
    ) > 0:

        for _, row in (
            plot_seed_table.iterrows()
        ):

            seed_start = int(
                row["start_epoch"]
            )

            seed_end = int(
                row["end_epoch"]
            )

            ax1.plot(
                timestamps_analysis[
                    seed_start:
                    seed_end + 1
                ],
                kalman_change[
                    plot_cp_idx,
                    seed_start:
                    seed_end + 1
                ],
                linewidth=4,
                label=(
                    f"{row['direction']} seed "
                    f"{int(row['seed_rank'])}"
                )
            )


            if (
                detection_signal
                == "magnitude"
                and
                bool(
                    row[
                        "formation_start_extension_applied"
                    ]
                )
            ):

                raw_start = int(
                    row[
                        "raw_significant_start_epoch"
                    ]
                )

                ax1.axvline(
                    timestamps_analysis[
                        raw_start
                    ],
                    color="grey",
                    linestyle=":",
                    linewidth=1
                )


    ax1.set_ylabel(
        "Change [m]"
    )

    ax1.set_title(
        f"(a) {method_name} seed intervals\n"
        f"Corepoint Python index "
        f"{plot_cp_idx}"
    )

    ax1.grid(
        True
    )

    ax1.legend(
        loc="upper right",
        fontsize=8
    )


    # --------------------------------------------------------
    # BOTTOM — detection signal significance
    # --------------------------------------------------------

    if detection_signal == "magnitude":

        signal = (
            kalman_change[
                plot_cp_idx
            ]
        )

        sigma = (
            kalman_sigma_change[
                plot_cp_idx
            ]
        )

        ylabel = (
            "Change [m]"
        )

        signal_title = (
            "Kalman change magnitude"
        )

        signal_color = (
            "blue"
        )


    elif detection_signal == "rate":

        signal = (
            kalman_rate[
                plot_cp_idx
            ]
        )

        sigma = (
            kalman_sigma_rate[
                plot_cp_idx
            ]
        )

        ylabel = (
            "Change rate [m/day]"
        )

        signal_title = (
            "Kalman change rate"
        )

        signal_color = (
            "red"
        )


    else:

        raise ValueError(
            "detection_signal must be "
            "'magnitude' or 'rate'."
        )


    upper_lod = (
        Z_THRESHOLD
        * sigma
    )

    lower_lod = (
        -Z_THRESHOLD
        * sigma
    )


    ax2.plot(
        timestamps_analysis,
        signal,
        color=signal_color,
        linewidth=2,
        label=signal_title
    )


    ax2.plot(
        timestamps_analysis,
        upper_lod,
        "--",
        color="grey",
        label="+ LoD"
    )


    ax2.plot(
        timestamps_analysis,
        lower_lod,
        "--",
        color="grey",
        label="- LoD"
    )


    ax2.axhline(
        0,
        color="black",
        linewidth=0.8
    )


    ax2.set_ylabel(
        ylabel
    )

    ax2.set_xlabel(
        "Date"
    )

    ax2.set_title(
        f"(b) {signal_title} significance"
    )

    ax2.grid(
        True
    )

    ax2.legend(
        loc="upper right",
        fontsize=8
    )

    ax2.xaxis.set_major_formatter(
        date_formatter
    )

    ax2.tick_params(
        axis="x",
        rotation=15
    )


    plt.tight_layout()


    figure_name = (
        f"{method_name.lower().replace('-', '_')}_"
        f"seed_detection_corepoint_"
        f"{plot_cp_idx}.png"
    )


    figure_path = (
        cell12_fig_dir
        / figure_name
    )


    if SAVE_CELL12_FIGURES:

        plt.savefig(
            figure_path,
            dpi=300,
            bbox_inches="tight",
            facecolor="white"
        )

        print(
            f"{method_name} figure saved:"
        )

        print(
            figure_path
        )


    plt.show()

    return figure_path


# ------------------------------------------------------------
# 9. CREATE FIGURES
# ------------------------------------------------------------

kf_mag_figure_path = (
    plot_kf_seed_detection(
        seed_table=(
            kf_mag_seed_table
        ),
        method_name="KF-Mag",
        detection_signal="magnitude"
    )
)


kf_rate_figure_path = (
    plot_kf_seed_detection(
        seed_table=(
            kf_rate_seed_table
        ),
        method_name="KF-Rate",
        detection_signal="rate"
    )
)


# ------------------------------------------------------------
# 10. SAVE RUN CONFIGURATION
# ------------------------------------------------------------

if len(
    kf_mag_seed_table
) > 0:

    n_mag_extended = int(
        kf_mag_seed_table[
            "formation_start_extension_applied"
        ].sum()
    )

    mean_mag_extension = float(
        kf_mag_seed_table[
            "formation_extension_epochs"
        ].mean()
    )

else:

    n_mag_extended = 0

    mean_mag_extension = (
        np.nan
    )


kf_seed_detection_config = (
    pd.DataFrame([
        {
            "method": "KF-Mag",
            "seed_mode": (
                KALMAN_SEED_MODE
            ),
            "seed_direction": (
                SEED_DIRECTION
            ),
            "z_threshold": (
                Z_THRESHOLD
            ),
            "min_seed_duration_epochs": (
                MIN_SEED_DURATION
            ),
            "min_seed_magnitude": (
                MIN_SEED_MAGNITUDE
            ),
            "formation_start_extension_enabled": (
                EXTEND_TO_FORMATION_START
            ),
            "formation_baseline_tolerance": (
                FORMATION_BASELINE_TOLERANCE
            ),
            "formation_baseline_sigma_factor": (
                FORMATION_BASELINE_SIGMA_FACTOR
            ),
            "max_formation_extension_epochs": (
                MAX_FORMATION_EXTENSION_EPOCHS
            ),
            "min_extension_slope": (
                MIN_EXTENSION_SLOPE
            ),
            "n_checked_corepoints": (
                len(
                    seed_detection_indices
                )
            ),
            "n_detected_seeds": (
                len(
                    kf_mag_seed_table
                )
            ),
            "n_extended_seed_intervals": (
                n_mag_extended
            ),
            "mean_extension_epochs": (
                mean_mag_extension
            ),
            "plot_corepoint_index": (
                PLOT_COREPOINT_INDEX
            ),
            "figure_path": str(
                kf_mag_figure_path
            )
        },

        {
            "method": "KF-Rate",
            "seed_mode": (
                KALMAN_SEED_MODE
            ),
            "seed_direction": (
                SEED_DIRECTION
            ),
            "z_threshold": (
                Z_THRESHOLD
            ),
            "min_seed_duration_epochs": (
                MIN_SEED_DURATION
            ),
            "min_seed_magnitude": (
                MIN_SEED_MAGNITUDE
            ),
            "formation_start_extension_enabled": (
                False
            ),
            "formation_baseline_tolerance": (
                np.nan
            ),
            "formation_baseline_sigma_factor": (
                np.nan
            ),
            "max_formation_extension_epochs": (
                np.nan
            ),
            "min_extension_slope": (
                np.nan
            ),
            "n_checked_corepoints": (
                len(
                    seed_detection_indices
                )
            ),
            "n_detected_seeds": (
                len(
                    kf_rate_seed_table
                )
            ),
            "n_extended_seed_intervals": (
                0
            ),
            "mean_extension_epochs": (
                np.nan
            ),
            "plot_corepoint_index": (
                PLOT_COREPOINT_INDEX
            ),
            "figure_path": str(
                kf_rate_figure_path
            )
        }
    ])
)


display(
    kf_seed_detection_config
)


kf_seed_detection_config.to_csv(
    TABLES_DIR
    / "kf_seed_detection_config.csv",
    index=False
)


# Backward compatibility
kalman_seed_config = (
    kf_seed_detection_config.copy()
)


print(
    "\nCell 12 complete."
)

print(
    "KF-Mag detected seeds:",
    len(
        kf_mag_seed_table
    )
)

print(
    "KF-Rate detected seeds:",
    len(
        kf_rate_seed_table
    )
)

In [ ]:
# ============================================================
# CELL 12B — CONNECT KALMAN SEEDS TO PY4DGEO 4D-OBC
# ============================================================

# The reusable implementation is located in:
#
#   src/py4dgeo/segmentation_kalman.py
#
# Existing py4dgeo region growing is retained.
# Only find_seedpoints() is replaced by Kalman-derived seeds.
# ============================================================


# ------------------------------------------------------------
# 1. IMPORTS
# ------------------------------------------------------------

from py4dgeo.segmentation_kalman import (
    prepare_seed_table_for_py4dgeo,
    create_kalman_region_growing,
)


# ------------------------------------------------------------
# 2. REQUIRED CHECKS
# ------------------------------------------------------------

required_objects = [
    "analysis",
    "kalman_seed_tables",
    "kalman_seed_table_magnitude",
    "kalman_seed_table_rate",
]


for obj in required_objects:

    if obj not in globals():

        raise ValueError(
            f"{obj} is not defined. "
            "Please run Cell 12 first."
        )


# ------------------------------------------------------------
# 3. VALIDATE KF-MAG AND KF-RATE SEED CONVERSION
# ------------------------------------------------------------

print(
    "\nCell 12B validation"
)

print(
    "-------------------"
)


prepared_kalman_seed_tables = {}


for signal_name, table in (
    kalman_seed_tables.items()
):

    prepared = (
        prepare_seed_table_for_py4dgeo(
            seed_table=table,
            seed_mode="all",
            top_n=20,
            use_support_interval=True,
            ranking_columns=[
                "event_magnitude",
                "duration_epochs",
            ],
        )
    )

    prepared_kalman_seed_tables[
        signal_name
    ] = prepared


    print(
        f"\n{signal_name}:"
    )

    print(
        "  original seed intervals:",
        len(table)
    )

    print(
        "  py4dgeo-compatible seeds:",
        len(prepared)
    )


    display(
        prepared[
            [
                "corepoint_index_python",
                "start_epoch",
                "end_epoch",
                "py4dgeo_seed_start_epoch",
                "py4dgeo_seed_end_epoch",
                "py4dgeo_seed_duration_epochs",
                "event_magnitude",
                "py4dgeo_custom_seed_rank",
            ]
        ].head()
    )


# ------------------------------------------------------------
# 4. CREATE KF-MAG REGION-GROWING ALGORITHM
# ------------------------------------------------------------

kf_mag_region_growing = (
    create_kalman_region_growing(
        seed_table=(
            kalman_seed_table_magnitude
        ),
        method_label="KF-Mag",
        seed_mode="all",
        top_n=20,
        use_support_interval=True,
        ranking_columns=[
            "event_magnitude",
            "duration_epochs",
        ],
        window_width=14,
        minperiod=2,
        height_threshold=0.05,
        neighborhood_radius=1.0,
        min_segments=10,
        thresholds=[
            0.3,
            0.4,
            0.5,
            0.6,
            0.7,
            0.8,
            0.9,
        ],
    )
)


# ------------------------------------------------------------
# 5. CREATE KF-RATE REGION-GROWING ALGORITHM
# ------------------------------------------------------------

kf_rate_region_growing = (
    create_kalman_region_growing(
        seed_table=(
            kalman_seed_table_rate
        ),
        method_label="KF-Rate",
        seed_mode="all",
        top_n=20,
        use_support_interval=True,
        ranking_columns=[
            "event_magnitude",
            "duration_epochs",
        ],
        window_width=14,
        minperiod=2,
        height_threshold=0.05,
        neighborhood_radius=1.0,
        min_segments=10,
        thresholds=[
            0.3,
            0.4,
            0.5,
            0.6,
            0.7,
            0.8,
            0.9,
        ],
    )
)


# ------------------------------------------------------------
# 6. VERIFY find_seedpoints()
# ------------------------------------------------------------

kf_mag_py4dgeo_seeds = (
    kf_mag_region_growing.find_seedpoints()
)

kf_rate_py4dgeo_seeds = (
    kf_rate_region_growing.find_seedpoints()
)


print(
    "\nKF-Mag RegionGrowingSeed objects:",
    len(kf_mag_py4dgeo_seeds)
)

print(
    "KF-Rate RegionGrowingSeed objects:",
    len(kf_rate_py4dgeo_seeds)
)


# ------------------------------------------------------------
# 7. BACKWARD-COMPATIBILITY ALIAS
# ------------------------------------------------------------
# Temporary alias while Cell 13 is being restructured.

create_custom_4dobc_algorithm = (
    create_kalman_region_growing
)


print(
    "\nCell 12B complete."
)

print(
    "Kalman-derived seeds are ready for "
    "existing py4dgeo 4D-OBC region growing."
)

In [ ]:
# ============================================================
# CELL 13 — KF-MAG AND KF-RATE 4D-OBC EXTRACTION
# USING EXISTING PY4DGEO REGION GROWING
# ============================================================



# ------------------------------------------------------------
# 0. IMPORTS
# ------------------------------------------------------------

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from py4dgeo.segmentation_kalman import (
    create_kalman_region_growing
)


# ------------------------------------------------------------
# 1. USER SETTINGS
# ------------------------------------------------------------

REFERENCE_COREPOINT_INDEX = 15162
VISUALIZATION_COREPOINT_INDEX = 15162


# ------------------------------------------------------------
# Seed selection
# ------------------------------------------------------------
# "all"
#     Use every Kalman seed detected in Cell 12.
#
# "top_n"
#     Use only the highest-ranked Kalman seeds.

KF_4DOBC_SEED_MODE = "all"

TOP_N_KF_SEEDS = 20


# ------------------------------------------------------------
# Region-growing parameters
# ------------------------------------------------------------
# These remain the existing py4dgeo region-growing parameters.

WINDOW_WIDTH = 14

MINPERIOD = 2

HEIGHT_THRESHOLD = 0.05

NEIGHBORHOOD_RADIUS = 1.0

MIN_SEGMENTS = 10

THRESHOLDS = [
    0.3,
    0.4,
    0.5,
    0.6,
    0.7,
    0.8,
    0.9
]


# ------------------------------------------------------------
# Methods to run
# ------------------------------------------------------------

KF_METHODS = [
    "KF-Mag",
    "KF-Rate"
]


# ------------------------------------------------------------
# Active method for temporary downstream compatibility
# ------------------------------------------------------------

ACTIVE_KF_METHOD_FOR_DOWNSTREAM = "KF-Mag"


# ------------------------------------------------------------
# 2. CHECK REQUIRED VARIABLES
# ------------------------------------------------------------

required_objects = [
    "analysis",
    "timestamps_analysis",
    "kalman_seed_table_magnitude",
    "kalman_seed_table_rate"
]


for obj_name in required_objects:

    if obj_name not in globals():

        raise ValueError(
            f"{obj_name} is not defined. "
            "Please run Cells 11, 12, and 12B first."
        )


# ------------------------------------------------------------
# 3. CREATE METHOD -> SEED TABLE MAPPING
# ------------------------------------------------------------

kf_seed_tables = {

    "KF-Mag":
        kalman_seed_table_magnitude,

    "KF-Rate":
        kalman_seed_table_rate
}


# ------------------------------------------------------------
# 4. VALIDATE SETTINGS
# ------------------------------------------------------------

if KF_4DOBC_SEED_MODE not in [
    "all",
    "top_n"
]:

    raise ValueError(
        "KF_4DOBC_SEED_MODE must be "
        "'all' or 'top_n'."
    )


if (
    ACTIVE_KF_METHOD_FOR_DOWNSTREAM
    not in KF_METHODS
):

    raise ValueError(
        "ACTIVE_KF_METHOD_FOR_DOWNSTREAM "
        "must be 'KF-Mag' or 'KF-Rate'."
    )


# ------------------------------------------------------------
# 5. OBJECT SUMMARY FUNCTION
# ------------------------------------------------------------

def summarize_kf_4dobc_objects(
    method_name,
    objects
):
    """
    Create a summary table for extracted KF-Mag or KF-Rate
    4D-OBC objects.
    """

    records = []

    for object_number, obj in enumerate(
        objects,
        start=1
    ):

        object_indices = np.asarray(
            obj.indices,
            dtype=int
        )

        object_index_set = set(
            object_indices.tolist()
        )

        start_epoch = int(
            obj.start_epoch
        )

        end_epoch = int(
            obj.end_epoch
        )

        record = {

            "method":
                method_name,

            "object_number_user":
                object_number,

            "n_corepoints":
                len(object_indices),

            "start_epoch":
                start_epoch,

            "end_epoch":
                end_epoch,

            "duration_epochs":
                end_epoch
                - start_epoch
                + 1,

            "start_time":
                timestamps_analysis[
                    start_epoch
                ],

            "end_time":
                timestamps_analysis[
                    end_epoch
                ],

            "reference_corepoint_index_python":
                REFERENCE_COREPOINT_INDEX,

            "visualization_corepoint_index_python":
                VISUALIZATION_COREPOINT_INDEX,

            "object_contains_reference_corepoint":
                (
                    REFERENCE_COREPOINT_INDEX
                    in object_index_set
                ),

            "object_contains_visualization_corepoint":
                (
                    VISUALIZATION_COREPOINT_INDEX
                    in object_index_set
                )
        }


        # ----------------------------------------------------
        # Store generating py4dgeo seed when available
        # ----------------------------------------------------

        if hasattr(
            obj,
            "seed"
        ):

            seed = obj.seed

            if seed is not None:

                record[
                    "object_seed_corepoint_index_python"
                ] = int(
                    seed.index
                )

                record[
                    "object_seed_corepoint_number_user"
                ] = int(
                    seed.index + 1
                )

                record[
                    "object_seed_start_epoch"
                ] = int(
                    seed.start_epoch
                )

                record[
                    "object_seed_end_epoch"
                ] = int(
                    seed.end_epoch
                )

                record[
                    "object_seed_duration_epochs"
                ] = int(
                    seed.end_epoch
                    - seed.start_epoch
                    + 1
                )

            else:

                record[
                    "object_seed_corepoint_index_python"
                ] = np.nan

                record[
                    "object_seed_corepoint_number_user"
                ] = np.nan

                record[
                    "object_seed_start_epoch"
                ] = np.nan

                record[
                    "object_seed_end_epoch"
                ] = np.nan

                record[
                    "object_seed_duration_epochs"
                ] = np.nan

        else:

            record[
                "object_seed_corepoint_index_python"
            ] = np.nan

            record[
                "object_seed_corepoint_number_user"
            ] = np.nan

            record[
                "object_seed_start_epoch"
            ] = np.nan

            record[
                "object_seed_end_epoch"
            ] = np.nan

            record[
                "object_seed_duration_epochs"
            ] = np.nan


        records.append(
            record
        )


    return pd.DataFrame(
        records
    )


# ------------------------------------------------------------
# 6. RUN ONE KF 4D-OBC METHOD
# ------------------------------------------------------------

def run_kf_4dobc(
    method_name,
    seed_table
):
    """
    Run existing py4dgeo 4D-OBC region growing using externally
    supplied Kalman-derived temporal seeds.
    """

    print(
        "\n=========================================="
    )

    print(
        f"Running {method_name} 4D-OBC extraction"
    )

    print(
        "=========================================="
    )


    # --------------------------------------------------------
    # Validate seed table
    # --------------------------------------------------------

    if (
        seed_table is None
        or len(seed_table) == 0
    ):

        raise ValueError(
            f"{method_name} seed table is empty."
        )


    required_seed_columns = [
        "corepoint_index_python",
        "start_epoch",
        "end_epoch",
        "event_magnitude",
        "duration_epochs"
    ]


    missing_columns = [

        column

        for column
        in required_seed_columns

        if column
        not in seed_table.columns
    ]


    if missing_columns:

        raise ValueError(
            f"{method_name} seed table is missing: "
            + ", ".join(
                missing_columns
            )
        )


    print(
        "Input Kalman seed intervals:",
        len(seed_table)
    )


    print(
        "Unique seed corepoints:",
        seed_table[
            "corepoint_index_python"
        ].nunique()
    )


    # --------------------------------------------------------
    # Create Kalman-seeded py4dgeo algorithm
    # --------------------------------------------------------

    algorithm = (
        create_kalman_region_growing(

            seed_table=(
                seed_table
            ),

            method_label=(
                method_name
            ),

            seed_mode=(
                KF_4DOBC_SEED_MODE
            ),

            top_n=(
                TOP_N_KF_SEEDS
            ),

            use_support_interval=True,

            ranking_columns=[
                "event_magnitude",
                "duration_epochs"
            ],

            window_width=(
                WINDOW_WIDTH
            ),

            minperiod=(
                MINPERIOD
            ),

            height_threshold=(
                HEIGHT_THRESHOLD
            ),

            neighborhood_radius=(
                NEIGHBORHOOD_RADIUS
            ),

            min_segments=(
                MIN_SEGMENTS
            ),

            thresholds=(
                THRESHOLDS
            )
        )
    )


    # --------------------------------------------------------
    # Inspect prepared seeds BEFORE region growing
    # --------------------------------------------------------

    prepared_seed_table = (
        algorithm.prepared_seed_table.copy()
    )


    py4dgeo_seed_objects = (
        algorithm.find_seedpoints()
    )


    print(
        "Prepared Kalman seeds:",
        len(
            prepared_seed_table
        )
    )


    print(
        "RegionGrowingSeed objects:",
        len(
            py4dgeo_seed_objects
        )
    )


    # --------------------------------------------------------
    # Important verification
    # --------------------------------------------------------

    if (
        len(prepared_seed_table)
        != len(py4dgeo_seed_objects)
    ):

        raise RuntimeError(
            f"{method_name}: prepared seed count "
            "does not match RegionGrowingSeed count."
        )


    # --------------------------------------------------------
    # Clear previous region-growing results
    # --------------------------------------------------------

    analysis.invalidate_results(
        seeds=True,
        objects=True,
        smoothed_distances=False
    )


    # --------------------------------------------------------
    # Run existing py4dgeo region growing
    # --------------------------------------------------------

    print(
        f"\nRunning existing py4dgeo region growing "
        f"for {method_name}..."
    )


    start_time = time.time()


    objects = algorithm.run(
        analysis
    )


    elapsed = (
        time.time()
        - start_time
    )


    # --------------------------------------------------------
    # Retrieve seeds stored by analysis
    # --------------------------------------------------------

    analysis_seeds = (
        analysis.seeds
    )


    print(
        f"\n{method_name} extraction complete."
    )

    print(
        f"Elapsed time: "
        f"{elapsed:.2f} seconds"
    )

    print(
        "Kalman seeds supplied:",
        len(
            py4dgeo_seed_objects
        )
    )

    print(
        "Seeds stored by analysis:",
        len(
            analysis_seeds
        )
    )

    print(
        "Extracted 4D-OBC objects:",
        len(
            objects
        )
    )


    # --------------------------------------------------------
    # Seed summary
    # --------------------------------------------------------

    seed_records = []


    for seed_number, seed in enumerate(
        py4dgeo_seed_objects,
        start=1
    ):

        seed_records.append({

            "method":
                method_name,

            "seed_number_user":
                seed_number,

            "corepoint_index_python":
                int(
                    seed.index
                ),

            "corepoint_number_user":
                int(
                    seed.index + 1
                ),

            "start_epoch":
                int(
                    seed.start_epoch
                ),

            "end_epoch":
                int(
                    seed.end_epoch
                ),

            "duration_epochs":
                int(
                    seed.end_epoch
                    - seed.start_epoch
                    + 1
                ),

            "start_time":
                timestamps_analysis[
                    int(
                        seed.start_epoch
                    )
                ],

            "end_time":
                timestamps_analysis[
                    int(
                        seed.end_epoch
                    )
                ]
        })


    seed_summary = pd.DataFrame(
        seed_records
    )


    # --------------------------------------------------------
    # Object summary
    # --------------------------------------------------------

    object_summary = (
        summarize_kf_4dobc_objects(
            method_name=(
                method_name
            ),
            objects=(
                objects
            )
        )
    )


    # --------------------------------------------------------
    # Save tables
    # --------------------------------------------------------

    file_prefix = (
        method_name
        .lower()
        .replace(
            "-",
            "_"
        )
    )


    prepared_seed_table.to_csv(

        TABLES_DIR
        / f"{file_prefix}_prepared_seeds.csv",

        index=False
    )


    seed_summary.to_csv(

        TABLES_DIR
        / f"{file_prefix}_4dobc_seed_summary.csv",

        index=False
    )


    object_summary.to_csv(

        TABLES_DIR
        / f"{file_prefix}_4dobc_object_summary.csv",

        index=False
    )


    # --------------------------------------------------------
    # Run configuration
    # --------------------------------------------------------

    run_config = pd.DataFrame([
        {

            "method":
                method_name,

            "seed_source":
                "Kalman-derived temporal intervals",

            "region_growing":
                "existing py4dgeo RegionGrowingAlgorithm",

            "seed_mode":
                KF_4DOBC_SEED_MODE,

            "top_n_seeds":
                (
                    TOP_N_KF_SEEDS

                    if
                    KF_4DOBC_SEED_MODE
                    == "top_n"

                    else
                    np.nan
                ),

            "seed_ranking":
                "event_magnitude, duration_epochs",

            "use_support_interval":
                True,

            "temporal_post_correction":
                False,

            "object_deduplication":
                False,

            "window_width":
                WINDOW_WIDTH,

            "minperiod":
                MINPERIOD,

            "height_threshold":
                HEIGHT_THRESHOLD,

            "neighborhood_radius":
                NEIGHBORHOOD_RADIUS,

            "min_segments":
                MIN_SEGMENTS,

            "thresholds":
                str(
                    THRESHOLDS
                ),

            "n_input_kalman_seeds":
                len(
                    seed_table
                ),

            "n_prepared_kalman_seeds":
                len(
                    prepared_seed_table
                ),

            "n_region_growing_seed_objects":
                len(
                    py4dgeo_seed_objects
                ),

            "n_analysis_stored_seeds":
                len(
                    analysis_seeds
                ),

            "n_extracted_objects":
                len(
                    objects
                ),

            "elapsed_seconds":
                elapsed
        }
    ])


    run_config.to_csv(

        TABLES_DIR
        / f"{file_prefix}_4dobc_run_config.csv",

        index=False
    )


    # --------------------------------------------------------
    # Return complete result
    # --------------------------------------------------------

    return {

        "method":
            method_name,

        "input_seed_table":
            seed_table.copy(),

        "prepared_seed_table":
            prepared_seed_table,

        "region_growing_seeds":
            py4dgeo_seed_objects,

        "analysis_seeds":
            analysis_seeds,

        "objects":
            objects,

        "seed_summary":
            seed_summary,

        "object_summary":
            object_summary,

        "run_config":
            run_config,

        "elapsed_seconds":
            elapsed
    }


# ------------------------------------------------------------
# 7. RUN KF-MAG AND KF-RATE
# ------------------------------------------------------------

kf_4dobc_results = {}


for method_name in KF_METHODS:

    if (
        method_name
        not in kf_seed_tables
    ):

        raise ValueError(
            f"{method_name} is missing from "
            "kf_seed_tables."
        )


    kf_4dobc_results[
        method_name
    ] = run_kf_4dobc(

        method_name=(
            method_name
        ),

        seed_table=(
            kf_seed_tables[
                method_name
            ]
        )
    )


# ------------------------------------------------------------
# 8. CREATE CLEAN METHOD-SPECIFIC ALIASES
# ------------------------------------------------------------

# KF-Mag

kf_mag_objects = (
    kf_4dobc_results[
        "KF-Mag"
    ][
        "objects"
    ]
)


kf_mag_seeds = (
    kf_4dobc_results[
        "KF-Mag"
    ][
        "region_growing_seeds"
    ]
)


kf_mag_seed_summary = (
    kf_4dobc_results[
        "KF-Mag"
    ][
        "seed_summary"
    ]
)


kf_mag_object_summary = (
    kf_4dobc_results[
        "KF-Mag"
    ][
        "object_summary"
    ]
)


kf_mag_run_config = (
    kf_4dobc_results[
        "KF-Mag"
    ][
        "run_config"
    ]
)


# ------------------------------------------------------------
# KF-Rate
# ------------------------------------------------------------

kf_rate_objects = (
    kf_4dobc_results[
        "KF-Rate"
    ][
        "objects"
    ]
)


kf_rate_seeds = (
    kf_4dobc_results[
        "KF-Rate"
    ][
        "region_growing_seeds"
    ]
)


kf_rate_seed_summary = (
    kf_4dobc_results[
        "KF-Rate"
    ][
        "seed_summary"
    ]
)


kf_rate_object_summary = (
    kf_4dobc_results[
        "KF-Rate"
    ][
        "object_summary"
    ]
)


kf_rate_run_config = (
    kf_4dobc_results[
        "KF-Rate"
    ][
        "run_config"
    ]
)


# ------------------------------------------------------------
# 9. TEMPORARY BACKWARD-COMPATIBILITY ALIASES
# ------------------------------------------------------------
# These prevent later notebook cells from immediately breaking
# while they are being renamed/restructured.
#
# Remove these aliases after the remaining cells have been
# converted to KF-Mag / KF-Rate terminology.

wf2_magnitude_objects = (
    kf_mag_objects
)

wf2_rate_objects = (
    kf_rate_objects
)

wf2_magnitude_seeds = (
    kf_mag_seeds
)

wf2_rate_seeds = (
    kf_rate_seeds
)

wf2_magnitude_seed_summary = (
    kf_mag_seed_summary
)

wf2_rate_seed_summary = (
    kf_rate_seed_summary
)

wf2_magnitude_object_summary = (
    kf_mag_object_summary
)

wf2_rate_object_summary = (
    kf_rate_object_summary
)


# There is no longer a separate raw/corrected object set.
# The Kalman intervals are supplied directly as seeds.

wf2_magnitude_objects_raw = (
    kf_mag_objects
)

wf2_rate_objects_raw = (
    kf_rate_objects
)

wf2_magnitude_objects_corrected_before_dedup = (
    kf_mag_objects
)

wf2_rate_objects_corrected_before_dedup = (
    kf_rate_objects
)


# Empty compatibility tables because temporal correction and
# deduplication are no longer performed in KF-Mag / KF-Rate.

wf2_magnitude_temporal_correction_table = (
    pd.DataFrame()
)

wf2_rate_temporal_correction_table = (
    pd.DataFrame()
)

wf2_magnitude_duplicate_table = (
    pd.DataFrame()
)

wf2_rate_duplicate_table = (
    pd.DataFrame()
)


# ------------------------------------------------------------
# 10. ACTIVE DOWNSTREAM METHOD
# ------------------------------------------------------------

active_result = (
    kf_4dobc_results[
        ACTIVE_KF_METHOD_FOR_DOWNSTREAM
    ]
)


kalman_4dobc_seeds = (
    active_result[
        "region_growing_seeds"
    ]
)


kalman_4dobc_objects = (
    active_result[
        "objects"
    ]
)


kalman_4dobc_objects_raw = (
    active_result[
        "objects"
    ]
)


kalman_4dobc_seed_summary = (
    active_result[
        "seed_summary"
    ]
)


kalman_4dobc_object_summary = (
    active_result[
        "object_summary"
    ]
)


kalman_4dobc_run_config = (
    active_result[
        "run_config"
    ]
)


print(
    "\nActive downstream KF method:",
    ACTIVE_KF_METHOD_FOR_DOWNSTREAM
)

print(
    "Active final objects:",
    len(
        kalman_4dobc_objects
    )
)


# ------------------------------------------------------------
# 11. COMBINED KF-MAG / KF-RATE SUMMARY
# ------------------------------------------------------------

kf_4dobc_combined_summary = (
    pd.DataFrame([
        {

            "method":
                "KF-Mag",

            "n_input_kalman_seeds":
                len(
                    kalman_seed_table_magnitude
                ),

            "n_region_growing_seeds":
                len(
                    kf_mag_seeds
                ),

            "n_final_objects":
                len(
                    kf_mag_objects
                ),

            "temporal_post_correction":
                False,

            "deduplication_applied":
                False,

            "elapsed_seconds":
                kf_4dobc_results[
                    "KF-Mag"
                ][
                    "elapsed_seconds"
                ]
        },

        {

            "method":
                "KF-Rate",

            "n_input_kalman_seeds":
                len(
                    kalman_seed_table_rate
                ),

            "n_region_growing_seeds":
                len(
                    kf_rate_seeds
                ),

            "n_final_objects":
                len(
                    kf_rate_objects
                ),

            "temporal_post_correction":
                False,

            "deduplication_applied":
                False,

            "elapsed_seconds":
                kf_4dobc_results[
                    "KF-Rate"
                ][
                    "elapsed_seconds"
                ]
        }
    ])
)


display(
    kf_4dobc_combined_summary
)


kf_4dobc_combined_summary.to_csv(

    TABLES_DIR
    / "kf_mag_rate_4dobc_summary.csv",

    index=False
)


# ------------------------------------------------------------
# 12. OBJECT COUNT COMPARISON
# ------------------------------------------------------------

n_4dobc = (

    len(
        original_4dobc_objects
    )

    if
    "original_4dobc_objects"
    in globals()

    else
    np.nan
)


plt.figure(
    figsize=(8, 4)
)


plt.bar(
    [
        "4DOBC",
        "KF-Mag",
        "KF-Rate"
    ],
    [
        n_4dobc,
        len(
            kf_mag_objects
        ),
        len(
            kf_rate_objects
        )
    ]
)


plt.ylabel(
    "Number of extracted objects"
)


plt.title(
    "4D-OBC object count comparison"
)


plt.tight_layout()

plt.show()


# ------------------------------------------------------------
# 13. FINAL VERIFICATION
# ------------------------------------------------------------

print(
    "\n=========================================="
)

print(
    "CELL 13 FINAL SUMMARY"
)

print(
    "=========================================="
)


print(
    "4DOBC objects:",
    n_4dobc
)


print(
    "KF-Mag input seeds:",
    len(
        kalman_seed_table_magnitude
    )
)


print(
    "KF-Mag RegionGrowingSeed objects:",
    len(
        kf_mag_seeds
    )
)


print(
    "KF-Mag extracted objects:",
    len(
        kf_mag_objects
    )
)


print(
    "\nKF-Rate input seeds:",
    len(
        kalman_seed_table_rate
    )
)


print(
    "KF-Rate RegionGrowingSeed objects:",
    len(
        kf_rate_seeds
    )
)


print(
    "KF-Rate extracted objects:",
    len(
        kf_rate_objects
    )
)


print(
    "\nTemporal post-correction: DISABLED"
)

print(
    "Duplicate suppression/NMS: DISABLED"
)

print(
    "Region growing: existing py4dgeo implementation"
)


print(
    "\nCell 13 complete: "
    "KF-Mag and KF-Rate 4D-OBC extraction finished."
)

In [ ]:
# ============================================================
# CELL 14 — OPTIONAL SPATIAL SUPPORT VALIDATION
# FOR KF-MAG-NMS AND KF-RATE-NMS
# ============================================================

# Reusable implementation:
#
#   src/py4dgeo/m3c2_kalman.py
#       -> validate_spatial_support()

# ============================================================


# ------------------------------------------------------------
# 0. IMPORTS
# ------------------------------------------------------------

import numpy as np
import pandas as pd

from py4dgeo.m3c2_kalman import (
    validate_spatial_support
)


# ------------------------------------------------------------
# 1. USER SETTINGS
# ------------------------------------------------------------

# ------------------------------------------------------------
# Master switch
# ------------------------------------------------------------

USE_SPATIAL_SUPPORT = True


# ------------------------------------------------------------
# Spatial-support parameters
# Used only when USE_SPATIAL_SUPPORT = True
# ------------------------------------------------------------

SPATIAL_SUPPORT_RADIUS = 1.5

MIN_NEIGHBOUR_SUPPORT = 3

MIN_SUPPORT_FRACTION = 0.30

SUPPORT_TIME_TOLERANCE = 5

NEIGHBOUR_MAGNITUDE_RATIO = 0.5


# ------------------------------------------------------------
# 2. CHECK REQUIRED VARIABLES
# ------------------------------------------------------------

required_objects = [
    "analysis",
    "kalman_change",
    "kalman_rate",
    "kalman_seed_table_magnitude",
    "kalman_seed_table_rate",
    "timestamps_analysis",
    "TABLES_DIR",
]


for obj_name in required_objects:

    if obj_name not in globals():

        raise ValueError(
            f"{obj_name} is not defined. "
            "Please run Cells 11–13 first."
        )


corepoints_for_support = np.asarray(
    analysis.corepoints.cloud
)


# ------------------------------------------------------------
# 3. HELPER — PREPARE UNVALIDATED TABLE
# ------------------------------------------------------------

def prepare_seed_table_without_spatial_support(
    seed_table
):
    """
    Prepare the original KF seed table for downstream NMS when
    spatial-support validation is disabled.

    No seeds are rejected.
    """

    table = seed_table.copy()


    # --------------------------------------------------------
    # Preserve support interval terminology for compatibility
    # --------------------------------------------------------

    if (
        "support_start_epoch"
        not in table.columns
    ):

        table[
            "support_start_epoch"
        ] = table[
            "start_epoch"
        ]


    if (
        "support_end_epoch"
        not in table.columns
    ):

        table[
            "support_end_epoch"
        ] = table[
            "end_epoch"
        ]


    table[
        "support_start_epoch"
    ] = table[
        "support_start_epoch"
    ].fillna(
        table[
            "start_epoch"
        ]
    ).astype(
        int
    )


    table[
        "support_end_epoch"
    ] = table[
        "support_end_epoch"
    ].fillna(
        table[
            "end_epoch"
        ]
    ).astype(
        int
    )


    table[
        "support_duration_epochs"
    ] = (
        table[
            "support_end_epoch"
        ]
        - table[
            "support_start_epoch"
        ]
        + 1
    )


    # --------------------------------------------------------
    # Columns expected later by NMS / summaries
    # --------------------------------------------------------

    if (
        "support_fraction"
        not in table.columns
    ):

        table[
            "support_fraction"
        ] = np.nan


    if (
        "neighbour_support_count"
        not in table.columns
    ):

        table[
            "neighbour_support_count"
        ] = np.nan


    if (
        "n_neighbours"
        not in table.columns
    ):

        table[
            "n_neighbours"
        ] = np.nan


    table[
        "spatial_support_used"
    ] = False


    table[
        "spatial_support_radius"
    ] = np.nan


    table[
        "min_neighbour_support"
    ] = np.nan


    table[
        "min_support_fraction"
    ] = np.nan


    table[
        "support_time_tolerance"
    ] = np.nan


    table[
        "neighbour_magnitude_ratio"
    ] = np.nan


    # --------------------------------------------------------
    # Rank without spatial-support information
    # --------------------------------------------------------

    table = (
        table
        .sort_values(
            [
                "event_magnitude",
                "duration_epochs"
            ],
            ascending=[
                False,
                False
            ]
        )
        .reset_index(
            drop=True
        )
    )


    table[
        "spatial_support_rank"
    ] = np.arange(
        1,
        len(table) + 1,
        dtype=int
    )


    return table


# ------------------------------------------------------------
# 4. KF-MAG
# ------------------------------------------------------------

print(
    "\n=========================================="
)

print(
    "Preparing KF-Mag seeds for NMS"
)

print(
    "=========================================="
)


if USE_SPATIAL_SUPPORT:

    print(
        "Spatial support validation: ENABLED"
    )


    kf_mag_supported_seeds = (
        validate_spatial_support(

            seed_table=(
                kalman_seed_table_magnitude
            ),

            corepoints=(
                corepoints_for_support
            ),

            kalman_change=(
                kalman_change
            ),

            kalman_rate=(
                kalman_rate
            ),

            timestamps=(
                timestamps_analysis
            ),

            spatial_support_radius=(
                SPATIAL_SUPPORT_RADIUS
            ),

            min_neighbour_support=(
                MIN_NEIGHBOUR_SUPPORT
            ),

            min_support_fraction=(
                MIN_SUPPORT_FRACTION
            ),

            support_time_tolerance=(
                SUPPORT_TIME_TOLERANCE
            ),

            neighbour_magnitude_ratio=(
                NEIGHBOUR_MAGNITUDE_RATIO
            ),
        )
    )


else:

    print(
        "Spatial support validation: DISABLED"
    )

    print(
        "Original KF-Mag seeds will be "
        "passed directly to NMS."
    )


    kf_mag_supported_seeds = (
        prepare_seed_table_without_spatial_support(
            kalman_seed_table_magnitude
        )
    )


print(
    "KF-Mag original seeds:",
    len(
        kalman_seed_table_magnitude
    )
)


print(
    "KF-Mag seeds passed to NMS:",
    len(
        kf_mag_supported_seeds
    )
)


# ------------------------------------------------------------
# 5. KF-RATE
# ------------------------------------------------------------

print(
    "\n=========================================="
)

print(
    "Preparing KF-Rate seeds for NMS"
)

print(
    "=========================================="
)


if USE_SPATIAL_SUPPORT:

    print(
        "Spatial support validation: ENABLED"
    )


    kf_rate_supported_seeds = (
        validate_spatial_support(

            seed_table=(
                kalman_seed_table_rate
            ),

            corepoints=(
                corepoints_for_support
            ),

            kalman_change=(
                kalman_change
            ),

            kalman_rate=(
                kalman_rate
            ),

            timestamps=(
                timestamps_analysis
            ),

            spatial_support_radius=(
                SPATIAL_SUPPORT_RADIUS
            ),

            min_neighbour_support=(
                MIN_NEIGHBOUR_SUPPORT
            ),

            min_support_fraction=(
                MIN_SUPPORT_FRACTION
            ),

            support_time_tolerance=(
                SUPPORT_TIME_TOLERANCE
            ),

            neighbour_magnitude_ratio=(
                NEIGHBOUR_MAGNITUDE_RATIO
            ),
        )
    )


else:

    print(
        "Spatial support validation: DISABLED"
    )

    print(
        "Original KF-Rate seeds will be "
        "passed directly to NMS."
    )


    kf_rate_supported_seeds = (
        prepare_seed_table_without_spatial_support(
            kalman_seed_table_rate
        )
    )


print(
    "KF-Rate original seeds:",
    len(
        kalman_seed_table_rate
    )
)


print(
    "KF-Rate seeds passed to NMS:",
    len(
        kf_rate_supported_seeds
    )
)


# ------------------------------------------------------------
# 6. CLEAN DOWNSTREAM METHOD TABLE
# ------------------------------------------------------------

supported_seed_tables = {

    "KF-Mag-NMS":
        kf_mag_supported_seeds,

    "KF-Rate-NMS":
        kf_rate_supported_seeds
}


# ------------------------------------------------------------
# 7. TEMPORARY BACKWARD COMPATIBILITY
# ------------------------------------------------------------

improved_seed_tables = {

    "magnitude":
        kf_mag_supported_seeds,

    "rate":
        kf_rate_supported_seeds
}


improved_seed_table_magnitude = (
    kf_mag_supported_seeds.copy()
)


improved_seed_table_rate = (
    kf_rate_supported_seeds.copy()
)


# ------------------------------------------------------------
# 8. SAVE TABLES
# ------------------------------------------------------------

if USE_SPATIAL_SUPPORT:

    mag_filename = (
        "kf_mag_spatially_supported_seeds.csv"
    )

    rate_filename = (
        "kf_rate_spatially_supported_seeds.csv"
    )

else:

    mag_filename = (
        "kf_mag_seeds_for_nms_"
        "without_spatial_support.csv"
    )

    rate_filename = (
        "kf_rate_seeds_for_nms_"
        "without_spatial_support.csv"
    )


kf_mag_supported_seeds.to_csv(

    TABLES_DIR
    / mag_filename,

    index=False
)


kf_rate_supported_seeds.to_csv(

    TABLES_DIR
    / rate_filename,

    index=False
)


# ------------------------------------------------------------
# 9. CONFIGURATION TABLE
# ------------------------------------------------------------

spatial_support_config = (
    pd.DataFrame([
        {

            "method":
                "KF-Mag-NMS",

            "spatial_support_enabled":
                USE_SPATIAL_SUPPORT,

            "input_seeds":
                len(
                    kalman_seed_table_magnitude
                ),

            "seeds_passed_to_nms":
                len(
                    kf_mag_supported_seeds
                ),

            "seeds_removed_by_spatial_support":
                (
                    len(
                        kalman_seed_table_magnitude
                    )
                    - len(
                        kf_mag_supported_seeds
                    )
                    if USE_SPATIAL_SUPPORT
                    else 0
                ),

            "spatial_support_radius":
                (
                    SPATIAL_SUPPORT_RADIUS
                    if USE_SPATIAL_SUPPORT
                    else np.nan
                ),

            "min_neighbour_support":
                (
                    MIN_NEIGHBOUR_SUPPORT
                    if USE_SPATIAL_SUPPORT
                    else np.nan
                ),

            "min_support_fraction":
                (
                    MIN_SUPPORT_FRACTION
                    if USE_SPATIAL_SUPPORT
                    else np.nan
                ),

            "support_time_tolerance":
                (
                    SUPPORT_TIME_TOLERANCE
                    if USE_SPATIAL_SUPPORT
                    else np.nan
                ),

            "neighbour_magnitude_ratio":
                (
                    NEIGHBOUR_MAGNITUDE_RATIO
                    if USE_SPATIAL_SUPPORT
                    else np.nan
                )
        },

        {

            "method":
                "KF-Rate-NMS",

            "spatial_support_enabled":
                USE_SPATIAL_SUPPORT,

            "input_seeds":
                len(
                    kalman_seed_table_rate
                ),

            "seeds_passed_to_nms":
                len(
                    kf_rate_supported_seeds
                ),

            "seeds_removed_by_spatial_support":
                (
                    len(
                        kalman_seed_table_rate
                    )
                    - len(
                        kf_rate_supported_seeds
                    )
                    if USE_SPATIAL_SUPPORT
                    else 0
                ),

            "spatial_support_radius":
                (
                    SPATIAL_SUPPORT_RADIUS
                    if USE_SPATIAL_SUPPORT
                    else np.nan
                ),

            "min_neighbour_support":
                (
                    MIN_NEIGHBOUR_SUPPORT
                    if USE_SPATIAL_SUPPORT
                    else np.nan
                ),

            "min_support_fraction":
                (
                    MIN_SUPPORT_FRACTION
                    if USE_SPATIAL_SUPPORT
                    else np.nan
                ),

            "support_time_tolerance":
                (
                    SUPPORT_TIME_TOLERANCE
                    if USE_SPATIAL_SUPPORT
                    else np.nan
                ),

            "neighbour_magnitude_ratio":
                (
                    NEIGHBOUR_MAGNITUDE_RATIO
                    if USE_SPATIAL_SUPPORT
                    else np.nan
                )
        }
    ])
)


display(
    spatial_support_config
)


spatial_support_config.to_csv(

    TABLES_DIR
    / "kf_spatial_support_config.csv",

    index=False
)


# ------------------------------------------------------------
# 10. FINAL SUMMARY
# ------------------------------------------------------------

print(
    "\n=========================================="
)

print(
    "CELL 14 FINAL SUMMARY"
)

print(
    "=========================================="
)


print(
    "Spatial support enabled:",
    USE_SPATIAL_SUPPORT
)


print(
    "\nKF-Mag input seeds:",
    len(
        kalman_seed_table_magnitude
    )
)


print(
    "KF-Mag seeds passed to NMS:",
    len(
        kf_mag_supported_seeds
    )
)


if USE_SPATIAL_SUPPORT:

    print(
        "KF-Mag seeds removed by spatial support:",
        len(
            kalman_seed_table_magnitude
        )
        - len(
            kf_mag_supported_seeds
        )
    )


print(
    "\nKF-Rate input seeds:",
    len(
        kalman_seed_table_rate
    )
)


print(
    "KF-Rate seeds passed to NMS:",
    len(
        kf_rate_supported_seeds
    )
)


if USE_SPATIAL_SUPPORT:

    print(
        "KF-Rate seeds removed by spatial support:",
        len(
            kalman_seed_table_rate
        )
        - len(
            kf_rate_supported_seeds
        )
    )


print(
    "\nCell 14 complete: "
    "seeds prepared for NMS."
)

In [ ]:
# ============================================================
# CELL 15 — KF-MAG-NMS AND KF-RATE-NMS 4D-OBC EXTRACTION
# ============================================================
# Existing py4dgeo RegionGrowingAlgorithm is retained.
# ============================================================


# ------------------------------------------------------------
# 0. IMPORTS
# ------------------------------------------------------------

import time
import numpy as np
import pandas as pd

from py4dgeo.m3c2_kalman import (
    non_maximum_suppression
)

from py4dgeo.segmentation_kalman import (
    create_kalman_region_growing
)


# ------------------------------------------------------------
# 1. USER SETTINGS
# ------------------------------------------------------------

REFERENCE_COREPOINT_INDEX = 15162

VISUALIZATION_COREPOINT_INDEX = 15162


# ------------------------------------------------------------
# NMS parameters
# ------------------------------------------------------------

NMS_RADIUS = 3.0

MIN_NMS_SUPPORT_COUNT = 3

MIN_NMS_SUPPORT_FRACTION = 0.30


# ------------------------------------------------------------
# Pre-growth skip rule
# ------------------------------------------------------------

SKIP_SEED_IF_INSIDE_ACCEPTED_OBJECT = True

SEED_TEMPORAL_OVERLAP_SKIP_THRESHOLD = 0.30


# ------------------------------------------------------------
# Post-growth duplicate safety check
# ------------------------------------------------------------

REMOVE_DUPLICATE_NMS_OBJECTS = True

DUPLICATE_TEMPORAL_IOU_THRESHOLD = 0.30

DUPLICATE_SPATIAL_IOU_THRESHOLD = 0.20


# ------------------------------------------------------------
# Existing py4dgeo region-growing parameters
# ------------------------------------------------------------

WINDOW_WIDTH = 14

MINPERIOD = 2

HEIGHT_THRESHOLD = 0.05

NEIGHBORHOOD_RADIUS = 1.0

MIN_SEGMENTS = 10

THRESHOLDS = [
    0.3,
    0.4,
    0.5,
    0.6,
    0.7,
    0.8,
    0.9,
]


NMS_METHODS = [
    "KF-Mag-NMS",
    "KF-Rate-NMS",
]


ACTIVE_NMS_METHOD_FOR_DOWNSTREAM = (
    "KF-Mag-NMS"
)


# ------------------------------------------------------------
# 2. CHECK REQUIRED VARIABLES
# ------------------------------------------------------------

required_objects = [
    "analysis",
    "supported_seed_tables",
    "timestamps_analysis",
    "TABLES_DIR",
    "USE_SPATIAL_SUPPORT",
]


for obj_name in required_objects:

    if obj_name not in globals():

        raise ValueError(
            f"{obj_name} is not defined. "
            "Please run Cell 14 first."
        )


corepoints_nms = np.asarray(
    analysis.corepoints.cloud
)


# ------------------------------------------------------------
# 3. TEMPORAL / SPATIAL HELPERS
# ------------------------------------------------------------

def interval_overlap(
    a_start,
    a_end,
    b_start,
    b_end
):

    return max(
        0,
        min(
            a_end,
            b_end
        )
        - max(
            a_start,
            b_start
        )
        + 1
    )


def interval_iou(
    a_start,
    a_end,
    b_start,
    b_end
):

    intersection = (
        interval_overlap(
            a_start,
            a_end,
            b_start,
            b_end
        )
    )

    union = (
        max(
            a_end,
            b_end
        )
        - min(
            a_start,
            b_start
        )
        + 1
    )

    return (
        intersection / union
        if union > 0
        else 0.0
    )


def seed_object_overlap_fraction(
    seed_start,
    seed_end,
    object_start,
    object_end
):

    overlap = interval_overlap(
        seed_start,
        seed_end,
        object_start,
        object_end
    )

    duration = (
        seed_end
        - seed_start
        + 1
    )

    return (
        overlap / duration
        if duration > 0
        else 0.0
    )


def spatial_iou(
    indices_a,
    indices_b
):

    set_a = set(
        np.asarray(
            indices_a,
            dtype=int
        ).tolist()
    )

    set_b = set(
        np.asarray(
            indices_b,
            dtype=int
        ).tolist()
    )

    if (
        len(set_a) == 0
        or len(set_b) == 0
    ):

        return 0.0

    return (
        len(
            set_a.intersection(
                set_b
            )
        )
        /
        len(
            set_a.union(
                set_b
            )
        )
    )


# ------------------------------------------------------------
# 4. OBJECT WRAPPER
# ------------------------------------------------------------

class NMSSeedAssignedObject:

    def __init__(
        self,
        obj,
        parent_seed
    ):

        self.original_obj = obj

        self.indices = np.asarray(
            obj.indices,
            dtype=int
        )

        self.original_start_epoch = int(
            obj.start_epoch
        )

        self.original_end_epoch = int(
            obj.end_epoch
        )

        self.parent_seed = (
            parent_seed
        )

        self.matched_seed = (
            parent_seed
        )

        # NMS seed interval defines temporal extent.
        self.start_epoch = int(
            parent_seed[
                "support_start_epoch"
            ]
        )

        self.end_epoch = int(
            parent_seed[
                "support_end_epoch"
            ]
        )

        if hasattr(
            obj,
            "seed"
        ):

            self.seed = obj.seed


# ------------------------------------------------------------
# 5. CHECK WHETHER NMS SEED IS ALREADY REPRESENTED
# ------------------------------------------------------------

def seed_already_represented(
    seed_row,
    accepted_objects
):

    if not SKIP_SEED_IF_INSIDE_ACCEPTED_OBJECT:

        return (
            False,
            None,
            np.nan
        )


    seed_cp = int(
        seed_row[
            "corepoint_index_python"
        ]
    )

    seed_start = int(
        seed_row[
            "support_start_epoch"
        ]
    )

    seed_end = int(
        seed_row[
            "support_end_epoch"
        ]
    )


    for object_number, obj in enumerate(
        accepted_objects,
        start=1
    ):

        object_indices = set(
            np.asarray(
                obj.indices,
                dtype=int
            ).tolist()
        )


        if seed_cp not in object_indices:
            continue


        overlap_fraction = (
            seed_object_overlap_fraction(
                seed_start,
                seed_end,
                int(
                    obj.start_epoch
                ),
                int(
                    obj.end_epoch
                )
            )
        )


        if (
            overlap_fraction
            >= SEED_TEMPORAL_OVERLAP_SKIP_THRESHOLD
        ):

            return (
                True,
                object_number,
                overlap_fraction
            )


    return (
        False,
        None,
        np.nan
    )


# ------------------------------------------------------------
# 6. GROW ONE NMS SEED
# ------------------------------------------------------------

def grow_one_nms_seed(
    method_name,
    seed_row
):

    one_seed_table = pd.DataFrame(
        [
            seed_row
        ]
    )


    algorithm = (
        create_kalman_region_growing(

            seed_table=(
                one_seed_table
            ),

            method_label=(
                f"{method_name}_seed_"
                f"{int(seed_row['nms_rank'])}"
            ),

            seed_mode="all",

            top_n=1,

            use_support_interval=True,

            ranking_columns=[
                "nms_rank"
            ],

            window_width=(
                WINDOW_WIDTH
            ),

            minperiod=(
                MINPERIOD
            ),

            height_threshold=(
                HEIGHT_THRESHOLD
            ),

            neighborhood_radius=(
                NEIGHBORHOOD_RADIUS
            ),

            min_segments=(
                MIN_SEGMENTS
            ),

            thresholds=(
                THRESHOLDS
            )
        )
    )


    analysis.invalidate_results(
        seeds=True,
        objects=True,
        smoothed_distances=False
    )


    objects = algorithm.run(
        analysis
    )


    analysis_seeds = (
        analysis.seeds
    )


    if len(objects) == 0:

        return (
            None,
            analysis_seeds
        )


    parent_cp = int(
        seed_row[
            "corepoint_index_python"
        ]
    )


    selected_object = None


    # Prefer object that actually contains parent seed.
    for obj in objects:

        indices = set(
            np.asarray(
                obj.indices,
                dtype=int
            ).tolist()
        )

        if parent_cp in indices:

            selected_object = obj

            break


    if selected_object is None:

        selected_object = (
            objects[0]
        )


    assigned_object = (
        NMSSeedAssignedObject(
            obj=selected_object,
            parent_seed=seed_row
        )
    )


    return (
        assigned_object,
        analysis_seeds
    )


# ------------------------------------------------------------
# 7. POST-GROWTH DUPLICATE CHECK
# ------------------------------------------------------------

def object_is_duplicate(
    candidate,
    accepted_objects
):

    if not REMOVE_DUPLICATE_NMS_OBJECTS:

        return (
            False,
            None,
            np.nan,
            np.nan
        )


    for object_number, kept in enumerate(
        accepted_objects,
        start=1
    ):

        t_iou = interval_iou(
            int(
                candidate.start_epoch
            ),
            int(
                candidate.end_epoch
            ),
            int(
                kept.start_epoch
            ),
            int(
                kept.end_epoch
            )
        )


        s_iou = spatial_iou(
            candidate.indices,
            kept.indices
        )


        if (
            t_iou
            >= DUPLICATE_TEMPORAL_IOU_THRESHOLD
            and
            s_iou
            >= DUPLICATE_SPATIAL_IOU_THRESHOLD
        ):

            return (
                True,
                object_number,
                t_iou,
                s_iou
            )


    return (
        False,
        None,
        np.nan,
        np.nan
    )


# ------------------------------------------------------------
# 8. OBJECT SUMMARY
# ------------------------------------------------------------

def summarize_nms_objects(
    method_name,
    objects
):

    records = []


    for object_number, obj in enumerate(
        objects,
        start=1
    ):

        seed = (
            obj.matched_seed
        )

        object_indices = set(
            np.asarray(
                obj.indices,
                dtype=int
            ).tolist()
        )


        records.append({

            "method":
                method_name,

            "object_number_user":
                object_number,

            "object_generating_seed_corepoint":
                int(
                    seed[
                        "corepoint_index_python"
                    ]
                ),

            "object_generating_seed_user_number":
                int(
                    seed[
                        "corepoint_index_python"
                    ]
                    + 1
                ),

            "object_generating_nms_rank":
                int(
                    seed[
                        "nms_rank"
                    ]
                ),

            "object_generating_pre_nms_rank":
                int(
                    seed[
                        "pre_nms_rank"
                    ]
                ),

            "start_epoch":
                int(
                    obj.start_epoch
                ),

            "end_epoch":
                int(
                    obj.end_epoch
                ),

            "duration_epochs":
                int(
                    obj.end_epoch
                    - obj.start_epoch
                    + 1
                ),

            "direction":
                seed[
                    "direction"
                ],

            "event_magnitude":
                float(
                    seed[
                        "event_magnitude"
                    ]
                ),

            "abs_net_change":
                float(
                    seed[
                        "abs_net_change"
                    ]
                ),

            "nms_support_count":
                int(
                    seed[
                        "nms_support_count"
                    ]
                ),

            "nms_support_fraction":
                float(
                    seed[
                        "nms_support_fraction"
                    ]
                ),

            "spatial_support_used":
                bool(
                    USE_SPATIAL_SUPPORT
                ),

            "n_corepoints":
                len(
                    obj.indices
                ),

            "object_contains_reference_corepoint":
                (
                    REFERENCE_COREPOINT_INDEX
                    in object_indices
                ),

            "object_contains_visualization_corepoint":
                (
                    VISUALIZATION_COREPOINT_INDEX
                    in object_indices
                ),

            "start_time":
                timestamps_analysis[
                    int(
                        obj.start_epoch
                    )
                ],

            "end_time":
                timestamps_analysis[
                    int(
                        obj.end_epoch
                    )
                ],
        })


    return pd.DataFrame(
        records
    )


# ------------------------------------------------------------
# 9. RUN ONE NMS METHOD
# ------------------------------------------------------------

def run_kf_nms_method(
    method_name,
    input_seed_table
):

    print(
        "\n=========================================="
    )

    print(
        f"Running {method_name}"
    )

    print(
        "=========================================="
    )


    start_all = (
        time.time()
    )


    # --------------------------------------------------------
    # Apply NMS
    # --------------------------------------------------------

    nms_seed_table = (
        non_maximum_suppression(

            seed_table=(
                input_seed_table
            ),

            corepoints=(
                corepoints_nms
            ),

            nms_radius=(
                NMS_RADIUS
            ),

            min_support_count=(
                MIN_NMS_SUPPORT_COUNT
            ),

            min_support_fraction=(
                MIN_NMS_SUPPORT_FRACTION
            ),
        )
    )


    if len(
        nms_seed_table
    ) == 0:

        raise ValueError(
            f"NMS removed all seeds for "
            f"{method_name}."
        )


    print(
        "Seeds entering NMS:",
        len(
            input_seed_table
        )
    )


    print(
        "Seeds retained after NMS:",
        len(
            nms_seed_table
        )
    )


    # --------------------------------------------------------
    # Ranked seed -> object extraction
    # --------------------------------------------------------

    accepted_objects = []

    all_py4dgeo_seeds = []

    generation_records = []

    skipped_records = []

    duplicate_records = []


    for _, seed_row in (
        nms_seed_table.iterrows()
    ):

        nms_rank = int(
            seed_row[
                "nms_rank"
            ]
        )

        seed_cp = int(
            seed_row[
                "corepoint_index_python"
            ]
        )


        represented, (
            existing_object_number
        ), overlap_fraction = (
            seed_already_represented(
                seed_row,
                accepted_objects
            )
        )


        if represented:

            skipped_records.append({

                "method":
                    method_name,

                "nms_rank":
                    nms_rank,

                "seed_corepoint":
                    seed_cp,

                "reason":
                    "already_represented_by_accepted_object",

                "existing_object_number":
                    existing_object_number,

                "seed_temporal_overlap_fraction":
                    overlap_fraction,
            })

            continue


        print(
            f"Growing {method_name} object "
            f"from NMS seed {nms_rank} "
            f"(corepoint {seed_cp})"
        )


        candidate_object, seeds = (
            grow_one_nms_seed(
                method_name=(
                    method_name
                ),
                seed_row=(
                    seed_row
                )
            )
        )


        all_py4dgeo_seeds.extend(
            seeds
        )


        if candidate_object is None:

            generation_records.append({

                "method":
                    method_name,

                "nms_rank":
                    nms_rank,

                "seed_corepoint":
                    seed_cp,

                "status":
                    "no_object_generated"
            })

            continue


        duplicate, (
            duplicate_of
        ), t_iou, s_iou = (
            object_is_duplicate(
                candidate_object,
                accepted_objects
            )
        )


        if duplicate:

            duplicate_records.append({

                "method":
                    method_name,

                "candidate_nms_rank":
                    nms_rank,

                "candidate_seed_corepoint":
                    seed_cp,

                "duplicate_of_object_number":
                    duplicate_of,

                "temporal_iou":
                    t_iou,

                "spatial_iou":
                    s_iou,

                "decision":
                    "removed_after_growth",
            })

            continue


        accepted_objects.append(
            candidate_object
        )


        generation_records.append({

            "method":
                method_name,

            "object_number_user":
                len(
                    accepted_objects
                ),

            "nms_rank":
                nms_rank,

            "seed_corepoint":
                seed_cp,

            "status":
                "accepted_new_object",

            "start_epoch":
                int(
                    candidate_object.start_epoch
                ),

            "end_epoch":
                int(
                    candidate_object.end_epoch
                ),

            "duration_epochs":
                int(
                    candidate_object.end_epoch
                    - candidate_object.start_epoch
                    + 1
                ),

            "n_corepoints":
                len(
                    candidate_object.indices
                ),
        })


    elapsed = (
        time.time()
        - start_all
    )


    generation_table = pd.DataFrame(
        generation_records
    )

    skipped_seed_table = pd.DataFrame(
        skipped_records
    )

    duplicate_table = pd.DataFrame(
        duplicate_records
    )


    object_summary = (
        summarize_nms_objects(
            method_name=(
                method_name
            ),
            objects=(
                accepted_objects
            )
        )
    )


    # --------------------------------------------------------
    # Configuration
    # --------------------------------------------------------

    run_config = pd.DataFrame([
        {

            "method":
                method_name,

            "spatial_support_enabled":
                USE_SPATIAL_SUPPORT,

            "n_input_seeds":
                len(
                    input_seed_table
                ),

            "n_nms_seeds":
                len(
                    nms_seed_table
                ),

            "n_skipped_before_growth":
                len(
                    skipped_seed_table
                ),

            "n_post_growth_duplicates_removed":
                len(
                    duplicate_table
                ),

            "n_final_objects":
                len(
                    accepted_objects
                ),

            "nms_radius":
                NMS_RADIUS,

            "min_nms_support_count":
                MIN_NMS_SUPPORT_COUNT,

            "min_nms_support_fraction":
                MIN_NMS_SUPPORT_FRACTION,

            "pre_growth_skip_enabled":
                SKIP_SEED_IF_INSIDE_ACCEPTED_OBJECT,

            "seed_overlap_skip_threshold":
                SEED_TEMPORAL_OVERLAP_SKIP_THRESHOLD,

            "post_growth_duplicate_check":
                REMOVE_DUPLICATE_NMS_OBJECTS,

            "duplicate_temporal_iou_threshold":
                DUPLICATE_TEMPORAL_IOU_THRESHOLD,

            "duplicate_spatial_iou_threshold":
                DUPLICATE_SPATIAL_IOU_THRESHOLD,

            "window_width":
                WINDOW_WIDTH,

            "minperiod":
                MINPERIOD,

            "height_threshold":
                HEIGHT_THRESHOLD,

            "neighborhood_radius":
                NEIGHBORHOOD_RADIUS,

            "min_segments":
                MIN_SEGMENTS,

            "thresholds":
                str(
                    THRESHOLDS
                ),

            "elapsed_seconds":
                elapsed,
        }
    ])


    # --------------------------------------------------------
    # Save
    # --------------------------------------------------------

    file_prefix = (
        method_name
        .lower()
        .replace(
            "-",
            "_"
        )
    )


    nms_seed_table.to_csv(
        TABLES_DIR
        / f"{file_prefix}_seed_table.csv",
        index=False
    )


    generation_table.to_csv(
        TABLES_DIR
        / f"{file_prefix}_object_generation.csv",
        index=False
    )


    skipped_seed_table.to_csv(
        TABLES_DIR
        / f"{file_prefix}_skipped_seeds.csv",
        index=False
    )


    duplicate_table.to_csv(
        TABLES_DIR
        / f"{file_prefix}_duplicate_objects.csv",
        index=False
    )


    object_summary.to_csv(
        TABLES_DIR
        / f"{file_prefix}_object_summary.csv",
        index=False
    )


    run_config.to_csv(
        TABLES_DIR
        / f"{file_prefix}_run_config.csv",
        index=False
    )


    print(
        f"\n{method_name} complete."
    )


    print(
        "NMS seeds:",
        len(
            nms_seed_table
        )
    )


    print(
        "Final objects:",
        len(
            accepted_objects
        )
    )


    return {

        "method":
            method_name,

        "input_seed_table":
            input_seed_table.copy(),

        "nms_seed_table":
            nms_seed_table,

        "objects":
            accepted_objects,

        "seeds":
            all_py4dgeo_seeds,

        "generation_table":
            generation_table,

        "skipped_seed_table":
            skipped_seed_table,

        "duplicate_table":
            duplicate_table,

        "object_summary":
            object_summary,

        "run_config":
            run_config,

        "elapsed_seconds":
            elapsed,
    }


# ------------------------------------------------------------
# 10. RUN BOTH METHODS
# ------------------------------------------------------------

kf_nms_results = {}


for method_name in NMS_METHODS:

    if (
        method_name
        not in supported_seed_tables
    ):

        raise ValueError(
            f"{method_name} is missing from "
            "supported_seed_tables."
        )


    kf_nms_results[
        method_name
    ] = run_kf_nms_method(

        method_name=(
            method_name
        ),

        input_seed_table=(
            supported_seed_tables[
                method_name
            ]
        )
    )


# ------------------------------------------------------------
# 11. CLEAN ALIASES
# ------------------------------------------------------------

kf_mag_nms_seed_table = (
    kf_nms_results[
        "KF-Mag-NMS"
    ][
        "nms_seed_table"
    ]
)


kf_rate_nms_seed_table = (
    kf_nms_results[
        "KF-Rate-NMS"
    ][
        "nms_seed_table"
    ]
)


kf_mag_nms_objects = (
    kf_nms_results[
        "KF-Mag-NMS"
    ][
        "objects"
    ]
)


kf_rate_nms_objects = (
    kf_nms_results[
        "KF-Rate-NMS"
    ][
        "objects"
    ]
)


kf_mag_nms_seeds = (
    kf_nms_results[
        "KF-Mag-NMS"
    ][
        "seeds"
    ]
)


kf_rate_nms_seeds = (
    kf_nms_results[
        "KF-Rate-NMS"
    ][
        "seeds"
    ]
)


kf_mag_nms_object_summary = (
    kf_nms_results[
        "KF-Mag-NMS"
    ][
        "object_summary"
    ]
)


kf_rate_nms_object_summary = (
    kf_nms_results[
        "KF-Rate-NMS"
    ][
        "object_summary"
    ]
)


# ------------------------------------------------------------
# 12. TEMPORARY OLD-NAME ALIASES
# ------------------------------------------------------------

wf3_magnitude_nms_seed_table = (
    kf_mag_nms_seed_table
)

wf3_rate_nms_seed_table = (
    kf_rate_nms_seed_table
)

wf3_magnitude_objects = (
    kf_mag_nms_objects
)

wf3_rate_objects = (
    kf_rate_nms_objects
)

wf3_magnitude_seeds = (
    kf_mag_nms_seeds
)

wf3_rate_seeds = (
    kf_rate_nms_seeds
)


# ------------------------------------------------------------
# 13. ACTIVE DOWNSTREAM NMS METHOD
# ------------------------------------------------------------

active_nms_result = (
    kf_nms_results[
        ACTIVE_NMS_METHOD_FOR_DOWNSTREAM
    ]
)


nms_seed_table = (
    active_nms_result[
        "nms_seed_table"
    ]
)


improved_4dobc_objects = (
    active_nms_result[
        "objects"
    ]
)


improved_4dobc_seeds = (
    active_nms_result[
        "seeds"
    ]
)


improved_4dobc_object_summary = (
    active_nms_result[
        "object_summary"
    ]
)


improved_4dobc_run_config = (
    active_nms_result[
        "run_config"
    ]
)


# ------------------------------------------------------------
# 14. FINAL FIVE-METHOD SUMMARY
# ------------------------------------------------------------

n_4dobc = (
    len(
        original_4dobc_objects
    )
    if
    "original_4dobc_objects"
    in globals()
    else
    np.nan
)


five_method_summary = pd.DataFrame([
    {
        "method": "4DOBC",
        "n_final_objects": n_4dobc,
    },
    {
        "method": "KF-Mag",
        "n_final_objects": len(
            kf_mag_objects
        ),
    },
    {
        "method": "KF-Rate",
        "n_final_objects": len(
            kf_rate_objects
        ),
    },
    {
        "method": "KF-Mag-NMS",
        "n_final_objects": len(
            kf_mag_nms_objects
        ),
        "n_nms_seeds": len(
            kf_mag_nms_seed_table
        ),
        "spatial_support_enabled":
            USE_SPATIAL_SUPPORT,
    },
    {
        "method": "KF-Rate-NMS",
        "n_final_objects": len(
            kf_rate_nms_objects
        ),
        "n_nms_seeds": len(
            kf_rate_nms_seed_table
        ),
        "spatial_support_enabled":
            USE_SPATIAL_SUPPORT,
    },
])


display(
    five_method_summary
)


five_method_summary.to_csv(
    TABLES_DIR
    / "five_method_summary.csv",
    index=False
)


# ------------------------------------------------------------
# 15. FINAL STATUS
# ------------------------------------------------------------

print(
    "\n=========================================="
)

print(
    "CELL 15 FINAL SUMMARY"
)

print(
    "=========================================="
)


print(
    "Spatial support enabled:",
    USE_SPATIAL_SUPPORT
)


print(
    "\n4DOBC objects:",
    n_4dobc
)


print(
    "KF-Mag objects:",
    len(
        kf_mag_objects
    )
)


print(
    "KF-Rate objects:",
    len(
        kf_rate_objects
    )
)


print(
    "\nKF-Mag-NMS seeds:",
    len(
        kf_mag_nms_seed_table
    )
)


print(
    "KF-Mag-NMS objects:",
    len(
        kf_mag_nms_objects
    )
)


print(
    "\nKF-Rate-NMS seeds:",
    len(
        kf_rate_nms_seed_table
    )
)


print(
    "KF-Rate-NMS objects:",
    len(
        kf_rate_nms_objects
    )
)


print(
    "\nCell 15 complete."
)

In [ ]:
# ============================================================
# CELL 16 — MULTI-METHOD OBJECT VISUALIZATION
# WITH HIERARCHICAL OBJECT MATCHING
# ============================================================

# ------------------------------------------------------------
# 1. USER SETTINGS
# ------------------------------------------------------------

REFERENCE_COREPOINT_INDEX = 15162
VISUALIZATION_COREPOINT_INDEX = 15162


# ------------------------------------------------------------
# Object selection
# ------------------------------------------------------------

OBJECT_SELECTION_MODE = "object_number"

# options:
#
#   "object_number"
#       Plot objects listed in SELECTED_OBJECT_NUMBERS
#
#   "contains_reference"
#       Plot the first object containing
#       REFERENCE_COREPOINT_INDEX


SELECTED_OBJECT_NUMBERS = [
    1,
    2
]


# ------------------------------------------------------------
# Plot settings
# ------------------------------------------------------------

CRANGE = 1.0

SAVE_METHOD_COMPARISON_FIGURES = True


# ------------------------------------------------------------
# Exact fixed colours
# ------------------------------------------------------------

SELECTED_METHOD_HULL_COLOR = "#00FFFF"
# bright cyan, solid

REFERENCE_METHOD_HULL_COLOR = "#FF00FF"
# bright magenta, dashed

VISUALIZATION_CP_COLOR = "#000000"

GENERATING_SEED_COLOR = "#0066FF"


# ------------------------------------------------------------
# Output directory
# ------------------------------------------------------------

method_comparison_fig_dir = (
    FIGURES_DIR
    / "method_object_comparisons"
)

method_comparison_fig_dir.mkdir(
    exist_ok=True,
    parents=True
)


# ------------------------------------------------------------
# 2. CHECK REQUIRED VARIABLES
# ------------------------------------------------------------

required_objects = [
    "analysis",
    "timestamps_analysis",

    "original_4dobc_objects",

    "kf_mag_objects",
    "kf_rate_objects",

    "kf_mag_nms_objects",
    "kf_rate_nms_objects",

    "kf_mag_seeds",
    "kf_rate_seeds",

    "kf_mag_nms_seeds",
    "kf_rate_nms_seeds",
]


for obj_name in required_objects:

    if obj_name not in globals():

        raise ValueError(
            f"{obj_name} is not defined. "
            "Please run Cells 13–15 first."
        )


# ------------------------------------------------------------
# 3. IMPORTS AND SHARED VARIABLES
# ------------------------------------------------------------

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.dates as mdates

from scipy.spatial import ConvexHull
from matplotlib.patches import Polygon


cloud = np.asarray(
    analysis.corepoints.cloud
)


dtFmt = mdates.DateFormatter(
    "%b-%d"
)


# ------------------------------------------------------------
# 4. FINAL METHOD OBJECT SETS
# ------------------------------------------------------------

method_objects = {

    "4DOBC":
        original_4dobc_objects,

    "KF-Mag":
        kf_mag_objects,

    "KF-Rate":
        kf_rate_objects,

    "KF-Mag-NMS":
        kf_mag_nms_objects,

    "KF-Rate-NMS":
        kf_rate_nms_objects,
}


# ------------------------------------------------------------
# 5. FINAL METHOD SEED SETS
# ------------------------------------------------------------
# 4DOBC uses an empty external-seed list here.
#
# If explicit event magnitude is unavailable for a candidate
# object, it is calculated from the object's smoothed change
# signal for ranking.

method_seeds = {

    "4DOBC":
        [],

    "KF-Mag":
        kf_mag_seeds,

    "KF-Rate":
        kf_rate_seeds,

    "KF-Mag-NMS":
        kf_mag_nms_seeds,

    "KF-Rate-NMS":
        kf_rate_nms_seeds,
}


# ------------------------------------------------------------
# 6. DEFINE THE SIX REQUESTED COMPARISONS
# ------------------------------------------------------------

COMPARISON_SPECS = [

    {
        "comparison_id":
            "4dobc_vs_kf_mag",

        "reference_method":
            "4DOBC",

        "selected_method":
            "KF-Mag",
    },

    {
        "comparison_id":
            "4dobc_vs_kf_rate",

        "reference_method":
            "4DOBC",

        "selected_method":
            "KF-Rate",
    },

    {
        "comparison_id":
            "4dobc_vs_kf_mag_nms",

        "reference_method":
            "4DOBC",

        "selected_method":
            "KF-Mag-NMS",
    },

    {
        "comparison_id":
            "4dobc_vs_kf_rate_nms",

        "reference_method":
            "4DOBC",

        "selected_method":
            "KF-Rate-NMS",
    },

    {
        "comparison_id":
            "kf_mag_vs_kf_mag_nms",

        "reference_method":
            "KF-Mag",

        "selected_method":
            "KF-Mag-NMS",
    },

    {
        "comparison_id":
            "kf_rate_vs_kf_rate_nms",

        "reference_method":
            "KF-Rate",

        "selected_method":
            "KF-Rate-NMS",
    },
]


# ============================================================
# 7. HELPER — FIND OBJECT CONTAINING COREPOINT
# ============================================================

def find_object_containing_corepoint(
    objects,
    cp_idx
):
    """
    Return zero-based index of the first object containing
    the requested corepoint.
    """

    for object_index, obj in enumerate(
        objects
    ):

        indices = np.asarray(
            obj.indices,
            dtype=int
        )

        if int(cp_idx) in indices:

            return object_index

    return None


# ============================================================
# 8. TEMPORAL OVERLAP
# ============================================================

def calculate_temporal_overlap(
    start_a,
    end_a,
    start_b,
    end_b
):
    """
    Inclusive temporal overlap in epochs.
    """

    start_a = int(start_a)
    end_a = int(end_a)

    start_b = int(start_b)
    end_b = int(end_b)


    return max(
        0,

        min(
            end_a,
            end_b
        )

        - max(
            start_a,
            start_b
        )

        + 1
    )


# ============================================================
# 9. SPATIAL OVERLAP
# ============================================================

def calculate_spatial_overlap(
    indices_a,
    indices_b
):
    """
    Spatial overlap as number of shared corepoints.
    """

    set_a = set(
        np.asarray(
            indices_a,
            dtype=int
        ).tolist()
    )

    set_b = set(
        np.asarray(
            indices_b,
            dtype=int
        ).tolist()
    )


    return len(
        set_a.intersection(
            set_b
        )
    )


# ============================================================
# 10. SPATIAL IoU
# ============================================================

def calculate_spatial_iou(
    indices_a,
    indices_b
):
    """
    Corepoint-based spatial IoU.

    Used for evaluation/reporting, not object ranking.
    """

    set_a = set(
        np.asarray(
            indices_a,
            dtype=int
        ).tolist()
    )

    set_b = set(
        np.asarray(
            indices_b,
            dtype=int
        ).tolist()
    )


    union = set_a.union(
        set_b
    )


    if len(union) == 0:

        return 0.0


    intersection = set_a.intersection(
        set_b
    )


    return (
        len(intersection)
        / len(union)
    )


# ============================================================
# 11. TEMPORAL IoU
# ============================================================

def calculate_temporal_iou(
    start_a,
    end_a,
    start_b,
    end_b
):
    """
    Inclusive temporal IoU.

    Used for evaluation/reporting, not object ranking.
    """

    intersection = (
        calculate_temporal_overlap(
            start_a,
            end_a,
            start_b,
            end_b
        )
    )


    union = (
        max(
            int(end_a),
            int(end_b)
        )

        - min(
            int(start_a),
            int(start_b)
        )

        + 1
    )


    return (
        intersection / union
        if union > 0
        else 0.0
    )


# ============================================================
# 12. FIND PY4DGEO SEED FOR OBJECT
# ============================================================

def find_seed_for_object(
    seeds,
    obj
):
    """
    Find a py4dgeo seed whose corepoint lies inside the object
    and whose temporal interval overlaps the object's interval.
    """

    if seeds is None:

        return None


    obj_start = int(
        obj.start_epoch
    )

    obj_end = int(
        obj.end_epoch
    )


    obj_indices = set(
        np.asarray(
            obj.indices,
            dtype=int
        ).tolist()
    )


    for seed in seeds:

        seed_idx = int(
            seed.index
        )


        if seed_idx not in obj_indices:

            continue


        overlap = calculate_temporal_overlap(

            obj_start,
            obj_end,

            int(
                seed.start_epoch
            ),

            int(
                seed.end_epoch
            )
        )


        if overlap > 0:

            return seed


    return None


# ============================================================
# 13. GET GENERATING-SEED INFORMATION FOR VISUALIZATION
# ============================================================

def get_object_generating_seed_info(
    obj,
    seeds
):
    """
    Return generating-seed metadata.

    KF-Mag-NMS / KF-Rate-NMS:
        use matched_seed attached in Cell 15.

    KF-Mag / KF-Rate:
        use RegionGrowingSeed as fallback.
    """

    # --------------------------------------------------------
    # A. NMS object with matched seed metadata
    # --------------------------------------------------------

    if (
        hasattr(
            obj,
            "matched_seed"
        )
        and
        obj.matched_seed is not None
    ):

        matched_seed = (
            obj.matched_seed
        )


        if (
            "support_start_epoch"
            in matched_seed
            and
            not pd.isna(
                matched_seed[
                    "support_start_epoch"
                ]
            )
        ):

            seed_start = int(
                matched_seed[
                    "support_start_epoch"
                ]
            )

        else:

            seed_start = int(
                matched_seed[
                    "start_epoch"
                ]
            )


        if (
            "support_end_epoch"
            in matched_seed
            and
            not pd.isna(
                matched_seed[
                    "support_end_epoch"
                ]
            )
        ):

            seed_end = int(
                matched_seed[
                    "support_end_epoch"
                ]
            )

        else:

            seed_end = int(
                matched_seed[
                    "end_epoch"
                ]
            )


        return {

            "corepoint":
                int(
                    matched_seed[
                        "corepoint_index_python"
                    ]
                ),

            "start_epoch":
                seed_start,

            "end_epoch":
                seed_end,

            "duration_epochs":
                (
                    seed_end
                    - seed_start
                    + 1
                ),

            "direction":
                matched_seed.get(
                    "direction",
                    None
                ),

            "event_magnitude":
                (
                    float(
                        matched_seed[
                            "event_magnitude"
                        ]
                    )

                    if
                    "event_magnitude"
                    in matched_seed
                    and
                    not pd.isna(
                        matched_seed[
                            "event_magnitude"
                        ]
                    )

                    else
                    np.nan
                ),

            "nms_rank":
                (
                    int(
                        matched_seed[
                            "nms_rank"
                        ]
                    )

                    if
                    "nms_rank"
                    in matched_seed
                    and
                    not pd.isna(
                        matched_seed[
                            "nms_rank"
                        ]
                    )

                    else
                    np.nan
                )
        }


    # --------------------------------------------------------
    # B. KF-Mag / KF-Rate RegionGrowingSeed
    # --------------------------------------------------------

    selected_seed = (
        find_seed_for_object(
            seeds,
            obj
        )
    )


    if selected_seed is not None:

        seed_start = int(
            selected_seed.start_epoch
        )

        seed_end = int(
            selected_seed.end_epoch
        )


        return {

            "corepoint":
                int(
                    selected_seed.index
                ),

            "start_epoch":
                seed_start,

            "end_epoch":
                seed_end,

            "duration_epochs":
                (
                    seed_end
                    - seed_start
                    + 1
                ),

            "direction":
                None,

            "event_magnitude":
                np.nan,

            "nms_rank":
                np.nan,
        }


    # --------------------------------------------------------
    # C. No generating seed found
    # --------------------------------------------------------

    return {

        "corepoint":
            np.nan,

        "start_epoch":
            np.nan,

        "end_epoch":
            np.nan,

        "duration_epochs":
            np.nan,

        "direction":
            None,

        "event_magnitude":
            np.nan,

        "nms_rank":
            np.nan,
    }


# ============================================================
# 14. GET OBJECT EVENT MAGNITUDE AND SEED DURATION FOR RANKING
# ============================================================

def get_object_ranking_seed_information(
    obj,
    seeds
):
    """
    Return:
        event_magnitude
        seed_duration

    These are ranking criteria 3 and 4.

    Priority:
        1. matched_seed metadata when available
        2. py4dgeo RegionGrowingSeed
        3. object interval as fallback

    If event magnitude is unavailable as metadata, it is
    calculated from the Kalman-smoothed / smoothed M3C2 change
    values over the corresponding seed/object interval.
    """

    event_magnitude = 0.0

    seed_start = None
    seed_end = None


    # --------------------------------------------------------
    # A. matched_seed metadata
    # --------------------------------------------------------

    if (
        hasattr(
            obj,
            "matched_seed"
        )
        and
        obj.matched_seed is not None
    ):

        matched_seed = (
            obj.matched_seed
        )


        if (
            "support_start_epoch"
            in matched_seed
            and
            not pd.isna(
                matched_seed[
                    "support_start_epoch"
                ]
            )
        ):

            seed_start = int(
                matched_seed[
                    "support_start_epoch"
                ]
            )

        else:

            seed_start = int(
                matched_seed[
                    "start_epoch"
                ]
            )


        if (
            "support_end_epoch"
            in matched_seed
            and
            not pd.isna(
                matched_seed[
                    "support_end_epoch"
                ]
            )
        ):

            seed_end = int(
                matched_seed[
                    "support_end_epoch"
                ]
            )

        else:

            seed_end = int(
                matched_seed[
                    "end_epoch"
                ]
            )


        if (
            "event_magnitude"
            in matched_seed
            and
            not pd.isna(
                matched_seed[
                    "event_magnitude"
                ]
            )
        ):

            event_magnitude = float(
                matched_seed[
                    "event_magnitude"
                ]
            )


    # --------------------------------------------------------
    # B. py4dgeo seed
    # --------------------------------------------------------

    if (
        seed_start is None
        or seed_end is None
    ):

        selected_seed = (
            find_seed_for_object(
                seeds,
                obj
            )
        )


        if selected_seed is not None:

            seed_start = int(
                selected_seed.start_epoch
            )

            seed_end = int(
                selected_seed.end_epoch
            )


    # --------------------------------------------------------
    # C. Object interval fallback
    # --------------------------------------------------------

    if seed_start is None:

        seed_start = int(
            obj.start_epoch
        )


    if seed_end is None:

        seed_end = int(
            obj.end_epoch
        )


    seed_duration = (
        seed_end
        - seed_start
        + 1
    )


    # --------------------------------------------------------
    # D. Calculate magnitude if metadata did not provide it
    # --------------------------------------------------------

    if (
        not np.isfinite(
            event_magnitude
        )
        or event_magnitude <= 0.0
    ):

        object_indices = np.asarray(
            obj.indices,
            dtype=int
        )


        if (
            len(
                object_indices
            ) > 0
            and
            0 <= seed_start
            < analysis.smoothed_distances.shape[1]
            and
            0 <= seed_end
            < analysis.smoothed_distances.shape[1]
        ):

            object_change = (

                analysis.smoothed_distances[
                    object_indices,
                    seed_end
                ]

                -

                analysis.smoothed_distances[
                    object_indices,
                    seed_start
                ]
            )


            if np.any(
                np.isfinite(
                    object_change
                )
            ):

                event_magnitude = float(
                    np.nanmax(
                        np.abs(
                            object_change
                        )
                    )
                )


    if not np.isfinite(
        event_magnitude
    ):

        event_magnitude = 0.0


    return (
        float(
            event_magnitude
        ),
        int(
            seed_duration
        )
    )


# ============================================================
# 15. FIND BEST REFERENCE OBJECT USING REQUIRED RANKING
# ============================================================

def find_best_reference_match(
    selected_obj,
    reference_objects,
    reference_seeds
):
    """
    Match one selected object to the best reference-method object.

    STRICT RANKING:
    ----------------
    1. Largest temporal overlap
    2. Largest spatial overlap
    3. Largest event magnitude
    4. Longest seed duration

    Spatial and temporal IoU are calculated only for evaluation
    after the best object is selected.
    """

    if len(reference_objects) == 0:

        return {
            "reference_obj_idx":
                None,

            "temporal_overlap_epochs":
                np.nan,

            "spatial_overlap_corepoints":
                np.nan,

            "reference_event_magnitude":
                np.nan,

            "reference_seed_duration_epochs":
                np.nan,

            "spatial_iou":
                np.nan,

            "temporal_iou":
                np.nan,
        }


    selected_indices = np.asarray(
        selected_obj.indices,
        dtype=int
    )


    selected_start = int(
        selected_obj.start_epoch
    )


    selected_end = int(
        selected_obj.end_epoch
    )


    best_idx = None


    best_score = (
        -1,
        -1,
        -np.inf,
        -1
    )


    best_temporal_overlap = np.nan

    best_spatial_overlap = np.nan

    best_event_magnitude = np.nan

    best_seed_duration = np.nan


    for (
        reference_idx,
        reference_obj
    ) in enumerate(
        reference_objects
    ):


        # ----------------------------------------------------
        # CRITERION 1 — TEMPORAL OVERLAP
        # ----------------------------------------------------

        temporal_overlap = (
            calculate_temporal_overlap(

                selected_start,
                selected_end,

                int(
                    reference_obj.start_epoch
                ),

                int(
                    reference_obj.end_epoch
                )
            )
        )


        # ----------------------------------------------------
        # CRITERION 2 — SPATIAL OVERLAP
        # ----------------------------------------------------

        spatial_overlap = (
            calculate_spatial_overlap(
                selected_indices,
                reference_obj.indices
            )
        )


        # ----------------------------------------------------
        # CRITERIA 3–4
        # ----------------------------------------------------

        (
            event_magnitude,
            seed_duration
        ) = (
            get_object_ranking_seed_information(

                obj=(
                    reference_obj
                ),

                seeds=(
                    reference_seeds
                )
            )
        )


        # ----------------------------------------------------
        # STRICT LEXICOGRAPHIC RANKING
        # ----------------------------------------------------

        current_score = (

            int(
                temporal_overlap
            ),

            int(
                spatial_overlap
            ),

            float(
                event_magnitude
            ),

            int(
                seed_duration
            )
        )


        if current_score > best_score:

            best_score = (
                current_score
            )

            best_idx = (
                reference_idx
            )

            best_temporal_overlap = (
                temporal_overlap
            )

            best_spatial_overlap = (
                spatial_overlap
            )

            best_event_magnitude = (
                event_magnitude
            )

            best_seed_duration = (
                seed_duration
            )


    # --------------------------------------------------------
    # Evaluation metrics for selected match
    # --------------------------------------------------------

    best_reference_obj = (
        reference_objects[
            best_idx
        ]
    )


    spatial_iou = (
        calculate_spatial_iou(

            selected_indices,

            best_reference_obj.indices
        )
    )


    temporal_iou = (
        calculate_temporal_iou(

            selected_start,
            selected_end,

            int(
                best_reference_obj.start_epoch
            ),

            int(
                best_reference_obj.end_epoch
            )
        )
    )


    return {

        "reference_obj_idx":
            best_idx,

        "temporal_overlap_epochs":
            int(
                best_temporal_overlap
            ),

        "spatial_overlap_corepoints":
            int(
                best_spatial_overlap
            ),

        "reference_event_magnitude":
            float(
                best_event_magnitude
            ),

        "reference_seed_duration_epochs":
            int(
                best_seed_duration
            ),

        "spatial_iou":
            float(
                spatial_iou
            ),

        "temporal_iou":
            float(
                temporal_iou
            ),
    }


# ============================================================
# 16. ROBUST CONVEX HULL
# ============================================================

def add_convex_hull(
    ax,
    points_xy,
    edgecolor,
    linewidth,
    linestyle,
    label,
    zorder
):
    """
    Safely add an unfilled convex hull.
    """

    points_xy = np.asarray(
        points_xy,
        dtype=float
    )


    if points_xy.shape[0] < 3:

        print(
            f"Convex hull skipped for '{label}': "
            "fewer than 3 points."
        )

        return False


    unique_points = np.unique(
        points_xy,
        axis=0
    )


    if unique_points.shape[0] < 3:

        print(
            f"Convex hull skipped for '{label}': "
            "fewer than 3 unique points."
        )

        return False


    try:

        hull = ConvexHull(
            unique_points
        )


        polygon_points = (
            unique_points[
                hull.vertices,
                :2
            ]
        )


        ax.add_patch(
            Polygon(

                polygon_points,

                fill=False,

                edgecolor=(
                    edgecolor
                ),

                linewidth=(
                    linewidth
                ),

                linestyle=(
                    linestyle
                ),

                alpha=1.0,

                label=(
                    label
                ),

                zorder=(
                    zorder
                )
            )
        )


        return True


    except Exception as exc:

        print(
            f"Convex hull could not be "
            f"created for '{label}':"
        )

        print(
            exc
        )

        return False


# ============================================================
# 17. MAIN GENERIC COMPARISON FUNCTION
# ============================================================

def plot_method_object_comparison(
    reference_method,
    selected_method,
    selected_object_number
):
    """
    Plot one selected object from selected_method against the
    highest-ranked matching object from reference_method.
    """

    reference_objects = (
        method_objects[
            reference_method
        ]
    )


    selected_objects = (
        method_objects[
            selected_method
        ]
    )


    reference_seeds = (
        method_seeds[
            reference_method
        ]
    )


    selected_seeds = (
        method_seeds[
            selected_method
        ]
    )


    if len(selected_objects) == 0:

        print(
            f"No {selected_method} objects available."
        )

        return None


    # --------------------------------------------------------
    # Select object from SECOND / selected method
    # --------------------------------------------------------

    if (
        OBJECT_SELECTION_MODE
        == "object_number"
    ):

        selected_obj_idx = (
            int(
                selected_object_number
            )
            - 1
        )


        if (
            selected_obj_idx < 0
            or
            selected_obj_idx
            >= len(
                selected_objects
            )
        ):

            print(
                f"Skipping {selected_method} object "
                f"{selected_object_number}: only "
                f"{len(selected_objects)} objects available."
            )

            return None


    elif (
        OBJECT_SELECTION_MODE
        == "contains_reference"
    ):

        selected_obj_idx = (
            find_object_containing_corepoint(

                selected_objects,

                REFERENCE_COREPOINT_INDEX
            )
        )


        if selected_obj_idx is None:

            print(
                f"No {selected_method} object contains "
                f"reference corepoint "
                f"{REFERENCE_COREPOINT_INDEX}."
            )

            return None


    else:

        raise ValueError(
            "OBJECT_SELECTION_MODE must be "
            "'object_number' or 'contains_reference'."
        )


    selected_obj = (
        selected_objects[
            selected_obj_idx
        ]
    )


    selected_object_number_user = (
        selected_obj_idx + 1
    )


    selected_idxs = np.asarray(
        selected_obj.indices,
        dtype=int
    )


    selected_index_set = set(
        selected_idxs.tolist()
    )


    # --------------------------------------------------------
    # Match selected object to reference method
    # --------------------------------------------------------

    match_result = (
        find_best_reference_match(

            selected_obj=(
                selected_obj
            ),

            reference_objects=(
                reference_objects
            ),

            reference_seeds=(
                reference_seeds
            )
        )
    )


    reference_obj_idx = (
        match_result[
            "reference_obj_idx"
        ]
    )


    temporal_overlap_epochs = (
        match_result[
            "temporal_overlap_epochs"
        ]
    )


    spatial_overlap_corepoints = (
        match_result[
            "spatial_overlap_corepoints"
        ]
    )


    matched_reference_event_magnitude = (
        match_result[
            "reference_event_magnitude"
        ]
    )


    matched_reference_seed_duration = (
        match_result[
            "reference_seed_duration_epochs"
        ]
    )


    spatial_iou = (
        match_result[
            "spatial_iou"
        ]
    )


    temporal_iou = (
        match_result[
            "temporal_iou"
        ]
    )


    # --------------------------------------------------------
    # Matched reference object
    # --------------------------------------------------------

    if reference_obj_idx is None:

        reference_obj = None

        reference_object_number_user = (
            np.nan
        )

        reference_idxs = np.array(
            [],
            dtype=int
        )


    else:

        reference_obj = (
            reference_objects[
                reference_obj_idx
            ]
        )


        reference_object_number_user = (
            reference_obj_idx + 1
        )


        reference_idxs = np.asarray(
            reference_obj.indices,
            dtype=int
        )


    # --------------------------------------------------------
    # Selected object temporal information
    # --------------------------------------------------------

    selected_start = int(
        selected_obj.start_epoch
    )


    selected_end = int(
        selected_obj.end_epoch
    )


    epoch_of_interest = int(
        (
            selected_start
            + selected_end
        )
        / 2
    )


    visualization_cp_idx = int(
        VISUALIZATION_COREPOINT_INDEX
    )


    reference_cp_idx = int(
        REFERENCE_COREPOINT_INDEX
    )


    # --------------------------------------------------------
    # Selected object's generating seed
    # --------------------------------------------------------

    selected_seed_info = (
        get_object_generating_seed_info(

            obj=(
                selected_obj
            ),

            seeds=(
                selected_seeds
            )
        )
    )


    selected_generating_cp = (
        selected_seed_info[
            "corepoint"
        ]
    )


    # --------------------------------------------------------
    # Selected object's ranking seed information
    # --------------------------------------------------------

    (
        selected_event_magnitude,
        selected_seed_duration
    ) = (
        get_object_ranking_seed_information(

            obj=(
                selected_obj
            ),

            seeds=(
                selected_seeds
            )
        )
    )


    object_contains_visualization_cp = (
        visualization_cp_idx
        in selected_index_set
    )


    object_contains_reference_cp = (
        reference_cp_idx
        in selected_index_set
    )


    # ========================================================
    # DIAGNOSTIC OUTPUT
    # ========================================================

    print(
        "\n=================================================="
    )

    print(
        f"{reference_method} vs {selected_method}"
    )

    print(
        "=================================================="
    )


    print(
        f"Selected {selected_method} object:",
        selected_object_number_user
    )


    print(
        f"Matched {reference_method} object:",
        reference_object_number_user
    )


    print(
        "\nMATCHING RANKING:"
    )


    print(
        "1. Temporal overlap [epochs]:",
        temporal_overlap_epochs
    )


    print(
        "2. Spatial overlap [corepoints]:",
        spatial_overlap_corepoints
    )


    print(
        "3. Matched reference event magnitude:",
        matched_reference_event_magnitude
    )


    print(
        "4. Matched reference seed duration:",
        matched_reference_seed_duration
    )


    print(
        "\nEVALUATION METRICS:"
    )


    print(
        "Spatial IoU:",
        spatial_iou
    )


    print(
        "Temporal IoU:",
        temporal_iou
    )


    print(
        f"\n{selected_method} start/end:",
        selected_start,
        selected_end
    )


    print(
        f"{selected_method} duration:",
        selected_end
        - selected_start
        + 1
    )


    print(
        f"{selected_method} event magnitude:",
        selected_event_magnitude
    )


    print(
        f"{selected_method} seed duration:",
        selected_seed_duration
    )


    print(
        "Visualisation corepoint:",
        visualization_cp_idx
    )


    print(
        "Reference corepoint:",
        reference_cp_idx
    )


    print(
        f"{selected_method} generating seed:",
        selected_generating_cp
    )


    print(
        f"{selected_method} NMS rank:",
        selected_seed_info[
            "nms_rank"
        ]
    )


    print(
        f"{selected_method} contains "
        "visualisation corepoint:",
        object_contains_visualization_cp
    )


    print(
        f"{selected_method} contains "
        "reference corepoint:",
        object_contains_reference_cp
    )


    if reference_obj is not None:

        print(
            f"{reference_method} start/end:",
            int(
                reference_obj.start_epoch
            ),
            int(
                reference_obj.end_epoch
            )
        )


    # ========================================================
    # VALUES USED IN BOTH PANELS
    # ========================================================

    magnitudes_of_interest = (

        analysis.smoothed_distances[
            :,
            epoch_of_interest
        ]

        -

        analysis.smoothed_distances[
            :,
            selected_start
        ]
    )


    visualization_timeseries = (
        analysis.smoothed_distances[
            visualization_cp_idx
        ]
    )


    if not pd.isna(
        selected_generating_cp
    ):

        generating_seed_timeseries = (
            analysis.smoothed_distances[
                int(
                    selected_generating_cp
                )
            ]
        )


    else:

        generating_seed_timeseries = (
            None
        )


    cmap = plt.get_cmap(
        "seismic_r"
    ).copy()


    norm = mcolors.CenteredNorm(
        halfrange=CRANGE
    )


    cmap_values = norm(
        magnitudes_of_interest
    )


    # ========================================================
    # CREATE FIGURE
    # ========================================================

    fig, axs = plt.subplots(

        1,
        2,

        figsize=(
            15,
            7
        ),

        gridspec_kw={
            "width_ratios":
                [
                    1.45,
                    1.0
                ]
        }
    )


    ax1, ax2 = axs


    # ========================================================
    # LEFT PANEL — SELECTED METHOD TEMPORAL BEHAVIOUR
    # ========================================================

    for idx in selected_idxs[
        ::10
    ]:

        ax1.plot(

            timestamps_analysis,

            analysis.smoothed_distances[
                idx
            ],

            color=cmap(
                cmap_values[
                    idx
                ]
            ),

            linewidth=0.5,

            alpha=0.35
        )


    # --------------------------------------------------------
    # Visualisation corepoint
    # --------------------------------------------------------

    ax1.plot(

        timestamps_analysis,

        visualization_timeseries,

        color=(
            VISUALIZATION_CP_COLOR
        ),

        linewidth=1.7,

        label=(
            f"Visualisation corepoint "
            f"{visualization_cp_idx}"
        ),

        zorder=6
    )


    # --------------------------------------------------------
    # Selected generating seed
    # --------------------------------------------------------

    if (
        generating_seed_timeseries
        is not None
        and
        int(
            selected_generating_cp
        )
        != visualization_cp_idx
    ):

        ax1.plot(

            timestamps_analysis,

            generating_seed_timeseries,

            color=(
                GENERATING_SEED_COLOR
            ),

            linestyle="--",

            linewidth=1.4,

            label=(
                f"{selected_method} generating seed "
                f"{int(selected_generating_cp)}"
            ),

            zorder=5
        )


    # --------------------------------------------------------
    # Selected method temporal interval
    # --------------------------------------------------------

    ax1.axvspan(

        timestamps_analysis[
            selected_start
        ],

        timestamps_analysis[
            selected_end
        ],

        color=(
            SELECTED_METHOD_HULL_COLOR
        ),

        alpha=0.18,

        label=(
            f"{selected_method} timespan"
        ),

        zorder=1
    )


    # --------------------------------------------------------
    # Matched reference interval
    # --------------------------------------------------------

    if reference_obj is not None:

        reference_start = int(
            reference_obj.start_epoch
        )

        reference_end = int(
            reference_obj.end_epoch
        )


        ax1.axvspan(

            timestamps_analysis[
                reference_start
            ],

            timestamps_analysis[
                reference_end
            ],

            color=(
                REFERENCE_METHOD_HULL_COLOR
            ),

            alpha=0.13,

            hatch="//",

            edgecolor=(
                REFERENCE_METHOD_HULL_COLOR
            ),

            linewidth=1.2,

            label=(
                f"Matched {reference_method} timespan"
            ),

            zorder=2
        )


    ax1.xaxis.set_major_formatter(
        dtFmt
    )


    ax1.set_title(
        f"{selected_method} temporal object behaviour\n"
        f"{selected_method} object "
        f"{selected_object_number_user} matched to "
        f"{reference_method} object "
        f"{reference_object_number_user}"
    )


    ax1.set_xlabel(
        "Date"
    )


    ax1.set_ylabel(
        "Distance [m]"
    )


    ax1.tick_params(
        axis="x",
        rotation=15
    )


    ax1.grid(
        True,
        alpha=0.35
    )


    ax1.legend(
        fontsize=8
    )


    # ========================================================
    # RIGHT PANEL — SPATIAL HULL COMPARISON
    # ========================================================

    selected_subset_cloud = (
        cloud[
            selected_idxs,
            :2
        ]
    )


    if reference_obj is not None:

        reference_subset_cloud = (
            cloud[
                reference_idxs,
                :2
            ]
        )


    else:

        reference_subset_cloud = (
            np.empty(
                (
                    0,
                    2
                )
            )
        )


    scatter = ax2.scatter(

        cloud[
            :,
            0
        ],

        cloud[
            :,
            1
        ],

        c=(
            magnitudes_of_interest
        ),

        cmap="seismic_r",

        vmin=-CRANGE,

        vmax=CRANGE,

        s=1,

        zorder=1
    )


    plt.colorbar(

        scatter,

        format="%.2f",

        label="Change magnitude [m]",

        ax=ax2
    )


    # --------------------------------------------------------
    # SELECTED METHOD FIRST
    # Cyan solid
    # --------------------------------------------------------

    add_convex_hull(

        ax=ax2,

        points_xy=(
            selected_subset_cloud
        ),

        edgecolor=(
            SELECTED_METHOD_HULL_COLOR
        ),

        linewidth=3.2,

        linestyle="-",

        label=(
            f"{selected_method} hull "
            f"(object "
            f"{selected_object_number_user})"
        ),

        zorder=5
    )


    # --------------------------------------------------------
    # REFERENCE METHOD SECOND
    # Magenta dashed + higher z-order
    # --------------------------------------------------------

    if reference_obj is not None:

        add_convex_hull(

            ax=ax2,

            points_xy=(
                reference_subset_cloud
            ),

            edgecolor=(
                REFERENCE_METHOD_HULL_COLOR
            ),

            linewidth=3.8,

            linestyle=(
                0,
                (
                    6,
                    3
                )
            ),

            label=(
                f"{reference_method} hull "
                f"(object "
                f"{reference_object_number_user})"
            ),

            zorder=8
        )


    # --------------------------------------------------------
    # Visualisation corepoint
    # --------------------------------------------------------

    ax2.scatter(

        cloud[
            visualization_cp_idx,
            0
        ],

        cloud[
            visualization_cp_idx,
            1
        ],

        marker="*",

        color=(
            VISUALIZATION_CP_COLOR
        ),

        s=170,

        linewidths=0.8,

        edgecolors="white",

        label=(
            f"Visualisation corepoint "
            f"{visualization_cp_idx}"
        ),

        zorder=10
    )


    # --------------------------------------------------------
    # Selected generating seed
    # --------------------------------------------------------

    if (
        not pd.isna(
            selected_generating_cp
        )
        and
        int(
            selected_generating_cp
        )
        != visualization_cp_idx
    ):

        ax2.scatter(

            cloud[
                int(
                    selected_generating_cp
                ),
                0
            ],

            cloud[
                int(
                    selected_generating_cp
                ),
                1
            ],

            marker="o",

            facecolors="none",

            edgecolors=(
                GENERATING_SEED_COLOR
            ),

            s=135,

            linewidths=2.4,

            label=(
                f"{selected_method} generating seed "
                f"{int(selected_generating_cp)}"
            ),

            zorder=9
        )


    # --------------------------------------------------------
    # Right-panel title
    # --------------------------------------------------------

    ax2.set_title(
        f"{selected_method} on {reference_method}\n"
        f"Temporal overlap: "
        f"{temporal_overlap_epochs} epochs | "
        f"Spatial overlap: "
        f"{spatial_overlap_corepoints} corepoints\n"
        f"Spatial IoU: "
        f"{spatial_iou:.3f} | "
        f"Temporal IoU: "
        f"{temporal_iou:.3f}"
    )


    ax2.set_xlabel(
        "X [m]"
    )


    ax2.set_ylabel(
        "Y [m]"
    )


    ax2.set_aspect(
        "equal"
    )


    ax2.legend(
        loc="upper right",
        fontsize=8
    )


    plt.tight_layout()


    # ========================================================
    # SAVE FIGURE
    # ========================================================

    safe_reference_name = (
        reference_method
        .lower()
        .replace(
            "-",
            "_"
        )
    )


    safe_selected_name = (
        selected_method
        .lower()
        .replace(
            "-",
            "_"
        )
    )


    figure_path = (

        method_comparison_fig_dir

        /

        (
            f"{safe_selected_name}_on_"
            f"{safe_reference_name}_"
            f"selected_object_"
            f"{selected_object_number_user}_"
            f"visualcp_"
            f"{visualization_cp_idx}.png"
        )
    )


    if SAVE_METHOD_COMPARISON_FIGURES:

        plt.savefig(

            figure_path,

            dpi=300,

            bbox_inches="tight",

            facecolor="white"
        )


        print(
            "Saved comparison figure:"
        )

        print(
            figure_path
        )


    plt.show()


    # ========================================================
    # RETURN COMPARISON RECORD
    # ========================================================

    return {

        "reference_method":
            reference_method,

        "selected_method":
            selected_method,

        "selection_mode":
            OBJECT_SELECTION_MODE,

        "selected_object_number_user":
            selected_object_number_user,

        "selected_object_index_python":
            selected_obj_idx,

        "matched_reference_object_number_user":
            reference_object_number_user,

        "matched_reference_object_index_python":
            (
                reference_obj_idx
                if
                reference_obj_idx is not None
                else
                np.nan
            ),


        # ----------------------------------------------------
        # Matching criteria
        # ----------------------------------------------------

        "rank_1_temporal_overlap_epochs":
            temporal_overlap_epochs,

        "rank_2_spatial_overlap_corepoints":
            spatial_overlap_corepoints,

        "rank_3_reference_event_magnitude":
            matched_reference_event_magnitude,

        "rank_4_reference_seed_duration_epochs":
            matched_reference_seed_duration,


        # ----------------------------------------------------
        # Evaluation metrics
        # ----------------------------------------------------

        "spatial_iou":
            spatial_iou,

        "temporal_iou":
            temporal_iou,


        # ----------------------------------------------------
        # Corepoint diagnostics
        # ----------------------------------------------------

        "reference_corepoint_index_python":
            reference_cp_idx,

        "visualization_corepoint_index_python":
            visualization_cp_idx,

        "selected_object_contains_reference_corepoint":
            object_contains_reference_cp,

        "selected_object_contains_visualization_corepoint":
            object_contains_visualization_cp,


        # ----------------------------------------------------
        # Selected generating seed
        # ----------------------------------------------------

        "selected_generating_seed_corepoint":
            selected_generating_cp,

        "selected_generating_seed_start_epoch":
            selected_seed_info[
                "start_epoch"
            ],

        "selected_generating_seed_end_epoch":
            selected_seed_info[
                "end_epoch"
            ],

        "selected_generating_seed_duration_epochs":
            selected_seed_info[
                "duration_epochs"
            ],

        "selected_generating_seed_direction":
            selected_seed_info[
                "direction"
            ],

        "selected_generating_seed_event_magnitude":
            selected_seed_info[
                "event_magnitude"
            ],

        "selected_generating_seed_nms_rank":
            selected_seed_info[
                "nms_rank"
            ],


        # ----------------------------------------------------
        # Selected object
        # ----------------------------------------------------

        "selected_start_epoch":
            selected_start,

        "selected_end_epoch":
            selected_end,

        "selected_duration_epochs":
            (
                selected_end
                - selected_start
                + 1
            ),

        "selected_n_corepoints":
            len(
                selected_idxs
            ),

        "selected_event_magnitude_for_ranking":
            selected_event_magnitude,

        "selected_seed_duration_for_ranking":
            selected_seed_duration,


        # ----------------------------------------------------
        # Reference object
        # ----------------------------------------------------

        "reference_start_epoch":
            (
                int(
                    reference_obj.start_epoch
                )
                if
                reference_obj is not None
                else
                np.nan
            ),

        "reference_end_epoch":
            (
                int(
                    reference_obj.end_epoch
                )
                if
                reference_obj is not None
                else
                np.nan
            ),

        "reference_duration_epochs":
            (
                int(
                    reference_obj.end_epoch
                )
                - int(
                    reference_obj.start_epoch
                )
                + 1
                if
                reference_obj is not None
                else
                np.nan
            ),

        "reference_n_corepoints":
            (
                len(
                    reference_idxs
                )
                if
                reference_obj is not None
                else
                np.nan
            ),


        # ----------------------------------------------------
        # Plot information
        # ----------------------------------------------------

        "epoch_of_interest":
            epoch_of_interest,

        "mean_change_magnitude_selected_object":
            float(
                np.nanmean(
                    magnitudes_of_interest[
                        selected_idxs
                    ]
                )
            ),

        "max_abs_change_magnitude_selected_object":
            float(
                np.nanmax(
                    np.abs(
                        magnitudes_of_interest[
                            selected_idxs
                        ]
                    )
                )
            ),

        "selected_hull_color":
            SELECTED_METHOD_HULL_COLOR,

        "reference_hull_color":
            REFERENCE_METHOD_HULL_COLOR,

        "figure_path":
            str(
                figure_path
            )
    }


# ============================================================
# 18. RUN ALL SIX COMPARISONS
# ============================================================

method_comparison_records = []


for comparison_spec in (
    COMPARISON_SPECS
):

    reference_method = (
        comparison_spec[
            "reference_method"
        ]
    )


    selected_method = (
        comparison_spec[
            "selected_method"
        ]
    )


    print(
        "\n\n##################################################"
    )

    print(
        f"COMPARISON: "
        f"{reference_method} vs {selected_method}"
    )

    print(
        "##################################################"
    )


    for object_number in (
        SELECTED_OBJECT_NUMBERS
    ):

        record = (
            plot_method_object_comparison(

                reference_method=(
                    reference_method
                ),

                selected_method=(
                    selected_method
                ),

                selected_object_number=(
                    object_number
                )
            )
        )


        if record is not None:

            record[
                "comparison_id"
            ] = (
                comparison_spec[
                    "comparison_id"
                ]
            )


            method_comparison_records.append(
                record
            )


# ============================================================
# 19. COMPLETE OBJECT-MATCHING SUMMARY
# ============================================================

method_comparison_summary = (
    pd.DataFrame(
        method_comparison_records
    )
)


display(
    method_comparison_summary
)


method_comparison_summary.to_csv(

    TABLES_DIR
    / "all_method_object_visualization_comparisons.csv",

    index=False
)


# ============================================================
# 20. PAIRWISE SUMMARY
# ============================================================

if len(
    method_comparison_summary
) > 0:

    pairwise_visualization_summary = (

        method_comparison_summary

        .groupby(
            [
                "reference_method",
                "selected_method"
            ],
            as_index=False
        )

        .agg(

            n_visualized_pairs=(
                "selected_object_number_user",
                "count"
            ),

            mean_temporal_overlap_epochs=(
                "rank_1_temporal_overlap_epochs",
                "mean"
            ),

            mean_spatial_overlap_corepoints=(
                "rank_2_spatial_overlap_corepoints",
                "mean"
            ),

            mean_spatial_iou=(
                "spatial_iou",
                "mean"
            ),

            mean_temporal_iou=(
                "temporal_iou",
                "mean"
            ),

            max_spatial_iou=(
                "spatial_iou",
                "max"
            ),

            max_temporal_iou=(
                "temporal_iou",
                "max"
            )
        )
    )


else:

    pairwise_visualization_summary = (
        pd.DataFrame()
    )


display(
    pairwise_visualization_summary
)


pairwise_visualization_summary.to_csv(

    TABLES_DIR
    / "pairwise_method_visualization_summary.csv",

    index=False
)


# ============================================================
# 21. COMPLETION MESSAGE
# ============================================================

print(
    "\n=========================================="
)

print(
    "CELL 16 COMPLETE"
)

print(
    "=========================================="
)


print(
    "\nObject matching hierarchy:"
)


print(
    "1. Largest temporal overlap"
)


print(
    "2. Largest spatial overlap"
)


print(
    "3. Largest event magnitude"
)


print(
    "4. Longest seed duration"
)


print(
    "\nSpatial and temporal IoU are evaluation "
    "metrics only."
)


print(
    "\nComparisons configured:",
    len(
        COMPARISON_SPECS
    )
)


print(
    "Comparison figures generated:",
    len(
        method_comparison_summary
    )
)


print(
    "\nSelected / second method hull:"
)

print(
    SELECTED_METHOD_HULL_COLOR,
    "(bright cyan, solid)"
)


print(
    "\nReference / first method hull:"
)

print(
    REFERENCE_METHOD_HULL_COLOR,
    "(bright magenta, dashed, drawn above selected method)"
)


print(
    "\nComparisons:"
)


for comparison_spec in (
    COMPARISON_SPECS
):

    print(
        " - "
        f"{comparison_spec['reference_method']} "
        "vs "
        f"{comparison_spec['selected_method']}"
    )

In [ ]:
# ============================================================
# CELL 16A — TARGET COREPOINT / GENERATING-SEED DIAGNOSTICS
# WITH CORRESPONDING ORIGINAL 4DOBC OBJECT
# ============================================================


# ------------------------------------------------------------
# 1. USER SETTINGS
# ------------------------------------------------------------

TARGET_COREPOINT_INDEX = 15162


DIAGNOSTIC_METHODS = [
    "4DOBC",
    "KF-Mag",
    "KF-Mag-NMS",
]


# ------------------------------------------------------------
# Object selection
# ------------------------------------------------------------

OBJECT_SELECTION_MODE = "contains_target"

# options:
#
#   "contains_target"
#       Select first object from each method containing the
#       target corepoint.
#
#   "object_number"
#       Use OBJECT_NUMBERS_TO_CHECK below.


OBJECT_NUMBERS_TO_CHECK = {

    "4DOBC": 1,

    "KF-Mag": 1,

    "KF-Mag-NMS": 1,
}


# ------------------------------------------------------------
# Plot settings
# ------------------------------------------------------------

SAVE_DIAGNOSTIC_FIGURES = True


# Final-method object interval
METHOD_OBJECT_COLOR = "grey"

# Native py4dgeo interval for KF-Mag-NMS
NATIVE_OBJECT_COLOR = "orange"

# Corresponding original 4DOBC object
ORIGINAL_4DOBC_COLOR = "magenta"

# Target signal / target seed
TARGET_COLOR = "blue"

# Object-generating corepoint/seed
GENERATOR_COLOR = "green"


# ------------------------------------------------------------
# Output directory
# ------------------------------------------------------------

diagnostic_fig_dir = (

    FIGURES_DIR
    / "target_generating_seed_original_4dobc_diagnostics"
)

diagnostic_fig_dir.mkdir(
    exist_ok=True,
    parents=True
)


# ============================================================
# 2. CHECK REQUIRED VARIABLES
# ============================================================

required_objects = [

    "analysis",
    "timestamps_analysis",

    "original_4dobc_objects",

    "kf_mag_objects",
    "kf_mag_nms_objects",

    "kf_mag_seeds",
    "kf_mag_nms_seeds",

    "kalman_change",
    "kalman_sigma_change",

    "kalman_seed_table_magnitude",
    "kf_mag_nms_seed_table",

    "TABLES_DIR",
]


for obj_name in required_objects:

    if obj_name not in globals():

        raise ValueError(
            f"{obj_name} is not defined. "
            "Please run the preceding extraction cells first."
        )


# ============================================================
# 3. IMPORTS
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates


dtFmt = mdates.DateFormatter(
    "%b-%d"
)


# ============================================================
# 4. METHOD DATA
# ============================================================

method_objects = {

    "4DOBC":
        original_4dobc_objects,

    "KF-Mag":
        kf_mag_objects,

    "KF-Mag-NMS":
        kf_mag_nms_objects,
}


method_py4dgeo_seeds = {

    "4DOBC":
        None,

    "KF-Mag":
        kf_mag_seeds,

    "KF-Mag-NMS":
        kf_mag_nms_seeds,
}


method_candidate_seed_tables = {

    "4DOBC":
        None,

    "KF-Mag":
        kalman_seed_table_magnitude,

    "KF-Mag-NMS":
        kf_mag_nms_seed_table,
}


# ============================================================
# 5. TEMPORAL OVERLAP
# ============================================================

def calculate_temporal_overlap(
    start_a,
    end_a,
    start_b,
    end_b
):

    return max(

        0,

        min(
            int(end_a),
            int(end_b)
        )

        - max(
            int(start_a),
            int(start_b)
        )

        + 1
    )


# ============================================================
# 6. TEMPORAL IoU
# ============================================================

def calculate_temporal_iou(
    start_a,
    end_a,
    start_b,
    end_b
):

    intersection = (
        calculate_temporal_overlap(
            start_a,
            end_a,
            start_b,
            end_b
        )
    )


    union = (

        max(
            int(end_a),
            int(end_b)
        )

        - min(
            int(start_a),
            int(start_b)
        )

        + 1
    )


    return (
        intersection / union
        if union > 0
        else 0.0
    )


# ============================================================
# 7. SPATIAL OVERLAP
# ============================================================

def calculate_spatial_overlap(
    indices_a,
    indices_b
):

    set_a = set(
        np.asarray(
            indices_a,
            dtype=int
        ).tolist()
    )


    set_b = set(
        np.asarray(
            indices_b,
            dtype=int
        ).tolist()
    )


    return len(
        set_a.intersection(
            set_b
        )
    )


# ============================================================
# 8. SPATIAL IoU
# ============================================================

def calculate_spatial_iou(
    indices_a,
    indices_b
):

    set_a = set(
        np.asarray(
            indices_a,
            dtype=int
        ).tolist()
    )


    set_b = set(
        np.asarray(
            indices_b,
            dtype=int
        ).tolist()
    )


    union = set_a.union(
        set_b
    )


    if len(union) == 0:

        return 0.0


    return (

        len(
            set_a.intersection(
                set_b
            )
        )

        / len(
            union
        )
    )


# ============================================================
# 9. FIND OBJECT CONTAINING TARGET
# ============================================================

def find_object_containing_target(
    objects,
    target_cp
):

    for object_idx, obj in enumerate(
        objects
    ):

        indices = np.asarray(
            obj.indices,
            dtype=int
        )


        if int(
            target_cp
        ) in indices:

            return object_idx


    return None


# ============================================================
# 10. FIND PY4DGEO SEED BELONGING TO OBJECT
# ============================================================

def find_seed_for_object(
    seeds,
    obj
):

    if seeds is None:

        return None


    object_indices = set(
        np.asarray(
            obj.indices,
            dtype=int
        ).tolist()
    )


    object_start = int(
        obj.start_epoch
    )

    object_end = int(
        obj.end_epoch
    )


    for seed in seeds:

        cp_idx = int(
            seed.index
        )


        if cp_idx not in object_indices:

            continue


        overlap = (
            calculate_temporal_overlap(

                object_start,
                object_end,

                int(
                    seed.start_epoch
                ),

                int(
                    seed.end_epoch
                )
            )
        )


        if overlap > 0:

            return seed


    return None


# ============================================================
# 11. ORIGINAL 4DOBC SEED INFORMATION
# ============================================================

def get_original_4dobc_seed_info(
    obj
):
    """
    Get original py4dgeo 4DOBC generating seed information.
    """

    if (
        hasattr(
            obj,
            "seed"
        )
        and
        obj.seed is not None
    ):

        seed = obj.seed


        return {

            "corepoint":
                int(
                    seed.index
                ),

            "start_epoch":
                int(
                    seed.start_epoch
                ),

            "end_epoch":
                int(
                    seed.end_epoch
                ),

            "duration_epochs":
                int(
                    seed.end_epoch
                    - seed.start_epoch
                    + 1
                ),
        }


    return {

        "corepoint":
            np.nan,

        "start_epoch":
            int(
                obj.start_epoch
            ),

        "end_epoch":
            int(
                obj.end_epoch
            ),

        "duration_epochs":
            int(
                obj.end_epoch
                - obj.start_epoch
                + 1
            ),
    }


# ============================================================
# 12. CALCULATE ORIGINAL 4DOBC EVENT MAGNITUDE
# ============================================================

def calculate_4dobc_event_magnitude(
    obj
):
    """
    Calculate maximum absolute change over the object's own
    temporal interval.

    Used only for matching criterion 3.
    """

    object_indices = np.asarray(
        obj.indices,
        dtype=int
    )


    if len(
        object_indices
    ) == 0:

        return 0.0


    start_epoch = int(
        obj.start_epoch
    )

    end_epoch = int(
        obj.end_epoch
    )


    change_values = (

        analysis.smoothed_distances[
            object_indices,
            end_epoch
        ]

        -

        analysis.smoothed_distances[
            object_indices,
            start_epoch
        ]
    )


    if not np.any(
        np.isfinite(
            change_values
        )
    ):

        return 0.0


    return float(
        np.nanmax(
            np.abs(
                change_values
            )
        )
    )


# ============================================================
# 13. MATCH KF OBJECT TO ORIGINAL 4DOBC OBJECT
# ============================================================

def find_corresponding_original_4dobc(
    selected_obj
):
    """
    Match selected KF object to corresponding original 4DOBC.

    Ranking:
        1. temporal overlap
        2. spatial overlap
        3. event magnitude
        4. seed duration
    """

    selected_indices = np.asarray(
        selected_obj.indices,
        dtype=int
    )


    selected_start = int(
        selected_obj.start_epoch
    )

    selected_end = int(
        selected_obj.end_epoch
    )


    best_idx = None


    best_score = (
        -1,
        -1,
        -np.inf,
        -1
    )


    best_record = None


    for original_idx, original_obj in enumerate(
        original_4dobc_objects
    ):

        # ----------------------------------------------------
        # 1. Temporal overlap
        # ----------------------------------------------------

        temporal_overlap = (
            calculate_temporal_overlap(

                selected_start,
                selected_end,

                int(
                    original_obj.start_epoch
                ),

                int(
                    original_obj.end_epoch
                )
            )
        )


        # ----------------------------------------------------
        # 2. Spatial overlap
        # ----------------------------------------------------

        spatial_overlap = (
            calculate_spatial_overlap(

                selected_indices,

                original_obj.indices
            )
        )


        # ----------------------------------------------------
        # 3. Event magnitude
        # ----------------------------------------------------

        event_magnitude = (
            calculate_4dobc_event_magnitude(
                original_obj
            )
        )


        # ----------------------------------------------------
        # 4. Original seed duration
        # ----------------------------------------------------

        original_seed_info = (
            get_original_4dobc_seed_info(
                original_obj
            )
        )


        seed_duration = int(
            original_seed_info[
                "duration_epochs"
            ]
        )


        current_score = (

            int(
                temporal_overlap
            ),

            int(
                spatial_overlap
            ),

            float(
                event_magnitude
            ),

            int(
                seed_duration
            ),
        )


        if current_score > best_score:

            best_score = (
                current_score
            )


            best_idx = (
                original_idx
            )


            best_record = {

                "temporal_overlap_epochs":
                    temporal_overlap,

                "spatial_overlap_corepoints":
                    spatial_overlap,

                "event_magnitude":
                    event_magnitude,

                "seed_duration_epochs":
                    seed_duration,

                "spatial_iou":
                    calculate_spatial_iou(

                        selected_indices,

                        original_obj.indices
                    ),

                "temporal_iou":
                    calculate_temporal_iou(

                        selected_start,
                        selected_end,

                        int(
                            original_obj.start_epoch
                        ),

                        int(
                            original_obj.end_epoch
                        )
                    ),
            }


    if best_idx is None:

        return (
            None,
            None
        )


    return (
        best_idx,
        best_record
    )


# ============================================================
# 14. GET KF-METHOD GENERATING SEED
# ============================================================

def get_method_generating_seed(
    method_name,
    obj
):

    # --------------------------------------------------------
    # KF-Mag-NMS
    # --------------------------------------------------------

    if (
        hasattr(
            obj,
            "matched_seed"
        )
        and
        obj.matched_seed is not None
    ):

        row = (
            obj.matched_seed
        )


        start_epoch = int(

            row[
                "support_start_epoch"
            ]

            if
            "support_start_epoch" in row
            and
            not pd.isna(
                row[
                    "support_start_epoch"
                ]
            )

            else
            row[
                "start_epoch"
            ]
        )


        end_epoch = int(

            row[
                "support_end_epoch"
            ]

            if
            "support_end_epoch" in row
            and
            not pd.isna(
                row[
                    "support_end_epoch"
                ]
            )

            else
            row[
                "end_epoch"
            ]
        )


        return {

            "corepoint":
                int(
                    row[
                        "corepoint_index_python"
                    ]
                ),

            "start_epoch":
                start_epoch,

            "end_epoch":
                end_epoch,

            "duration_epochs":
                end_epoch
                - start_epoch
                + 1,

            "direction":
                row.get(
                    "direction",
                    None
                ),

            "event_magnitude":
                (
                    float(
                        row[
                            "event_magnitude"
                        ]
                    )

                    if
                    "event_magnitude" in row
                    and
                    not pd.isna(
                        row[
                            "event_magnitude"
                        ]
                    )

                    else
                    np.nan
                ),

            "nms_rank":
                (
                    int(
                        row[
                            "nms_rank"
                        ]
                    )

                    if
                    "nms_rank" in row
                    and
                    not pd.isna(
                        row[
                            "nms_rank"
                        ]
                    )

                    else
                    np.nan
                ),

            "source":
                "matched NMS seed",
        }


    # --------------------------------------------------------
    # KF-Mag
    # --------------------------------------------------------

    py4dgeo_seed = (
        find_seed_for_object(

            method_py4dgeo_seeds[
                method_name
            ],

            obj
        )
    )


    if py4dgeo_seed is not None:

        cp_idx = int(
            py4dgeo_seed.index
        )


        start_epoch = int(
            py4dgeo_seed.start_epoch
        )


        end_epoch = int(
            py4dgeo_seed.end_epoch
        )


        event_magnitude = np.nan

        direction = None


        seed_table = (
            method_candidate_seed_tables[
                method_name
            ]
        )


        if (
            seed_table is not None
            and
            len(seed_table) > 0
        ):

            matched_rows = (

                seed_table[

                    (
                        seed_table[
                            "corepoint_index_python"
                        ]
                        == cp_idx
                    )

                    &

                    (
                        seed_table[
                            "start_epoch"
                        ]
                        == start_epoch
                    )

                    &

                    (
                        seed_table[
                            "end_epoch"
                        ]
                        == end_epoch
                    )
                ]
            )


            if len(
                matched_rows
            ) > 0:

                row = (
                    matched_rows.iloc[0]
                )


                if (
                    "event_magnitude"
                    in row
                    and
                    not pd.isna(
                        row[
                            "event_magnitude"
                        ]
                    )
                ):

                    event_magnitude = float(
                        row[
                            "event_magnitude"
                        ]
                    )


                if (
                    "direction"
                    in row
                ):

                    direction = (
                        row[
                            "direction"
                        ]
                    )


        return {

            "corepoint":
                cp_idx,

            "start_epoch":
                start_epoch,

            "end_epoch":
                end_epoch,

            "duration_epochs":
                end_epoch
                - start_epoch
                + 1,

            "direction":
                direction,

            "event_magnitude":
                event_magnitude,

            "nms_rank":
                np.nan,

            "source":
                "KF-Mag RegionGrowingSeed",
        }


    return {

        "corepoint":
            np.nan,

        "start_epoch":
            np.nan,

        "end_epoch":
            np.nan,

        "duration_epochs":
            np.nan,

        "direction":
            None,

        "event_magnitude":
            np.nan,

        "nms_rank":
            np.nan,

        "source":
            "not found",
    }


# ============================================================
# 15. GET TARGET SEEDS
# ============================================================

def get_target_seed_table(
    method_name
):

    seed_table = (
        method_candidate_seed_tables[
            method_name
        ]
    )


    if (
        seed_table is None
        or
        len(seed_table) == 0
    ):

        return pd.DataFrame()


    return (

        seed_table[

            seed_table[
                "corepoint_index_python"
            ]
            == TARGET_COREPOINT_INDEX
        ]

        .copy()
    )


# ============================================================
# 16. GET NATIVE OBJECT INTERVAL
# ============================================================

def get_native_object_interval(
    obj
):

    if (
        hasattr(
            obj,
            "original_start_epoch"
        )
        and
        hasattr(
            obj,
            "original_end_epoch"
        )
    ):

        return (

            int(
                obj.original_start_epoch
            ),

            int(
                obj.original_end_epoch
            ),

            True
        )


    return (

        int(
            obj.start_epoch
        ),

        int(
            obj.end_epoch
        ),

        False
    )


# ============================================================
# 17. METHOD SIGNAL
# ============================================================

def get_method_signal(
    method_name,
    cp_idx
):

    cp_idx = int(
        cp_idx
    )


    if method_name == "4DOBC":

        return (
            analysis.smoothed_distances[
                cp_idx
            ]
        )


    return (
        kalman_change[
            cp_idx
        ]
    )


# ============================================================
# 18. OBJECT SELECTION
# ============================================================

def select_diagnostic_object(
    method_name
):

    objects = (
        method_objects[
            method_name
        ]
    )


    if len(objects) == 0:

        return (
            None,
            None
        )


    if (
        OBJECT_SELECTION_MODE
        == "contains_target"
    ):

        object_idx = (
            find_object_containing_target(

                objects,

                TARGET_COREPOINT_INDEX
            )
        )


        if object_idx is None:

            return (
                None,
                None
            )


    elif (
        OBJECT_SELECTION_MODE
        == "object_number"
    ):

        object_idx = (

            int(
                OBJECT_NUMBERS_TO_CHECK[
                    method_name
                ]
            )

            - 1
        )


        if (
            object_idx < 0
            or
            object_idx >= len(objects)
        ):

            return (
                None,
                None
            )


    else:

        raise ValueError(
            "OBJECT_SELECTION_MODE must be "
            "'contains_target' or 'object_number'."
        )


    return (
        object_idx,
        objects[
            object_idx
        ]
    )


# ============================================================
# 19. PLOT ONE DIAGNOSTIC
# ============================================================

def plot_diagnostic(
    method_name
):

    (
        object_idx,
        selected_obj
    ) = select_diagnostic_object(
        method_name
    )


    if selected_obj is None:

        print(
            f"No suitable {method_name} object found."
        )

        return None


    object_number = (
        object_idx + 1
    )


    object_indices = set(
        np.asarray(
            selected_obj.indices,
            dtype=int
        ).tolist()
    )


    object_start = int(
        selected_obj.start_epoch
    )


    object_end = int(
        selected_obj.end_epoch
    )


    object_duration = (
        object_end
        - object_start
        + 1
    )


    # ========================================================
    # ORIGINAL 4DOBC OBJECT
    # ========================================================

    if method_name == "4DOBC":

        original_obj_idx = (
            object_idx
        )

        original_obj = (
            selected_obj
        )


        original_match_info = {

            "temporal_overlap_epochs":
                object_duration,

            "spatial_overlap_corepoints":
                len(
                    object_indices
                ),

            "event_magnitude":
                calculate_4dobc_event_magnitude(
                    selected_obj
                ),

            "seed_duration_epochs":
                get_original_4dobc_seed_info(
                    selected_obj
                )[
                    "duration_epochs"
                ],

            "spatial_iou":
                1.0,

            "temporal_iou":
                1.0,
        }


    else:

        (
            original_obj_idx,
            original_match_info
        ) = find_corresponding_original_4dobc(
            selected_obj
        )


        if original_obj_idx is None:

            original_obj = None

        else:

            original_obj = (
                original_4dobc_objects[
                    original_obj_idx
                ]
            )


    # --------------------------------------------------------
    # Original object metadata
    # --------------------------------------------------------

    if original_obj is not None:

        original_object_number = (
            original_obj_idx + 1
        )


        original_start = int(
            original_obj.start_epoch
        )


        original_end = int(
            original_obj.end_epoch
        )


        original_seed_info = (
            get_original_4dobc_seed_info(
                original_obj
            )
        )


    else:

        original_object_number = np.nan

        original_start = np.nan

        original_end = np.nan

        original_seed_info = {

            "corepoint":
                np.nan,

            "start_epoch":
                np.nan,

            "end_epoch":
                np.nan,

            "duration_epochs":
                np.nan,
        }


    # ========================================================
    # CURRENT METHOD GENERATING SEED
    # ========================================================

    if method_name == "4DOBC":

        generating_seed = (
            original_seed_info
            | {
                "direction":
                    None,

                "event_magnitude":
                    original_match_info[
                        "event_magnitude"
                    ],

                "nms_rank":
                    np.nan,

                "source":
                    "original 4DOBC seed",
            }
        )


    else:

        generating_seed = (
            get_method_generating_seed(

                method_name,

                selected_obj
            )
        )


    generating_cp = (
        generating_seed[
            "corepoint"
        ]
    )


    # --------------------------------------------------------
    # Target seed table
    # --------------------------------------------------------

    target_seed_table = (
        get_target_seed_table(
            method_name
        )
    )


    # --------------------------------------------------------
    # Native py4dgeo object interval
    # --------------------------------------------------------

    (
        native_start,
        native_end,
        has_distinct_native
    ) = get_native_object_interval(
        selected_obj
    )


    # ========================================================
    # DIAGNOSTIC TEXT
    # ========================================================

    print(
        "\n"
        + "=" * 82
    )


    print(
        f"{method_name} TARGET / GENERATING-SEED DIAGNOSTIC"
    )


    print(
        "=" * 82
    )


    print(
        "Selected object:",
        object_number
    )


    print(
        "Selected object start/end:",
        object_start,
        object_end
    )


    print(
        "Selected object duration:",
        object_duration
    )


    print(
        "Selected object contains target:",
        TARGET_COREPOINT_INDEX
        in object_indices
    )


    print(
        "\nGenerating seed source:",
        generating_seed[
            "source"
        ]
    )


    print(
        "Generating corepoint:",
        generating_cp
    )


    print(
        "Generating seed start/end:",
        generating_seed[
            "start_epoch"
        ],
        generating_seed[
            "end_epoch"
        ]
    )


    print(
        "Generating seed duration:",
        generating_seed[
            "duration_epochs"
        ]
    )


    if method_name != "4DOBC":

        print(
            "\nCorresponding original 4DOBC object:",
            original_object_number
        )


        print(
            "Original 4DOBC start/end:",
            original_start,
            original_end
        )


        print(
            "\n4DOBC MATCHING CRITERIA:"
        )


        print(
            "1. Temporal overlap:",
            original_match_info[
                "temporal_overlap_epochs"
            ],
            "epochs"
        )


        print(
            "2. Spatial overlap:",
            original_match_info[
                "spatial_overlap_corepoints"
            ],
            "corepoints"
        )


        print(
            "3. Original event magnitude:",
            original_match_info[
                "event_magnitude"
            ]
        )


        print(
            "4. Original seed duration:",
            original_match_info[
                "seed_duration_epochs"
            ]
        )


        print(
            "\nEvaluation spatial IoU:",
            original_match_info[
                "spatial_iou"
            ]
        )


        print(
            "Evaluation temporal IoU:",
            original_match_info[
                "temporal_iou"
            ]
        )


    # ========================================================
    # SIGNALS
    # ========================================================

    target_signal = (
        get_method_signal(

            method_name,

            TARGET_COREPOINT_INDEX
        )
    )


    if not pd.isna(
        generating_cp
    ):

        generating_signal = (
            get_method_signal(

                method_name,

                int(
                    generating_cp
                )
            )
        )


    else:

        generating_signal = (
            None
        )


    # ========================================================
    # FIGURE
    # ========================================================

    fig, ax = plt.subplots(
        figsize=(
            13,
            6.7
        )
    )


    # --------------------------------------------------------
    # Raw M3C2 target
    # --------------------------------------------------------

    ax.scatter(

        timestamps_analysis,

        analysis.distances[
            TARGET_COREPOINT_INDEX
        ],

        s=8,

        color="black",

        alpha=0.35,

        label=(
            f"Raw M3C2 target "
            f"{TARGET_COREPOINT_INDEX}"
        )
    )


    # --------------------------------------------------------
    # Main target signal
    # --------------------------------------------------------

    target_signal_label = (

        f"Smoothed M3C2 target "
        f"{TARGET_COREPOINT_INDEX}"

        if method_name == "4DOBC"

        else

        f"Kalman target "
        f"{TARGET_COREPOINT_INDEX}"
    )


    ax.plot(

        timestamps_analysis,

        target_signal,

        color=(
            TARGET_COLOR
        ),

        linewidth=2,

        label=(
            target_signal_label
        )
    )


    # --------------------------------------------------------
    # Generating corepoint signal
    # --------------------------------------------------------

    if (
        generating_signal is not None
        and
        int(
            generating_cp
        )
        != TARGET_COREPOINT_INDEX
    ):

        generator_label = (

            f"Original 4DOBC generating cp "
            f"{int(generating_cp)}"

            if method_name == "4DOBC"

            else

            f"{method_name} object-generating cp "
            f"{int(generating_cp)}"
        )


        ax.plot(

            timestamps_analysis,

            generating_signal,

            color=(
                GENERATOR_COLOR
            ),

            linestyle="--",

            linewidth=2,

            label=(
                generator_label
            )
        )


    # --------------------------------------------------------
    # Kalman uncertainty
    # --------------------------------------------------------

    if method_name in [
        "KF-Mag",
        "KF-Mag-NMS",
    ]:

        upper = (
            1.96
            * kalman_sigma_change[
                TARGET_COREPOINT_INDEX
            ]
        )


        lower = (
            -1.96
            * kalman_sigma_change[
                TARGET_COREPOINT_INDEX
            ]
        )


        ax.fill_between(

            timestamps_analysis,

            lower,

            upper,

            color=(
                TARGET_COLOR
            ),

            alpha=0.10,

            label="±1.96σ target uncertainty"
        )


    # ========================================================
    # SHADED TEMPORAL INTERVALS
    # ========================================================

    # --------------------------------------------------------
    # Current method object
    # --------------------------------------------------------

    ax.axvspan(

        timestamps_analysis[
            object_start
        ],

        timestamps_analysis[
            object_end
        ],

        color=(
            METHOD_OBJECT_COLOR
        ),

        alpha=0.27,

        label=(
            f"{method_name} object interval"
        )
    )


    # --------------------------------------------------------
    # Corresponding ORIGINAL 4DOBC object
    # --------------------------------------------------------

    if (
        method_name != "4DOBC"
        and
        original_obj is not None
    ):

        ax.axvspan(

            timestamps_analysis[
                original_start
            ],

            timestamps_analysis[
                original_end
            ],

            color=(
                ORIGINAL_4DOBC_COLOR
            ),

            alpha=0.12,

            hatch="//",

            edgecolor=(
                ORIGINAL_4DOBC_COLOR
            ),

            linewidth=1.2,

            label=(
                f"Matched original 4DOBC object "
                f"{original_object_number}"
            )
        )


    # --------------------------------------------------------
    # Native py4dgeo object interval
    # --------------------------------------------------------

    if (
        has_distinct_native
        and
        (
            native_start != object_start
            or
            native_end != object_end
        )
    ):

        ax.axvspan(

            timestamps_analysis[
                native_start
            ],

            timestamps_analysis[
                native_end
            ],

            color=(
                NATIVE_OBJECT_COLOR
            ),

            alpha=0.17,

            label="Native py4dgeo object interval"
        )


    # ========================================================
    # DETERMINE BAR POSITION
    # ========================================================

    finite_values = []


    for signal in [
        target_signal,
        generating_signal
    ]:

        if signal is None:

            continue


        arr = np.asarray(
            signal,
            dtype=float
        )


        arr = arr[
            np.isfinite(
                arr
            )
        ]


        if len(arr) > 0:

            finite_values.extend(
                arr.tolist()
            )


    if len(
        finite_values
    ) > 0:

        signal_min = float(
            np.min(
                finite_values
            )
        )


        signal_max = float(
            np.max(
                finite_values
            )
        )


        signal_range = max(
            signal_max - signal_min,
            0.5
        )


        y_base = (
            signal_min
            - 0.18
            * signal_range
        )


        y_step = (
            0.065
            * signal_range
        )


    else:

        y_base = -0.7

        y_step = 0.065


    current_y = (
        y_base
    )


    # ========================================================
    # HORIZONTAL BAR — CURRENT METHOD OBJECT
    # ========================================================

    ax.hlines(

        current_y,

        timestamps_analysis[
            object_start
        ],

        timestamps_analysis[
            object_end
        ],

        color=(
            METHOD_OBJECT_COLOR
        ),

        linewidth=8
    )


    ax.text(

        timestamps_analysis[
            object_start
        ],

        current_y
        + 0.20
        * y_step,

        f"{method_name} object {object_number}",

        fontsize=8,

        color=(
            METHOD_OBJECT_COLOR
        )
    )


    current_y -= (
        y_step
    )


    # ========================================================
    # HORIZONTAL BAR — ORIGINAL 4DOBC OBJECT
    # ========================================================

    if (
        method_name != "4DOBC"
        and
        original_obj is not None
    ):

        ax.hlines(

            current_y,

            timestamps_analysis[
                original_start
            ],

            timestamps_analysis[
                original_end
            ],

            color=(
                ORIGINAL_4DOBC_COLOR
            ),

            linewidth=6
        )


        ax.text(

            timestamps_analysis[
                original_start
            ],

            current_y
            + 0.20
            * y_step,

            (
                f"Matched original 4DOBC "
                f"object {original_object_number}"
            ),

            fontsize=8,

            color=(
                ORIGINAL_4DOBC_COLOR
            )
        )


        current_y -= (
            y_step
        )


    # ========================================================
    # HORIZONTAL BAR — NATIVE PY4DGEO OBJECT
    # ========================================================

    if (
        has_distinct_native
        and
        (
            native_start != object_start
            or
            native_end != object_end
        )
    ):

        ax.hlines(

            current_y,

            timestamps_analysis[
                native_start
            ],

            timestamps_analysis[
                native_end
            ],

            color=(
                NATIVE_OBJECT_COLOR
            ),

            linewidth=6
        )


        ax.text(

            timestamps_analysis[
                native_start
            ],

            current_y
            + 0.20
            * y_step,

            "Native py4dgeo object",

            fontsize=8,

            color=(
                NATIVE_OBJECT_COLOR
            )
        )


        current_y -= (
            y_step
        )


    # ========================================================
    # TARGET-COREPOINT SEEDS
    # ========================================================

    if len(
        target_seed_table
    ) > 0:

        for _, row in (
            target_seed_table.iterrows()
        ):

            if (
                "support_start_epoch"
                in row
                and
                not pd.isna(
                    row[
                        "support_start_epoch"
                    ]
                )
            ):

                seed_start = int(
                    row[
                        "support_start_epoch"
                    ]
                )


            else:

                seed_start = int(
                    row[
                        "start_epoch"
                    ]
                )


            if (
                "support_end_epoch"
                in row
                and
                not pd.isna(
                    row[
                        "support_end_epoch"
                    ]
                )
            ):

                seed_end = int(
                    row[
                        "support_end_epoch"
                    ]
                )


            else:

                seed_end = int(
                    row[
                        "end_epoch"
                    ]
                )


            ax.hlines(

                current_y,

                timestamps_analysis[
                    seed_start
                ],

                timestamps_analysis[
                    seed_end
                ],

                color=(
                    TARGET_COLOR
                ),

                linewidth=4
            )


            if (
                "nms_rank"
                in row
                and
                not pd.isna(
                    row[
                        "nms_rank"
                    ]
                )
            ):

                seed_label = (

                    f"{TARGET_COREPOINT_INDEX} "
                    f"NMS seed "
                    f"{int(row['nms_rank'])}"
                )


            else:

                seed_label = (

                    f"{TARGET_COREPOINT_INDEX} "
                    f"{method_name} seed"
                )


            ax.text(

                timestamps_analysis[
                    seed_start
                ],

                current_y
                + 0.20
                * y_step,

                seed_label,

                fontsize=8,

                color=(
                    TARGET_COLOR
                )
            )


            current_y -= (
                y_step
            )


    # ========================================================
    # CURRENT METHOD GENERATING SEED BAR
    # ========================================================

    if (
        not pd.isna(
            generating_seed[
                "start_epoch"
            ]
        )
        and
        not pd.isna(
            generating_seed[
                "end_epoch"
            ]
        )
    ):

        generator_start = int(
            generating_seed[
                "start_epoch"
            ]
        )


        generator_end = int(
            generating_seed[
                "end_epoch"
            ]
        )


        ax.hlines(

            current_y,

            timestamps_analysis[
                generator_start
            ],

            timestamps_analysis[
                generator_end
            ],

            color=(
                GENERATOR_COLOR
            ),

            linewidth=5
        )


        if not pd.isna(
            generating_seed[
                "nms_rank"
            ]
        ):

            generator_label = (

                f"{int(generating_cp)} "
                f"NMS seed "
                f"{int(generating_seed['nms_rank'])}"
            )


        else:

            generator_label = (

                f"{int(generating_cp)} "
                f"{method_name} generating seed"
            )


        ax.text(

            timestamps_analysis[
                generator_start
            ],

            current_y
            + 0.20
            * y_step,

            generator_label,

            fontsize=8,

            color=(
                GENERATOR_COLOR
            )
        )


        current_y -= (
            y_step
        )


    # ========================================================
    # ORIGINAL 4DOBC GENERATING SEED BAR
    # ========================================================

    if (
        method_name != "4DOBC"
        and
        original_obj is not None
        and
        not pd.isna(
            original_seed_info[
                "start_epoch"
            ]
        )
        and
        not pd.isna(
            original_seed_info[
                "end_epoch"
            ]
        )
    ):

        original_seed_start = int(
            original_seed_info[
                "start_epoch"
            ]
        )


        original_seed_end = int(
            original_seed_info[
                "end_epoch"
            ]
        )


        ax.hlines(

            current_y,

            timestamps_analysis[
                original_seed_start
            ],

            timestamps_analysis[
                original_seed_end
            ],

            color=(
                ORIGINAL_4DOBC_COLOR
            ),

            linewidth=4
        )


        original_seed_cp = (
            original_seed_info[
                "corepoint"
            ]
        )


        ax.text(

            timestamps_analysis[
                original_seed_start
            ],

            current_y
            + 0.20
            * y_step,

            (
                f"4DOBC seed "
                f"{original_seed_cp}"
            ),

            fontsize=8,

            color=(
                ORIGINAL_4DOBC_COLOR
            )
        )


        current_y -= (
            y_step
        )


    # ========================================================
    # AXES
    # ========================================================

    ax.set_xlabel(
        "Date"
    )


    ax.set_ylabel(
        "Change [m]"
    )


    ax.tick_params(
        axis="x",
        rotation=15
    )


    ax.xaxis.set_major_formatter(
        dtFmt
    )


    if method_name == "4DOBC":

        title_line_2 = (

            "Target corepoint vs original object-generating "
            "seed vs object interval"
        )


    else:

        title_line_2 = (

            f"Target corepoint vs {method_name} generating seed "
            f"vs matched original 4DOBC object"
        )


    ax.set_title(
        f"{method_name}\n"
        f"{title_line_2}"
    )


    ax.grid(
        True
    )


    ax.legend(
        fontsize=8,
        loc="upper left"
    )


    plt.tight_layout()


    # ========================================================
    # SAVE FIGURE
    # ========================================================

    safe_method = (

        method_name
        .lower()
        .replace(
            "-",
            "_"
        )
    )


    figure_path = (

        diagnostic_fig_dir

        /

        (
            f"{safe_method}_"
            f"target_{TARGET_COREPOINT_INDEX}_"
            f"object_{object_number}_"
            f"with_original_4dobc.png"
        )
    )


    if SAVE_DIAGNOSTIC_FIGURES:

        plt.savefig(

            figure_path,

            dpi=300,

            bbox_inches="tight",

            facecolor="white"
        )


        print(
            "\nSaved diagnostic figure:"
        )


        print(
            figure_path
        )


    plt.show()


    # ========================================================
    # RETURN SUMMARY RECORD
    # ========================================================

    return {

        "method":
            method_name,

        "selection_mode":
            OBJECT_SELECTION_MODE,

        "selected_object_number":
            object_number,

        "target_corepoint":
            TARGET_COREPOINT_INDEX,

        "selected_object_contains_target":
            (
                TARGET_COREPOINT_INDEX
                in object_indices
            ),

        "selected_start_epoch":
            object_start,

        "selected_end_epoch":
            object_end,

        "selected_duration_epochs":
            object_duration,

        "native_py4dgeo_start_epoch":
            native_start,

        "native_py4dgeo_end_epoch":
            native_end,

        "has_distinct_native_interval":
            has_distinct_native,

        "object_generating_corepoint":
            generating_cp,

        "generating_seed_source":
            generating_seed[
                "source"
            ],

        "generating_seed_start_epoch":
            generating_seed[
                "start_epoch"
            ],

        "generating_seed_end_epoch":
            generating_seed[
                "end_epoch"
            ],

        "generating_seed_duration_epochs":
            generating_seed[
                "duration_epochs"
            ],

        "generating_seed_event_magnitude":
            generating_seed[
                "event_magnitude"
            ],

        "generating_seed_nms_rank":
            generating_seed[
                "nms_rank"
            ],

        "matched_original_4dobc_object_number":
            original_object_number,

        "matched_original_4dobc_start_epoch":
            original_start,

        "matched_original_4dobc_end_epoch":
            original_end,

        "matched_original_4dobc_seed_corepoint":
            original_seed_info[
                "corepoint"
            ],

        "matched_original_4dobc_seed_start_epoch":
            original_seed_info[
                "start_epoch"
            ],

        "matched_original_4dobc_seed_end_epoch":
            original_seed_info[
                "end_epoch"
            ],

        "match_rank_1_temporal_overlap_epochs":
            original_match_info[
                "temporal_overlap_epochs"
            ],

        "match_rank_2_spatial_overlap_corepoints":
            original_match_info[
                "spatial_overlap_corepoints"
            ],

        "match_rank_3_original_event_magnitude":
            original_match_info[
                "event_magnitude"
            ],

        "match_rank_4_original_seed_duration":
            original_match_info[
                "seed_duration_epochs"
            ],

        "matched_original_spatial_iou":
            original_match_info[
                "spatial_iou"
            ],

        "matched_original_temporal_iou":
            original_match_info[
                "temporal_iou"
            ],

        "n_target_seed_intervals":
            len(
                target_seed_table
            ),

        "figure_path":
            str(
                figure_path
            ),
    }


# ============================================================
# 20. RUN DIAGNOSTICS
# ============================================================

diagnostic_records = []


for method_name in (
    DIAGNOSTIC_METHODS
):

    result = (
        plot_diagnostic(
            method_name
        )
    )


    if result is not None:

        diagnostic_records.append(
            result
        )


# ============================================================
# 21. SUMMARY TABLE
# ============================================================

seed_object_original_4dobc_diagnostic_summary = (

    pd.DataFrame(
        diagnostic_records
    )
)


display(
    seed_object_original_4dobc_diagnostic_summary
)


seed_object_original_4dobc_diagnostic_summary.to_csv(

    TABLES_DIR
    / (
        "4dobc_kf_mag_kf_mag_nms_"
        "target_seed_object_diagnostics.csv"
    ),

    index=False
)


# ============================================================
# 22. FINAL STATUS
# ============================================================

print(
    "\n=========================================="
)


print(
    "CELL 16A COMPLETE"
)


print(
    "=========================================="
)


print(
    "\nTarget corepoint:",
    TARGET_COREPOINT_INDEX
)


print(
    "\nObject selection mode:",
    OBJECT_SELECTION_MODE
)


print(
    "\nDiagnostics generated for:"
)


for method_name in (
    DIAGNOSTIC_METHODS
):

    print(
        " -",
        method_name
    )


print(
    "\nKF objects are matched to original 4DOBC using:"
)


print(
    "1. Largest temporal overlap"
)


print(
    "2. Largest spatial overlap"
)


print(
    "3. Largest event magnitude"
)


print(
    "4. Longest seed duration"
)


print(
    "\nGenerated figures:",
    len(
        seed_object_original_4dobc_diagnostic_summary
    )
)